# ARC Execution-RL v2 — offline Kaggle inference
**37% milestone / 60% stretch — both are unmeasured targets, not guarantees.**

G0 is the existing exact-16-token ARC Qwen3-4B baseline. P1 is the separately trained full-vocabulary program specialist. This notebook does not perform full-model RL within the hidden evaluation runtime.

It preserves G0 Attempt 1 and the base fallback. P1 can alter Attempt 2 only with matching paired-validation approval. No eligible P1 input means baseline-only execution, not an automatic 37% result.

All data, model weights and dependencies must already be mounted under `/kaggle/input`. This notebook contains no hosted-model API call or Astra weight dependency.

In [ ]:
from pathlib import Path
import os,sys,json,time
Path("/kaggle/working").mkdir(parents=True,exist_ok=True)
os.chdir("/kaggle/working")
Path("arc_exec_rl").mkdir(exist_ok=True)
sys.path.insert(0,"/kaggle/working")
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"


In [ ]:
# ========================= EDITABLE SETTINGS =========================
import json
import math
import os
import time
from pathlib import Path

NOTEBOOK_START_TIME = time.time()
RUN_PROFILE = "DATASET_SAFE_MAX"  # DATASET_SAFE_MAX | PARITY_FULL | COVERAGE_FAST

# Public diagnostic gates. 80/172 is 46.51%; literal 60% is 104/172.
TARGET_REQUESTED_PAIRS = 80
TARGET_MILESTONE = 0.37
TARGET_MILESTONE_PAIRS = math.ceil(172*TARGET_MILESTONE)
TARGET_LITERAL_60_PAIRS = math.ceil(172 * 0.60)

# Full competition coverage. Keep empty for every serious run.
ARC_DEV_KEYS = ""
NPROCS = 4

# Competition/model paths. Custom primary is disabled because the previous
# ARC-Spark package scored 0/172. Re-enable only after exact parity validation.
COMPETITION_DIR_OVERRIDE = ""
PRIMARY_MODEL_OVERRIDE = ""
ALLOW_CUSTOM_PRIMARY = False
REQUIRE_EXACT_16_TOKEN_IDS = True

# Qwen3-4B per-task adaptation. adamw_torch is the stable, known baseline.
TTT_OPTIMIZER = "adamw_torch"
TTT_LEARNING_RATE = 5e-5
TTT_LORA_R = 256
TTT_LORA_ALPHA = 32
MAX_TRAIN_AUGMENTS = 15        # 15 -> about 128 mixed views
FORCE_FULL_TTT = False         # True reproduces full TTT but risks coverage
MAX_ACCEPTED_TTT_LOSS = 5.0    # pathological/non-finite runs are reverted
ALLOW_EXACT_RETRIEVAL_SKIP = True

# Decode/search settings.
MAX_SEQ_LENGTH = 8192
SYMBOLIC_PROGRAMS = 500
ALLOW_SYMBOLIC_ATTEMPT2 = False
TTA_COLOR_PERMUTATIONS = 2     # 8 geometry x 2 color = 16 views
MAX_SHAPE_HYPOTHESES = 3
CONSTRAINED_BRANCH_CAP = 4
UNCONSTRAINED_BRANCH_CAP = 6
BEAM_CAP = 14
MAX_CANDIDATES_PER_VIEW = 6
MAX_PUZZLE_SECONDS = 900
MIN_PUZZLE_SECONDS = 240
RESERVE_SCORING_SECONDS = 55

# Runtime and diagnostics.
TOTAL_NOTEBOOK_HOURS = 12.0
FINAL_RESERVE_MINUTES = 45
RUN_PUBLIC_CPU_AUDIT = False
SAVE_DIAGNOSTICS_ZIP = True

SAFE_DIR = Path("/kaggle/working/arc_exec_rl_v2_run/anchor")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/kaggle/inference_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["ARC_DEV_KEYS"] = ARC_DEV_KEYS
os.environ["ARC_DEBUG_KEYS"] = ARC_DEV_KEYS
os.environ["ARC_OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["ARC_SAFE_ROUTE_PATH"] = str(SAFE_DIR / "safe_routes.json")
os.environ["ARC_ALLOW_SYMBOLIC_ATTEMPT2"] = "1" if ALLOW_SYMBOLIC_ATTEMPT2 else "0"
os.environ["ARC_ALLOW_EXACT_RETRIEVAL_SKIP"] = "1" if ALLOW_EXACT_RETRIEVAL_SKIP else "0"
os.environ["ARC_FORCE_FULL_TTT"] = "1" if FORCE_FULL_TTT else "0"
os.environ["ARC_MAX_ACCEPTED_TTT_LOSS"] = str(MAX_ACCEPTED_TTT_LOSS)
os.environ["ARC60_TTT_OPTIMIZER"] = TTT_OPTIMIZER
os.environ["ARC60_TTT_LR"] = str(TTT_LEARNING_RATE)
os.environ["ARC60_TTT_LORA_R"] = str(TTT_LORA_R)
os.environ["ARC60_TTT_LORA_ALPHA"] = str(TTT_LORA_ALPHA)
os.environ["ARC_MAX_SEQ_LENGTH"] = str(MAX_SEQ_LENGTH)
os.environ["ARC_MAX_TRAIN_AUGMENTS"] = str(MAX_TRAIN_AUGMENTS)
os.environ["ARC_SYMBOLIC_PROGRAMS"] = str(SYMBOLIC_PROGRAMS)
os.environ["ARC_TTA_COLOR_PERMUTATIONS"] = str(TTA_COLOR_PERMUTATIONS)
os.environ["ARC_MAX_SHAPE_HYPOTHESES"] = str(MAX_SHAPE_HYPOTHESES)
os.environ["ARC_CONSTRAINED_BRANCH_CAP"] = str(CONSTRAINED_BRANCH_CAP)
os.environ["ARC_UNCONSTRAINED_BRANCH_CAP"] = str(UNCONSTRAINED_BRANCH_CAP)
os.environ["ARC_BEAM_CAP"] = str(BEAM_CAP)
os.environ["ARC_MAX_CANDIDATES_PER_VIEW"] = str(MAX_CANDIDATES_PER_VIEW)
os.environ["ARC_MAX_PUZZLE_SECONDS"] = str(MAX_PUZZLE_SECONDS)
os.environ["ARC_MIN_PUZZLE_SECONDS"] = str(MIN_PUZZLE_SECONDS)
os.environ["ARC_RESERVE_SCORING_SECONDS"] = str(RESERVE_SCORING_SECONDS)
os.environ["PYTHONHASHSEED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

settings = {
    "run_profile": RUN_PROFILE,
    "target_requested_pairs": TARGET_REQUESTED_PAIRS,
    "target_literal_60_pairs": TARGET_LITERAL_60_PAIRS,
    "arc_dev_keys": ARC_DEV_KEYS,
    "nprocs": NPROCS,
    "allow_custom_primary": ALLOW_CUSTOM_PRIMARY,
    "ttt_optimizer": TTT_OPTIMIZER,
    "ttt_learning_rate": TTT_LEARNING_RATE,
    "ttt_lora_r": TTT_LORA_R,
    "max_train_augments": MAX_TRAIN_AUGMENTS,
    "force_full_ttt": FORCE_FULL_TTT,
    "allow_symbolic_attempt2": ALLOW_SYMBOLIC_ATTEMPT2,
    "tta_color_permutations": TTA_COLOR_PERMUTATIONS,
    "max_shape_hypotheses": MAX_SHAPE_HYPOTHESES,
    "final_reserve_minutes": FINAL_RESERVE_MINUTES,
}
(SAFE_DIR / "settings.json").write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))

# ---- Independently trained P1: NEVER load this path into the 16-token G0 worker. ----
PROGRAM_MODEL_DIR = ""               # Notebook 1's verified sft_merged or rl_merged directory.
PROGRAM_MODE = "promoted"            # off | shadow (development only) | promoted
PROGRAM_RESERVE_SECONDS = 5400        # Reserved only if a usable/eligible P1 is attached.
MIN_PRIMARY_COVERAGE = 0.95
PROGRAM_INFER = {"max_prompt_tokens": 7000, "max_new_tokens": 512, "samples": 8,
                 "temperature": 0.7, "top_p": 0.9, "per_task_seconds": 120.0,
                 "relational": True, "repair_rounds": 1,"microbatch":2,"stop_on_complete_json":True,"json_check_interval":8}
PROGRAM_OUTPUT = Path("/kaggle/working/arc_exec_rl_v2_run/program")
PROGRAM_READY = False
TARGET_ACCURACY = 0.60                # Reporting target only; cannot alter prediction or reward.

INPUT_ROOT = '/kaggle/input'
WHEELHOUSE = ''
INSTALL_OFFLINE_DEPS = False


## Embedded program modules and frozen baseline

In [ ]:
%%writefile arc_exec_rl/__init__.py
"""ARC execution-reward training and guarded hybrid inference.
No score or trained weights are implied by the package name.
"""
__version__ = "1.0.0"


In [ ]:
%%writefile arc_exec_rl/common.py
from __future__ import annotations
import hashlib
import json
import math
import os
import platform
import tempfile
import time
from pathlib import Path
from typing import Any


def truthy(value: str | None) -> bool:
    return str(value or "").strip().lower() in {"1", "true", "yes", "on"}


def stable_seed(*parts: Any) -> int:
    return int(hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:8], 16)


def json_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def file_hash(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path: str | Path, value: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".writing-", dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(value, f, separators=(",", ":"), sort_keys=True, allow_nan=False)
            f.flush()
            os.fsync(f.fileno())
        with open(tmp, encoding="utf-8") as f:
            json.load(f)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.unlink(tmp)


def append_event(path: str | Path, event: str, **fields: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Each process uses its own event file. No cross-process buffering assumptions.
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"event": event, "time": time.time(), **fields}, allow_nan=False) + "\n")
        f.flush()


def validate_grid(g: Any) -> list[list[int]]:
    # Strict: floats, booleans and stringified arrays are not accepted.
    if not isinstance(g, list) or not 1 <= len(g) <= 30:
        raise ValueError("Grid must have 1..30 rows")
    if not isinstance(g[0], list) or not 1 <= len(g[0]) <= 30:
        raise ValueError("Grid must have 1..30 columns")
    w = len(g[0])
    if any(not isinstance(row, list) or len(row) != w for row in g):
        raise ValueError("Ragged grid")
    if any(type(x) is not int or not 0 <= x <= 9 for row in g for x in row):
        raise ValueError("Cells must be integer colors 0..9")
    return g


def public_task(task: dict) -> dict:
    """The only view supplied to inference: all known demos, test INPUTS only."""
    if not task.get("train") or not task.get("test"):
        raise ValueError("Task requires demonstrations and test inputs")
    return {"train": [{"input": validate_grid(p["input"]), "output": validate_grid(p["output"])}
                      for p in task["train"]],
            "test": [{"input": validate_grid(p["input"])} for p in task["test"]]}


def validate_submission(tasks: dict, submission: dict) -> None:
    if set(tasks) != set(submission):
        raise ValueError("Submission IDs differ from challenge IDs")
    for key, task in tasks.items():
        pairs = submission[key]
        if not isinstance(pairs, list) or len(pairs) != len(task["test"]):
            raise ValueError(f"Wrong number of test records: {key}")
        for p in pairs:
            if set(p) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"Invalid attempt keys: {key}")
            validate_grid(p["attempt_1"])
            validate_grid(p["attempt_2"])
    # Duplicate attempts are legal. Do not invent a one-cell mutation.


def initial_submission(tasks: dict) -> dict:
    out = {k: [{"attempt_1": p["input"], "attempt_2": p["input"]} for p in public_task(t)["test"]]
           for k, t in tasks.items()}
    validate_submission(tasks, out)
    return out


def score_submission(tasks: dict, solutions: dict, submission: dict, candidates: dict | None = None) -> dict:
    validate_submission(tasks, submission)
    if set(tasks) != set(solutions):
        raise ValueError("Solutions must match the evaluated challenge subset exactly")
    first = either = fully = oracle = total = 0
    per_task = {}
    for key, task in tasks.items():
        ys = solutions[key]
        if len(ys) != len(task["test"]):
            raise ValueError(f"Solution count mismatch: {key}")
        wins = []
        for i, y in enumerate(ys):
            validate_grid(y)
            p = submission[key][i]
            a = p["attempt_1"] == y
            b = a or p["attempt_2"] == y
            all_grids = [p["attempt_1"], p["attempt_2"]]
            if candidates:
                all_grids += candidates.get(f"{key}:{i}", [])
            total += 1
            first += int(a)
            either += int(b)
            oracle += int(y in all_grids)
            wins.append(int(b))
        fully += int(all(wins))
        per_task[key] = {"pairs": len(wins), "exact": sum(wins), "fraction": sum(wins) / len(wins)}
    return {"tasks": len(tasks), "pairs": total, "attempt1_exact": first, "pass2_exact": either,
            "pair_micro_pass2": either / max(total, 1),
            "task_macro_pass2": sum(x["fraction"] for x in per_task.values()) / max(len(tasks), 1),
            "fully_solved_tasks": fully, "candidate_oracle_exact": oracle,
            "target_60_required_pairs": math.ceil(.60 * total),
            "target_60_pair_micro_met": either >= math.ceil(.60 * total),
            "per_task": per_task}


def environment_report() -> dict:
    import importlib.metadata as md
    r = {"python": platform.python_version(), "architecture": platform.machine(), "system": platform.platform()}
    for p in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "tokenizers"):
        try:
            r[p] = md.version(p)
        except md.PackageNotFoundError:
            r[p] = "NOT_INSTALLED"
    try:
        import torch
        r["cuda_available"] = torch.cuda.is_available()
        r["torch_cuda"] = torch.version.cuda
        r["gpus"] = [{"name": torch.cuda.get_device_name(i),
                      "memory_bytes": torch.cuda.get_device_properties(i).total_memory}
                     for i in range(torch.cuda.device_count())]
    except ImportError:
        r["cuda_available"] = False
    return r


def score_banks(tasks: dict,solutions: dict,banks: dict) -> dict:
    """Abstention-aware scoring: no candidates cannot accidentally count as a solution."""
    total=first=either=oracle=fully=0; pt={}
    if set(tasks)!=set(solutions): raise ValueError("Mismatched solutions")
    for key,t in tasks.items():
        ys=solutions[key]
        if len(ys)!=len(t["test"]): raise ValueError("Mismatched test outputs")
        wins=[]
        for i,y in enumerate(ys):
            validate_grid(y); candidates=banks.get(f"{key}:{i}",[])
            for g in candidates: validate_grid(g)
            a=bool(candidates) and candidates[0]==y
            b=any(g==y for g in candidates[:2]); o=any(g==y for g in candidates)
            first+=int(a);either+=int(b);oracle+=int(o);total+=1;wins.append(int(b))
        fully+=int(all(wins));pt[key]={"pairs":len(wins),"exact":sum(wins),"fraction":sum(wins)/len(wins)}
    return {"tasks":len(tasks),"pairs":total,"attempt1_exact":first,"pass2_exact":either,
            "pair_micro_pass2":either/max(total,1),"task_macro_pass2":sum(v['fraction'] for v in pt.values())/max(len(pt),1),
            "fully_solved_tasks":fully,"candidate_oracle_exact":oracle,"per_task":pt,
            "target_60_required_pairs":math.ceil(.60*total),"target_60_pair_micro_met":either>=math.ceil(.60*total)}


def atomic_candidate_pickle(path: str | Path,value: Any) -> None:
    """Only for our own legacy worker outputs, never for untrusted downloaded pickles."""
    import bz2,pickle
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix='.candidate-',dir=p.parent);os.close(fd)
    try:
        with bz2.BZ2File(tmp,'wb') as f: pickle.dump(value,f,protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp,p)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)


In [ ]:
%%writefile arc_exec_rl/corpus.py
"""Verified bootstrap program corpus and strict external-data ingestion.
The bootstrap is a curriculum starter, not a replacement for NVARC-scale diversity.
Independent reference implementations check its generated labels against the DSL.
"""
from __future__ import annotations
import argparse
import copy
import itertools
import json
import random
from collections import Counter, deque
from pathlib import Path
import numpy as np
from .common import validate_grid, json_hash, atomic_json, stable_seed, file_hash
from .dsl import execute, parse_program, program_family, DSL_VERSION


def _reference_crop(g, bg):
    pts=[(r,c) for r,row in enumerate(g) for c,x in enumerate(row) if x!=bg]
    if not pts: raise ValueError("empty crop")
    r0=min(p[0] for p in pts); r1=max(p[0] for p in pts)+1
    c0=min(p[1] for p in pts); c1=max(p[1] for p in pts)+1
    return [row[c0:c1] for row in g[r0:r1]]


def reference(program: dict, grid: list[list[int]]) -> list[list[int]]:
    """No call to execute/apply_op. Labels generated independently for this subset."""
    g=copy.deepcopy(grid)
    for s in program["steps"]:
        op=s["op"]; h=len(g); w=len(g[0])
        if op=="identity": pass
        elif op=="rotate":
            for _ in range(s["k"]): g=[list(row) for row in zip(*g)][::-1]
        elif op=="reflect":
            axis=s["axis"]
            if axis=="h": g=[row[::-1] for row in g]
            elif axis=="v": g=g[::-1]
            else: g=[list(row) for row in zip(*g)]
        elif op=="crop": g=_reference_crop(g,s["bg"])
        elif op=="recolor": g=[[s["mapping"].get(str(x),x) for x in row] for row in g]
        elif op=="scale":
            g=[[x for x in row for _ in range(s["col"])] for row in g for _ in range(s["row"])]
        elif op=="tile":
            # Each tiled row must own its cells. deepcopy preserves shared aliases;
            # repeating a list of rows would let a later connect operation paint
            # corresponding rows in other tiles unintentionally.
            g=[row*s["col"] for _ in range(s["row"]) for row in g]
        elif op=="mirror_concat":
            g=([row+row[::-1] for row in g] if s["axis"]=="h" else [row[:] for row in g+g[::-1]])
        elif op=="fill_holes":
            bg=s["bg"]; seen=set(); q=deque()
            for r in range(h):
                for c in range(w):
                    if (r in (0,h-1) or c in (0,w-1)) and g[r][c]==bg:
                        seen.add((r,c)); q.append((r,c))
            while q:
                r,c=q.popleft()
                for nr,nc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]:
                    if 0<=nr<h and 0<=nc<w and g[nr][nc]==bg and (nr,nc) not in seen:
                        seen.add((nr,nc)); q.append((nr,nc))
            g=[[s["color"] if x==bg and (r,c) not in seen else x for c,x in enumerate(row)] for r,row in enumerate(g)]
        elif op=="move_to_edge":
            bg=s["bg"]; pts=[(r,c) for r,row in enumerate(g) for c,x in enumerate(row) if x!=bg]
            r0=min(r for r,c in pts); r1=max(r for r,c in pts)+1
            c0=min(c for r,c in pts); c1=max(c for r,c in pts)+1
            dr,dc={"left":(0,-c0),"right":(0,w-c1),"up":(-r0,0),"down":(h-r1,0)}[s["direction"]]
            out=[[bg]*w for _ in range(h)]
            for r,c in pts: out[r+dr][c+dc]=g[r][c]
            g=out
        elif op=="connect":
            out=copy.deepcopy(g); col=s["color"]; bg=s["bg"]
            if s["axis"] in ("h","both"):
                for r,row in enumerate(g):
                    positions=[c for c,x in enumerate(row) if x==col]
                    if len(positions)>=2:
                        for c in range(min(positions),max(positions)+1):
                            if g[r][c] in (bg,col): out[r][c]=col
            if s["axis"] in ("v","both"):
                for c in range(w):
                    positions=[r for r in range(h) if g[r][c]==col]
                    if len(positions)>=2:
                        for r in range(min(positions),max(positions)+1):
                            if g[r][c] in (bg,col): out[r][c]=col
            g=out
        else: raise ValueError(f"Bootstrap has no independent reference for {op}")
        validate_grid(g)
    return g

BOOT_OPS=("rotate","reflect","crop","recolor","scale","tile","mirror_concat","fill_holes","move_to_edge","connect")


def parameterize(ops: tuple[str,...], rng: random.Random) -> dict:
    steps=[]
    for op in ops:
        s={"op":op}
        if op=="rotate": s["k"]=rng.choice([1,2,3])
        elif op=="reflect": s["axis"]=rng.choice(["h","v","d"])
        elif op=="crop": s["bg"]=0
        elif op=="recolor":
            a,b=rng.sample(range(1,10),2); s["mapping"]={str(a):b,str(b):a}
        elif op in ("scale","tile"): s.update(row=rng.choice([1,2]),col=rng.choice([1,2]))
        elif op=="mirror_concat": s["axis"]=rng.choice(["h","v"])
        elif op=="fill_holes": s.update(bg=0,color=rng.randint(1,9))
        elif op=="move_to_edge": s.update(bg=0,direction=rng.choice(["left","right","up","down"]))
        elif op=="connect": s.update(bg=0,color=rng.randint(1,9),axis=rng.choice(["h","v","both"]))
        steps.append(s)
    return {"steps":steps}


def scene(rng: random.Random) -> list[list[int]]:
    h=rng.randint(5,10); w=rng.randint(5,10); g=[[0]*w for _ in range(h)]
    for _ in range(rng.randint(1,4)):
        rh=rng.randint(1,min(h-2,4)); cw=rng.randint(1,min(w-2,4))
        r0=rng.randint(1,h-rh-1); c0=rng.randint(1,w-cw-1); col=rng.randint(1,9)
        hollow=rng.random()<.5 and rh>=3 and cw>=3
        for r in range(r0,r0+rh):
            for c in range(c0,c0+cw):
                if not hollow or r in (r0,r0+rh-1) or c in (c0,c0+cw-1): g[r][c]=col
    for _ in range(rng.randint(0,3)):
        g[rng.randrange(1,h-1)][rng.randrange(1,w-1)]=rng.randint(1,9)
    return g


def skeletons() -> list[tuple[str,...]]:
    # Entire operator-order signatures are assigned to one split.
    result=[(x,) for x in BOOT_OPS]
    result += [(a,b) for a,b in itertools.product(BOOT_OPS,repeat=2) if a!=b]
    result += [("crop",a,b) for a in ("rotate","reflect","recolor") for b in ("scale","tile","mirror_concat")]
    return result


def split_for_family(family: str) -> str:
    # Atomic families stay in train. Holdouts test new compositions of familiar ops.
    if "/" not in family: return "train"
    bucket=stable_seed("split-v1",family)%10
    return "development" if bucket==0 else "confirmation" if bucket==1 else "train"


def canonical_task_key(task: dict) -> str:
    """Conservative demo-order/D4/color invariant fingerprint.
    Color names are canonicalized within each pair, so this may over-group tasks
    with different cross-pair color bindings. It does NOT prove semantic disjointness.
    """
    reps=[]
    def encode(grids,k,flip):
        cmap={};seq=[]
        for grid in grids:
            a=np.rot90(np.asarray(grid),k)
            if flip: a=np.fliplr(a)
            result=[]
            for row in a:
                rr=[]
                for x in row:
                    x=int(x)
                    if x not in cmap: cmap[x]=len(cmap)
                    rr.append(cmap[x])
                result.append(rr)
            seq.append(result)
        return json.dumps(seq,separators=(",",":"))
    for flip in (False,True):
        for k in range(4):
            demos=sorted(encode([p['input'],p['output']],k,flip) for p in task['train'])
            queries=sorted(encode([p['input']],k,flip) for p in task['test'])
            reps.append(json.dumps([demos,queries],separators=(",",":")))
    return json_hash(min(reps))


def make_record(ops: tuple[str,...], seed: int) -> dict | None:
    rng=random.Random(seed); program=parameterize(ops,rng); pairs=[]; seen=set()
    for _ in range(160):
        g=scene(rng)
        if json_hash(g) in seen: continue
        try:
            y=reference(program,g)
            if execute(program,g)!=y: raise AssertionError("Independent executor mismatch")
        except ValueError: continue
        seen.add(json_hash(g)); pairs.append({"input":g,"output":y})
        if len(pairs)==7: break
    if len(pairs)<7 or all(p["input"]==p["output"] for p in pairs[:4]): return None
    # One consistent permutation over all ten colors, including background, per episode.
    palette=list(range(10)); rng.shuffle(palette)
    for pair in pairs:
        for name in ("input","output"):
            pair[name]=[[palette[v] for v in row] for row in pair[name]]
    program=copy.deepcopy(program)
    for step in program["steps"]:
        for name in ("bg","color","target"):
            if type(step.get(name)) is int: step[name]=palette[step[name]]
        if step["op"]=="recolor":
            step["mapping"]={str(palette[int(k)]):palette[v] for k,v in step["mapping"].items()}
    # Both independent reference and executor must still agree after re-binding colors.
    for pair in pairs:
        if reference(program,pair["input"])!=pair["output"] or execute(program,pair["input"])!=pair["output"]:
            raise AssertionError("Color-binding transform failed verification")
    support=pairs[:4]; queries=pairs[4:]
    family=program_family(program)
    task={"train":support,"test":[{"input":p["input"]} for p in queries]}
    return {"record_id":"boot-"+json_hash([program,seed])[:24],"family_id":family,
            "split":split_for_family(family),"source":"independent-bootstrap-v1","source_split":"synthetic",
            "license":"MIT","source_task_ids":[],"task":task,
            "query_outputs":[p["output"] for p in queries],"program":program,
            "verification":"independent-reference-and-executor-exact",
            "canonical_hash":canonical_task_key(task),"generator_metadata":{"palette":palette}}


def validate_record(r: dict, forbidden_ids: set[str] | None = None) -> dict:
    forbidden_ids=forbidden_ids or set()
    import re
    if not isinstance(r.get("record_id"),str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,96}",r["record_id"]):
        raise ValueError("Unsafe or missing record_id")
    for field in ("record_id","family_id","split","source","source_split","license","source_task_ids","task","query_outputs","program","verification"):
        if field not in r: raise ValueError(f"Missing field {field}")
    if r["source_split"] not in ("training","synthetic"): raise ValueError("Evaluation-derived records are not allowed")
    if r["split"] not in ("train","development","confirmation"): raise ValueError("Invalid split")
    if str(r["license"]).lower() not in {"mit", "apache-2.0", "cc-by-4.0", "cc0-1.0"}:
        raise ValueError("License requires explicit review before inclusion")
    if not isinstance(r["source_task_ids"],list) or any(not isinstance(x,str) for x in r["source_task_ids"]):
        raise ValueError("source_task_ids must be a string list")
    if not isinstance(r["family_id"],str) or not r["family_id"]:
        raise ValueError("Missing family identity")
    if forbidden_ids.intersection(r["source_task_ids"]): raise ValueError("Forbidden evaluation lineage")
    task=r["task"]
    if len(task["train"])<2 or len(task["test"])!=len(r["query_outputs"]) or not task["test"]: raise ValueError("Invalid pairs")
    if any("output" in p for p in task["test"]): raise ValueError("Test labels must stay in private reward column")
    p=parse_program(r["program"])
    for pair in task["train"]:
        validate_grid(pair["input"]); validate_grid(pair["output"])
        if execute(p,pair["input"])!=pair["output"]: raise ValueError("Program fails demonstration")
    for pair,y in zip(task["test"],r["query_outputs"]):
        validate_grid(y)
        if execute(p,pair["input"])!=y: raise ValueError("Program fails reward query")
    r=dict(r); r["canonical_hash"]=canonical_task_key(task)
    return r


def build_corpus(output_dir: str | Path, per_family: int = 24, seed: int = 42,
                 external_jsonl: str = "", forbidden_ids: set[str] | None = None) -> dict:
    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True)
    records=[]; attempts=0
    for ops in skeletons():
        for j in range(per_family):
            attempts+=1
            r=make_record(ops,stable_seed(seed,ops,j))
            if r: records.append(validate_record(r,forbidden_ids))
    if external_jsonl:
        with open(external_jsonl,encoding="utf-8") as f:
            for line in f:
                if line.strip(): records.append(validate_record(json.loads(line),forbidden_ids))
    family_splits={}; hashes={}; accepted=[]
    for r in records:
        old=family_splits.setdefault(r["family_id"],r["split"])
        if old!=r["split"]: raise ValueError("Family leakage across splits")
        h=r["canonical_hash"]
        if h in hashes:
            if hashes[h]!=r["split"]: raise ValueError("Canonical task leakage across splits")
            continue
        hashes[h]=r["split"]; accepted.append(r)
    for split in ("train","development","confirmation"):
        with (out/f"{split}.jsonl").open("w",encoding="utf-8") as f:
            for r in accepted:
                if r["split"]==split: f.write(json.dumps(r,separators=(",",":"))+"\n")
    counts=Counter(r["split"] for r in accepted)
    report={"status":"BUILT", "bootstrap_attempts":attempts,"accepted":len(accepted),"counts":dict(counts),
            "family_counts":{s:len({r['family_id'] for r in accepted if r['split']==s}) for s in counts},
            "dataset_hash":json_hash(sorted((r['record_id'],json_hash(r)) for r in accepted)),
            "file_hashes":{s+'.jsonl':file_hash(out/(s+'.jsonl')) for s in ('train','development','confirmation')},
            "dsl_version":DSL_VERSION,"bootstrap_only":all(r["source"]=="independent-bootstrap-v1" for r in accepted),
            "source_counts":dict(Counter(r["source"] for r in accepted)),
            "warning":"Bootstrap tests operator compositions, not full ARC-AGI-2 coverage. No accuracy claim.",
            "canonicalization_limit":"Conservative D4/demo-order/pair-color invariant; may over-group, cannot prove semantic independence",
            "validation_exposure":"development may be used for tuning; confirmation must remain locked"}
    atomic_json(out/"manifest.json",report)
    return report


def read_records(path: str | Path, expected_split: str | None = None) -> list[dict]:
    with Path(path).open(encoding="utf-8") as f: rows=[json.loads(line) for line in f if line.strip()]
    if expected_split and any(r["split"]!=expected_split for r in rows): raise ValueError("Wrong data split")
    return rows

if __name__=="__main__":
    p=argparse.ArgumentParser(); p.add_argument("--output",required=True); p.add_argument("--per-family",type=int,default=24)
    p.add_argument("--external",default=""); a=p.parse_args()
    print(json.dumps(build_corpus(a.output,a.per_family,external_jsonl=a.external),indent=2))


def verify_corpus_manifest(root: str | Path) -> dict:
    root=Path(root);m=json.loads((root/'manifest.json').read_text())
    if not m.get('file_hashes'): raise ValueError('Legacy corpus manifest: rebuild with v2 full content fingerprints')
    for n,h in m['file_hashes'].items():
        if Path(n).name!=n or file_hash(root/n)!=h: raise ValueError(f'Corpus file changed: {n}')
    return m


In [ ]:
%%writefile arc_exec_rl/curriculum.py
"""Training-only family sampler. Optimizes informative rollout opportunities,
not hidden-task routing. Exploration never drops below the configured fraction.
"""
from __future__ import annotations
from collections import defaultdict
import random

class FamilySampler:
    def __init__(self, records: list[dict], seed: int, exploration: float=.25):
        if not records or not 0<exploration<=1: raise ValueError('Invalid sampler input')
        self.rows=records;self.groups=defaultdict(list)
        for i,r in enumerate(records): self.groups[r['family_id']].append(i)
        self.families=sorted(self.groups);self.rng=random.Random(seed);self.exploration=exploration
        self.stats={k:{'count':0,'reward_sum':0.,'seconds_sum':0.} for k in self.families}
    def choose(self) -> dict:
        if self.rng.random()<self.exploration:
            family=self.rng.choice(self.families)
        else:
            weights=[]
            for k in self.families:
                s=self.stats[k];p=(s['reward_sum']+1.)/(s['count']+2.)
                # Proxy for mixed-success groups; smoothed observed execution cost.
                cost=(s['seconds_sum']+30.)/(s['count']+1.)
                weights.append((.02+4*p*(1-p))/(max(1.,cost)**.5))
            family=self.rng.choices(self.families,weights=weights,k=1)[0]
        return self.rows[self.rng.choice(self.groups[family])]
    def update(self,family: str,rewards: list[float],seconds: float):
        if family not in self.stats or not rewards or any(not 0<=r<=1 for r in rewards): raise ValueError('Bad reward statistics')
        s=self.stats[family];s['count']+=1;s['reward_sum']+=sum(rewards)/len(rewards);s['seconds_sum']+=max(0.,seconds)
    def state(self) -> dict:
        return {'families':self.families,'exploration':self.exploration,'stats':self.stats,'rng':self.rng.getstate()}
    def restore(self,state: dict):
        if state['families']!=self.families or state['exploration']!=self.exploration: raise ValueError('Sampler identity changed')
        def tup(x): return tuple(tup(t) for t in x) if isinstance(x,(list,tuple)) else x
        self.stats=state['stats'];self.rng.setstate(tup(state['rng']))


In [ ]:
%%writefile arc_exec_rl/dsl.py
"""Bounded, allow-listed JSON program interpreter. Never eval/exec model text.
This is a finite program language, NOT unrestricted Python program synthesis.
Every intermediate grid remains an exact <=30 x <=30 integer grid.
"""
from __future__ import annotations
import json
from collections import Counter, deque
from typing import Any
import numpy as np
from .common import validate_grid, json_hash

MAX_STEPS = 8
MAX_PROGRAM_BYTES = 8192
DSL_VERSION = "arc-json-2"


def mode(a: np.ndarray) -> int:
    count = Counter(map(int, a.ravel()))
    return min(count, key=lambda c: (-count[c], c))


def background(a: np.ndarray, value: int | str = "mode") -> int:
    if value == "mode":
        return mode(a)
    if type(value) is int and 0 <= value <= 9:
        return value
    raise ValueError("background is mode or a color")


def components(a: np.ndarray, bg: int, diagonal: bool = False, monochrome: bool = True) -> list[list[tuple[int,int]]]:
    h, w = a.shape
    seen: set[tuple[int,int]] = set()
    offsets = [(-1,0), (1,0), (0,-1), (0,1)]
    if diagonal:
        offsets += [(-1,-1),(-1,1),(1,-1),(1,1)]
    result = []
    for r in range(h):
        for c in range(w):
            if a[r,c] == bg or (r,c) in seen:
                continue
            seen.add((r,c)); q = deque([(r,c)]); obj = []
            while q:
                x, y = q.popleft(); obj.append((x,y))
                for dx,dy in offsets:
                    nx,ny = x+dx,y+dy
                    if 0 <= nx < h and 0 <= ny < w and (nx,ny) not in seen and a[nx,ny] != bg:
                        if not monochrome or a[nx,ny] == a[r,c]:
                            seen.add((nx,ny)); q.append((nx,ny))
            result.append(sorted(obj))
    return result


def bbox(cells: list[tuple[int,int]]) -> tuple[int,int,int,int]:
    if not cells:
        raise ValueError("Empty object")
    rr,cc = zip(*cells)
    return tuple(map(int, (min(rr),min(cc),max(rr)+1,max(cc)+1)))


def enclosed(mask: np.ndarray) -> list[list[tuple[int,int]]]:
    """Connected regions of False NOT connected to image border (4-connectivity)."""
    h,w = mask.shape
    seen = set(); holes = []
    for r in range(h):
        for c in range(w):
            if mask[r,c] or (r,c) in seen:
                continue
            q=deque([(r,c)]); seen.add((r,c)); region=[]; edge=False
            while q:
                x,y=q.popleft(); region.append((x,y))
                edge |= x in (0,h-1) or y in (0,w-1)
                for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx,ny=x+dx,y+dy
                    if 0 <= nx<h and 0 <= ny<w and not mask[nx,ny] and (nx,ny) not in seen:
                        seen.add((nx,ny)); q.append((nx,ny))
            if not edge:
                holes.append(region)
    return holes


def object_holes(obj: list[tuple[int,int]]) -> int:
    r0,c0,r1,c1=bbox(obj); mask=np.zeros((r1-r0,c1-c0),bool)
    for r,c in obj: mask[r-r0,c-c0]=True
    return len(enclosed(mask))


def _int(v: Any, low: int, high: int) -> int:
    if type(v) is not int or not low <= v <= high:
        raise ValueError(f"Expected integer {low}..{high}")
    return v

SCHEMA = {
    "identity": (), "rotate": ("k",), "reflect": ("axis",),
    "crop": ("bg",), "recolor": ("mapping",), "scale": ("row", "col"),
    "tile": ("row", "col"), "reduce": ("row", "col"),
    "translate": ("dr", "dc", "bg"), "fill_holes": ("color", "bg"),
    "crop_object": ("selector", "bg", "connectivity", "monochrome"),
    "keep_object": ("selector", "bg", "connectivity", "monochrome"),
    "move_to_edge": ("direction", "bg"), "pack_objects": ("axis", "order", "gap", "bg"),
    "connect": ("axis", "color", "bg"), "mirror_concat": ("axis",),
    "crop_color": ("color",), "draw_bbox": ("color", "bg", "fill"),
    "hole_recolor": ("mapping", "bg"), "legend_holes": ("axis", "side", "target", "bg"),
}


def parse_program(text: str | dict) -> dict:
    if isinstance(text,str):
        if len(text.encode()) > MAX_PROGRAM_BYTES: raise ValueError("Program too long")
        t=text.strip()
        if t.startswith("```json\n") and t.endswith("```"): t=t[8:-3].strip()
        elif t.startswith("```\n") and t.endswith("```"): t=t[4:-3].strip()
        def unique(pairs):
            out={}
            for k,v in pairs:
                if k in out: raise ValueError("Duplicate JSON key")
                out[k]=v
            return out
        p=json.loads(t,object_pairs_hook=unique,parse_constant=lambda v: (_ for _ in ()).throw(ValueError(v)))
    else:
        p=json.loads(json.dumps(text,allow_nan=False))
    if not isinstance(p,dict) or set(p)!={"steps"} or not isinstance(p["steps"],list) or not 1<=len(p["steps"])<=MAX_STEPS:
        raise ValueError("Expected {steps:[1..8 operations]}")
    for s in p["steps"]:
        if not isinstance(s,dict) or s.get("op") not in SCHEMA: raise ValueError("Unknown operation")
        if set(s)-({"op"}|set(SCHEMA[s["op"]])): raise ValueError("Unknown operation argument")
    if len(json.dumps(p)) > MAX_PROGRAM_BYTES: raise ValueError("Program too long")
    return p


def choose_object(a: np.ndarray, s: dict) -> list[tuple[int,int]]:
    bg=background(a,s.get("bg","mode"))
    conn=_int(s.get("connectivity",4),4,8)
    if conn not in (4,8): raise ValueError("connectivity must be 4 or 8")
    mono=s.get("monochrome",True)
    if type(mono) is not bool: raise ValueError("monochrome is boolean")
    objs=components(a,bg,conn==8,mono)
    if not objs: raise ValueError("No foreground object")
    rule=s.get("selector","largest")
    if rule=="largest": key=lambda o:(-len(o),bbox(o))
    elif rule=="smallest": key=lambda o:(len(o),bbox(o))
    elif rule=="top_left": key=lambda o:bbox(o)
    elif rule=="most_holes": key=lambda o:(-object_holes(o),bbox(o))
    elif rule=="least_holes": key=lambda o:(object_holes(o),bbox(o))
    else: raise ValueError("Unknown selector")
    return min(objs,key=key)


def apply_op(a: np.ndarray, s: dict) -> np.ndarray:
    op=s["op"]; bg=background(a,s.get("bg","mode")); h,w=a.shape
    if op=="identity": return a.copy()
    if op=="rotate": return np.rot90(a,_int(s.get("k",1),0,3)).copy()
    if op=="reflect":
        axis=s.get("axis","h")
        if axis=="h": return np.fliplr(a).copy()
        if axis=="v": return np.flipud(a).copy()
        if axis=="d": return a.T.copy()
        raise ValueError("Unknown reflect axis")
    if op in ("crop","crop_color"):
        mask=(a!=bg) if op=="crop" else (a==_int(s["color"],0,9))
        cells=list(zip(*np.where(mask))); r0,c0,r1,c1=bbox(cells)
        return a[r0:r1,c0:c1].copy()
    if op=="recolor":
        m=s.get("mapping",{}); out=a.copy()
        if not isinstance(m,dict): raise ValueError("mapping must be object")
        for k,v in m.items():
            if k not in list("0123456789"): raise ValueError("mapping keys are colors")
            out[a==int(k)]=_int(v,0,9)
        return out
    if op in ("scale","tile","reduce"):
        rh=_int(s.get("row",2),1,5); cw=_int(s.get("col",2),1,5)
        if op!="reduce" and (h*rh>30 or w*cw>30): raise ValueError("Output too large")
        if op=="scale": return np.repeat(np.repeat(a,rh,axis=0),cw,axis=1)
        if op=="tile": return np.tile(a,(rh,cw))
        if h%rh or w%cw: raise ValueError("Nonintegral reduction")
        out=np.empty((h//rh,w//cw),dtype=np.int8)
        for r in range(out.shape[0]):
            for c in range(out.shape[1]):
                b=a[r*rh:(r+1)*rh,c*cw:(c+1)*cw]
                if np.any(b!=b[0,0]): raise ValueError("Reduction requires uniform blocks")
                out[r,c]=b[0,0]
        return out
    if op=="translate":
        dr=_int(s.get("dr",0),-29,29); dc=_int(s.get("dc",0),-29,29); out=np.full_like(a,bg)
        # Out-of-bounds foreground is rejected, not silently lost.
        rr,cc=np.where(a!=bg)
        if np.any(rr+dr<0) or np.any(rr+dr>=h) or np.any(cc+dc<0) or np.any(cc+dc>=w): raise ValueError("Foreground would be clipped")
        out[rr+dr,cc+dc]=a[rr,cc]; return out
    if op=="fill_holes":
        out=a.copy(); col=_int(s.get("color",1),0,9)
        for hole in enclosed(a!=bg):
            for r,c in hole: out[r,c]=col
        return out
    if op in ("crop_object","keep_object"):
        obj=choose_object(a,s); out=np.full_like(a,bg)
        for r,c in obj: out[r,c]=a[r,c]
        if op=="keep_object": return out
        r0,c0,r1,c1=bbox(obj); return out[r0:r1,c0:c1].copy()
    if op=="move_to_edge":
        obj=list(zip(*np.where(a!=bg))); r0,c0,r1,c1=bbox(obj); d=s.get("direction","left")
        delta={"left":(0,-c0),"right":(0,w-c1),"up":(-r0,0),"down":(h-r1,0)}
        if d not in delta: raise ValueError("Bad direction")
        dr,dc=delta[d]; return apply_op(a,{"op":"translate","dr":dr,"dc":dc,"bg":bg})
    if op=="pack_objects":
        objs=components(a,bg); order=s.get("order","size_asc"); axis=s.get("axis","h"); gap=_int(s.get("gap",1),0,3)
        if not objs: raise ValueError("No objects")
        if order=="size_asc": objs.sort(key=lambda o:(len(o),bbox(o)))
        elif order=="size_desc": objs.sort(key=lambda o:(-len(o),bbox(o)))
        elif order=="position": objs.sort(key=bbox)
        elif order=="holes": objs.sort(key=lambda o:(object_holes(o),bbox(o)))
        else: raise ValueError("Bad pack ordering")
        patches=[]
        for obj in objs:
            r0,c0,r1,c1=bbox(obj); b=np.full((r1-r0,c1-c0),bg,np.int8)
            for r,c in obj: b[r-r0,c-c0]=a[r,c]
            patches.append(b)
        if axis=="v": patches=[p.T for p in patches]
        elif axis!="h": raise ValueError("Bad pack axis")
        oh=max(p.shape[0] for p in patches); ow=sum(p.shape[1] for p in patches)+gap*(len(patches)-1)
        if oh>30 or ow>30: raise ValueError("Packed output too large")
        out=np.full((oh,ow),bg,np.int8); x=0
        for p in patches: out[:p.shape[0],x:x+p.shape[1]]=p; x+=p.shape[1]+gap
        return out.T.copy() if axis=="v" else out
    if op=="connect":
        out=a.copy(); col=_int(s.get("color",1),0,9); axis=s.get("axis","h")
        if axis not in ("h","v","both"): raise ValueError("Bad connect axis")
        for transpose in ([False,True] if axis=="both" else [axis=="v"]):
            src=a.T if transpose else a; dst=out.T if transpose else out
            for r in range(src.shape[0]):
                pos=np.where(src[r]==col)[0]
                if len(pos)>=2:
                    for c in range(int(pos[0]),int(pos[-1])+1):
                        if src[r,c] in (bg,col): dst[r,c]=col
        return out
    if op=="mirror_concat":
        axis=s.get("axis","h")
        if axis=="h" and 2*w<=30: return np.concatenate([a,np.fliplr(a)],axis=1)
        if axis=="v" and 2*h<=30: return np.concatenate([a,np.flipud(a)],axis=0)
        raise ValueError("Invalid mirror concatenation")
    if op=="draw_bbox":
        out=a.copy(); r0,c0,r1,c1=bbox(list(zip(*np.where(a!=bg)))); col=_int(s.get("color",1),0,9)
        fill=s.get("fill",False)
        if type(fill) is not bool: raise ValueError("fill must be boolean")
        if fill: out[r0:r1,c0:c1]=col
        else:
            out[r0,c0:c1]=col; out[r1-1,c0:c1]=col; out[r0:r1,c0]=col; out[r0:r1,c1-1]=col
        return out
    if op=="hole_recolor":
        out=a.copy(); mapping=s.get("mapping",{})
        if not isinstance(mapping,dict): raise ValueError("mapping must be object")
        for obj in components(a,bg):
            key=str(object_holes(obj))
            if key in mapping:
                col=_int(mapping[key],0,9)
                for r,c in obj: out[r,c]=col
        return out
    if op=="legend_holes":
        # A monochrome separator splits local legend from target-color objects.
        axis=s.get("axis","h"); side=s.get("side","before"); target=_int(s.get("target",5),0,9)
        if axis not in ("h","v") or side not in ("before","after"): raise ValueError("Bad legend layout")
        src=a if axis=="h" else a.T; lines=[i for i,row in enumerate(src) if len(set(map(int,row)))==1 and row[0]!=bg]
        if len(lines)!=1: raise ValueError("Requires one full separator")
        cut=lines[0]
        leg=src[:cut] if side=="before" else src[cut+1:]
        field=src[cut+1:] if side=="before" else src[:cut]
        if not leg.size or not field.size: raise ValueError("Empty legend/field")
        mapping={}
        for obj in components(leg,bg):
            col=int(leg[obj[0]]); holes=object_holes(obj)
            if holes in mapping and mapping[holes]!=col: raise ValueError("Ambiguous legend")
            mapping[holes]=col
        changed=field.copy()
        for obj in components(field,bg):
            if int(field[obj[0]])!=target: continue
            holes=object_holes(obj)
            if holes not in mapping: raise ValueError("Legend missing key")
            for r,c in obj: changed[r,c]=mapping[holes]
        out=src.copy()
        if side=="before": out[cut+1:]=changed
        else: out[:cut]=changed
        return out if axis=="h" else out.T.copy()
    raise ValueError("Unimplemented operator")


def execute(program: str | dict, grid: list[list[int]]) -> list[list[int]]:
    p=parse_program(program); validate_grid(grid); a=np.array(grid,dtype=np.int8)
    for s in p["steps"]:
        a=apply_op(a,s)
        validate_grid(a.tolist())
    return a.tolist()


def replay(program: str | dict, pairs: list[dict]) -> list[bool]:
    result=[]
    for pair in pairs:
        try: result.append(execute(program,pair["input"])==pair["output"])
        except (ValueError,KeyError,TypeError,IndexError,OverflowError): result.append(False)
    return result


def program_family(program: str | dict) -> str:
    # Coarse composition signature, not a proof of semantic-family independence.
    p=parse_program(program)
    return "/".join(s["op"] + (":"+str(s.get("selector")) if "selector" in s else "") for s in p["steps"])


def normalized_program(program: str | dict) -> dict:
    """Remove only provably vacuous forms; no claim of complete equivalence checking."""
    p=parse_program(program);steps=[]
    for original in p['steps']:
        s=dict(original);op=s['op']
        if op=='identity': continue
        if op=='rotate' and s.get('k',1)==0: continue
        if op in ('scale','tile','reduce') and s.get('row',2)==1 and s.get('col',2)==1: continue
        if op=='translate' and s.get('dr',0)==0 and s.get('dc',0)==0: continue
        if op=='recolor':
            s['mapping']={k:v for k,v in s.get('mapping',{}).items() if str(v)!=k}
            if not s['mapping']: continue
        if steps and op=='rotate' and steps[-1]['op']=='rotate':
            k=(steps.pop().get('k',1)+s.get('k',1))%4
            if k: steps.append({'op':'rotate','k':k})
            continue
        if steps and op=='reflect' and steps[-1]['op']=='reflect' and steps[-1].get('axis','h')==s.get('axis','h'):
            steps.pop();continue
        steps.append(s)
    return {'steps':steps or [{'op':'identity'}]}


def program_id(program: str | dict) -> str:
    return json_hash(normalized_program(program))


In [ ]:
%%writefile arc_exec_rl/efficient.py
"""Bounded generation, task/program execution reuse, and memory-aware loss helpers.
Batches amortize calls; no claim that Hugging Face deduplicates identical prompt KV.
"""
from __future__ import annotations
import json
import time
from collections import OrderedDict
from .common import json_hash, public_task
from .dsl import parse_program


def trim_completion(ids: list[int], eos: int | list[int], stop_length: int | None=None) -> tuple[list[int],bool]:
    ends={eos} if isinstance(eos,int) else set(eos)
    values=list(ids if stop_length is None else ids[:stop_length])
    for i,t in enumerate(values):
        if t in ends: return values[:i+1],True
    return values,False


class RecordTokenCache:
    """Bounded in-process cache keyed by whole record and formatting settings.
    Fresh RL queries are never reused as an old prompt. No target enters render_prompt.
    """
    def __init__(self, tokenizer, max_entries: int=8192):
        self.tokenizer=tokenizer;self.max_entries=max_entries;self.cache=OrderedDict();self.hits=0;self.misses=0
    def get(self, record: dict, config: dict):
        from .prompts import render_prompt
        key=json_hash([record,config['relational'],config['max_prompt_tokens'],config['max_completion_tokens'],config['max_sequence_tokens']])
        if key in self.cache:
            self.hits+=1;self.cache.move_to_end(key);return self.cache[key]
        text,n,rel=render_prompt(self.tokenizer,record['task'],config['relational'],config['max_prompt_tokens'])
        prompt=self.tokenizer.encode(text,add_special_tokens=False)
        reply=self.tokenizer.encode(json.dumps(record['program'],separators=(',',':')),add_special_tokens=False)+[self.tokenizer.eos_token_id]
        if len(reply)>config['max_completion_tokens'] or len(prompt)+len(reply)>config['max_sequence_tokens']:
            raise ValueError('Supervised program exceeds length cap; no silent truncation')
        answer=(prompt,reply,rel);self.cache[key]=answer;self.misses+=1
        if len(self.cache)>self.max_entries: self.cache.popitem(last=False)
        return answer


def generate_programs(model,tokenizer,prompt: list[int],count: int,deadline: float,
                      device, max_new_tokens: int=512, microbatch: int=2,
                      sample: bool=True,temperature: float=1.,top_p: float=1.,
                      stop_on_json: bool=True,json_check_interval: int=8) -> tuple[list[dict],dict]:
    """HF generation with explicit sampler distribution and bounded OOM backoff.
    Completed JSON is an optional deterministic stopping event, not a logit mask.
    RL calls MUST use temperature=1/top_p=1; inference can use another policy.
    """
    import torch
    from transformers import GenerationConfig,StoppingCriteria,StoppingCriteriaList
    if not prompt or count<1 or microbatch<1 or max_new_tokens<1: raise ValueError('Invalid generation budget')
    if temperature<=0 or not 0<top_p<=1: raise ValueError('Invalid sampling controls')
    outputs=[];batch_size=min(count,microbatch);calls=0;backoffs=0;batch_trace=[]
    was_training=model.training;checkpointing=bool(getattr(model,'is_gradient_checkpointing',False))
    if checkpointing: model.gradient_checkpointing_disable()
    model.eval()
    class Stop(StoppingCriteria):
        def __init__(self,b): self.lengths=[None]*b;self.reasons=[None]*b
        def __call__(self,input_ids,scores,**kwargs):
            n=input_ids.shape[1]-len(prompt);done=torch.zeros(input_ids.shape[0],dtype=torch.bool,device=input_ids.device)
            expired=time.monotonic()>=deadline
            if expired or (stop_on_json and (n%json_check_interval==0 or n==max_new_tokens)):
                for i in range(input_ids.shape[0]):
                    if self.lengths[i] is not None: done[i]=True;continue
                    if expired: self.lengths[i]=n;self.reasons[i]='deadline';done[i]=True;continue
                    ids=input_ids[i,len(prompt):].tolist();ids,_=trim_completion(ids,tokenizer.eos_token_id)
                    try: parse_program(tokenizer.decode(ids,skip_special_tokens=True))
                    except (ValueError,TypeError,KeyError): continue
                    self.lengths[i]=n;self.reasons[i]='complete_json';done[i]=True
            for i,length in enumerate(self.lengths):
                if length is not None: done[i]=True
            return done
    try:
        while len(outputs)<count and time.monotonic()<deadline:
            b=min(batch_size,count-len(outputs));stop=Stop(b)
            inp=torch.tensor([prompt],dtype=torch.long,device=device)
            # Build from scratch: no inherited forced-EOS, min-length, repetition or top-k settings.
            g=GenerationConfig(do_sample=sample,num_beams=1,num_return_sequences=b if sample else 1,
                max_new_tokens=max_new_tokens,temperature=temperature if sample else 1.,
                top_p=top_p if sample else 1.,top_k=0,repetition_penalty=1.,
                eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id,
                use_cache=True,renormalize_logits=False)
            if not sample: b=1;stop=Stop(1)
            rng=torch.get_rng_state();cuda_rng=torch.cuda.get_rng_state(device) if str(device).startswith('cuda') else None
            try:
                with torch.inference_mode():
                    y=model.generate(input_ids=inp,attention_mask=torch.ones_like(inp),generation_config=g,
                        logits_to_keep=1,stopping_criteria=StoppingCriteriaList([stop]))
                calls+=1;batch_trace.append(b)
                for i,row in enumerate(y):
                    ids,eos=trim_completion(row[len(prompt):].tolist(),tokenizer.eos_token_id,stop.lengths[i])
                    text=tokenizer.decode(ids,skip_special_tokens=True)
                    try: parse_program(text);complete=True
                    except (ValueError,TypeError,KeyError): complete=False
                    reason='eos' if eos else stop.reasons[i] or 'length_limit'
                    terminal=bool(eos or (complete and reason!='deadline'))
                    outputs.append({'ids':ids,'text':text,'terminated':terminal,'complete_json':complete,'stop_reason':reason})
                del y
            except torch.cuda.OutOfMemoryError:
                # No gradient update was performed. Restore RNG and retry at a smaller batch.
                torch.set_rng_state(rng)
                if cuda_rng is not None: torch.cuda.set_rng_state(cuda_rng,device)
                torch.cuda.empty_cache();backoffs+=1
                if b==1: raise
                batch_size=max(1,b//2)
            finally: del inp
    finally:
        if checkpointing: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        model.train(was_training)
    return outputs,{'generation_calls':calls,'batch_sizes':batch_trace,'oom_backoffs':backoffs,
                    'requested':count,'returned':len(outputs),'forced_eos':False,
                    'sampler':{'temperature':temperature if sample else None,'top_p':top_p if sample else None,'top_k':0}}


In [ ]:
%%writefile arc_exec_rl/harvest.py
"""Optional bounded program supervision mined ONLY from official TRAINING pairs.
No evaluation file is opened. Fitting these finite pairs is not proof of a universal rule.
"""
from __future__ import annotations
import json,time
from pathlib import Path
import numpy as np
from .common import public_task,atomic_json,json_hash
from .dsl import execute,program_family,program_id
from .corpus import split_for_family,validate_record


def program_proposals(task: dict,max_programs: int=240):
    demos=task['train']; first=np.asarray(demos[0]['input']);bg=int(np.bincount(first.ravel(),minlength=10).argmax())
    colors=sorted({v for p in demos for row in p['output'] for v in row})
    ops=[{'op':'identity'}]+[{'op':'rotate','k':k} for k in (1,2,3)]+[{'op':'reflect','axis':a} for a in ('h','v','d')]
    for b in sorted({0,bg}):
        ops.append({'op':'crop','bg':b})
        for selector in ('largest','smallest','top_left','most_holes','least_holes'):
            for conn in (4,8): ops.append({'op':'crop_object','selector':selector,'bg':b,'connectivity':conn})
        for d in ('left','right','up','down'):ops.append({'op':'move_to_edge','direction':d,'bg':b})
        for col in colors:
            ops.append({'op':'fill_holes','color':col,'bg':b})
            for a in ('h','v','both'):ops.append({'op':'connect','axis':a,'color':col,'bg':b})
        for a in ('h','v'):
            for order in ('size_asc','size_desc','position','holes'):
                for gap in (0,1):ops.append({'op':'pack_objects','axis':a,'order':order,'gap':gap,'bg':b})
    for a in ('h','v'):ops.append({'op':'mirror_concat','axis':a})
    y=demos[0]['output'];h=len(demos[0]['input']);w=len(demos[0]['input'][0]);oh=len(y);ow=len(y[0])
    if oh%h==0 and ow%w==0 and 1<=oh//h<=5 and 1<=ow//w<=5:
        for op in ('scale','tile'):ops.append({'op':op,'row':oh//h,'col':ow//w})
    if h%oh==0 and w%ow==0 and 1<=h//oh<=5 and 1<=w//ow<=5:
        ops.append({'op':'reduce','row':h//oh,'col':w//ow})
    programs=[{'steps':[x]} for x in ops]
    for b in sorted({0,bg}):
        for k in (1,2,3): programs.append({'steps':[{'op':'crop','bg':b},{'op':'rotate','k':k}]})
        for a in ('h','v','d'):programs.append({'steps':[{'op':'crop','bg':b},{'op':'reflect','axis':a}]})
    yield from programs[:max_programs]


def fit_color_map(p: dict,demos: list[dict]) -> dict | None:
    m={}
    try:
        for pair in demos:
            got=execute(p,pair['input']);y=pair['output']
            if len(got)!=len(y) or len(got[0])!=len(y[0]):return None
            for row,out in zip(got,y):
                for a,b in zip(row,out):
                    if a in m and m[a]!=b:return None
                    m[a]=b
    except (ValueError,KeyError,TypeError,IndexError):return None
    if all(a==b for a,b in m.items()):return p
    return {'steps':p['steps']+[{'op':'recolor','mapping':{str(a):b for a,b in m.items() if a!=b}}]}


def harvest_training(data_dir: str,output_jsonl: str,max_seconds: float=180.,max_programs: int=240) -> dict:
    root=Path(data_dir); cp=root/'arc-agi_training_challenges.json';sp=root/'arc-agi_training_solutions.json'
    tasks=json.loads(cp.read_text());sols=json.loads(sp.read_text())
    if set(tasks)!=set(sols):raise ValueError('Training challenges and solutions disagree')
    deadline=time.monotonic()+max_seconds;records=[];seen=0;skipped_heldout_families=0
    for key in sorted(tasks):
        if time.monotonic()>=deadline:break
        task=public_task(tasks[key]);seen+=1
        if len(task['train'])<2:continue
        for p in program_proposals(task,max_programs):
            if time.monotonic()>=deadline:break
            fitted=fit_color_map(p,task['train'])
            if not fitted:continue
            family=program_family(fitted)
            # Never pull a held-out bootstrap operator composition back into training.
            if split_for_family(family)!='train':skipped_heldout_families+=1;continue
            try:
                if not all(execute(fitted,pair['input'])==y for pair,y in zip(task['test'],sols[key])):continue
                record={'record_id':'train-'+key,'family_id':family,'split':'train','source':'official-training-program-harvest',
                        'source_split':'training','license':'Apache-2.0','source_task_ids':[key],
                        'task':task,'query_outputs':sols[key],'program':fitted,
                        'verification':'exact-fit-all-public-training-pairs-not-independent-generator-proof'}
                records.append(validate_record(record));break
            except (ValueError,KeyError,TypeError,IndexError):continue
    p=Path(output_jsonl);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w') as f:
        for r in records:f.write(json.dumps(r,separators=(',',':'))+'\n')
    result={'training_tasks_examined':seen,'program_supervision_records':len(records),
            'heldout_compositions_rejected':skipped_heldout_families,
            'warning':'Only training supervision. Not an evaluation score or comprehensive ARC solver.'}
    atomic_json(p.with_suffix('.report.json'),result);return result


In [ ]:
%%writefile arc_exec_rl/hybrid.py
"""Standalone program stage and guarded merge with a frozen anchor submission.
Public solutions are never opened by generation/merge. Evaluation is a separate caller.
"""
from __future__ import annotations
import argparse
import copy
import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path
import numpy as np
from .common import atomic_json,append_event,file_hash,json_hash,public_task,validate_grid,validate_submission,score_submission
from .infer import DEFAULT_INFER,infer_task
from .dsl import DSL_VERSION


def task_cost(task: dict) -> int:
    return sum(len(p['input'])*len(p['input'][0])+len(p['output'])*len(p['output'][0]) for p in task['train'])+sum(len(p['input'])*len(p['input'][0]) for p in task['test'])


def balanced_shards(tasks: dict,n: int) -> list[list[str]]:
    shards=[[] for _ in range(n)]; costs=[0]*n
    for key in sorted(tasks,key=lambda k:(-task_cost(tasks[k]),k)):
        i=min(range(n),key=lambda j:costs[j]); shards[i].append(key);costs[i]+=task_cost(tasks[key])
    return shards


def merge_proposal(tasks: dict,anchor: dict,records: dict,min_program_agreement: int=2) -> tuple[dict,dict]:
    validate_submission(tasks,anchor); out=copy.deepcopy(anchor); changed=[]; candidate_bank={}
    for key,task in tasks.items():
        record=records.get(key,{"banks":[]})
        for i in range(len(task['test'])):
            bank=record.get('banks',[])
            choices=bank[i] if i<len(bank) else []
            candidate_bank[f'{key}:{i}']=[x['grid'] for x in choices]
            for c in choices:
                validate_grid(c['grid'])
                if c.get('demo_exact',0)!=len(task['train']) or len(task['train'])<2: continue
                if len(set(c.get('program_ids',[])))<min_program_agreement: continue
                if c['grid']==out[key][i]['attempt_1']: continue
                if c['grid']!=out[key][i]['attempt_2']:
                    out[key][i]['attempt_2']=c['grid']; changed.append(f'{key}:{i}')
                break
    validate_submission(tasks,out)
    assert all(out[k][i]['attempt_1']==anchor[k][i]['attempt_1'] for k,t in tasks.items() for i in range(len(t['test'])))
    return out,{"changed_attempt2":changed,"attempt1_changed":0,"candidate_bank":candidate_bank}


def implementation_id() -> str:
    """Promotions expire whenever the executor, generation or selection implementation changes."""
    here=Path(__file__).parent
    names=('dsl.py','infer.py','efficient.py','prompts.py','hybrid.py','common.py')
    return json_hash({n:file_hash(here/n) for n in names})


def gate_valid(gate: dict | None,weights_id: str,infer_config_hash: str) -> bool:
    return bool(gate and gate.get('passed') is True and gate.get('weights_id')==weights_id
                and gate.get('infer_config_hash')==infer_config_hash and gate.get('dsl_version')==DSL_VERSION
                and gate.get('scope')=='hybrid-paired-development'
                and gate.get('implementation_id')==implementation_id()
                and gate.get('validation_eligible') is True)


def assess_hybrid(tasks: dict,solutions: dict,anchor: dict,proposed: dict,families: dict,
                  weights_id: str,infer_cfg: dict,source_split: str,validation_manifest: dict | None=None) -> dict:
    if source_split not in ('synthetic-development','family-heldout-development'):
        raise ValueError('Public evaluation is a diagnostic, not a clean hybrid promotion set')
    if set(tasks)!=set(families): raise ValueError('Family manifest required for every task')
    a=score_submission(tasks,solutions,anchor);b=score_submission(tasks,solutions,proposed)
    fam={}
    for k in tasks: fam.setdefault(families[k],[]).append(b['per_task'][k]['fraction']-a['per_task'][k]['fraction'])
    # Equal-weight family means; do not pretend correlated pairs are independent trials.
    deltas=np.asarray([np.mean(v) for v in fam.values()],float)
    rng=np.random.default_rng(391)
    boot=np.asarray([rng.choice(deltas,len(deltas),replace=True).mean() for _ in range(1000)])
    lower=float(np.quantile(boot,.10)); gain=b['pass2_exact']-a['pass2_exact']
    changed1=sum(anchor[k][i]['attempt_1']!=proposed[k][i]['attempt_1'] for k,t in tasks.items() for i in range(len(t['test'])))
    vm=validation_manifest or {}
    validation_eligible=bool(vm.get('tasks_hash')==json_hash(tasks) and vm.get('bootstrap_only') is False
        and vm.get('training_source_records',1)==0 and vm.get('source_count',0)>=2)
    passed=validation_eligible and len(tasks)>=64 and len(fam)>=12 and gain>=3 and sum(deltas>0)>=3 and lower>0 and changed1==0
    return {"passed":bool(passed),"scope":"hybrid-paired-development","weights_id":weights_id,
            "implementation_id":implementation_id(),"validation_eligible":validation_eligible,
            "validation_manifest_hash":json_hash(vm),
            "infer_config_hash":json_hash({**DEFAULT_INFER,**infer_cfg}),"dsl_version":DSL_VERSION,
            "tasks":len(tasks),"families":len(fam),"exact_pair_gain":gain,
            "family_mean_delta":float(deltas.mean()),"one_sided_90_bootstrap_lower_delta":lower,
            "attempt1_changes":changed1,"source_split":source_split,
            "limitation":"Pilot heuristic plus cluster bootstrap; not a probability of 60% hidden accuracy",
            "anchor_report":a,"proposed_report":b}


def run_worker(work_file: str,rank: int):
    # CUDA_VISIBLE_DEVICES is assigned by parent BEFORE this process imports torch.
    from .modeling import load_program_model
    cfg=json.loads(Path(work_file).read_text());root=Path(cfg['output']); event=root/f'worker-{rank}.jsonl'
    try:
        model,tok=load_program_model(cfg['model'],'cuda:0'); append_event(event,'model_loaded',rank=rank)
        tasks=json.loads(Path(cfg['challenges']).read_text())
        for key in cfg['shards'][rank]:
            remaining=cfg['end_time']-time.time()
            if remaining<=1: break
            start=time.time(); r=infer_task(model,tok,tasks[key],cfg['infer'],time.monotonic()+remaining,key)
            atomic_json(root/'records'/f'{key}.json',r)
            append_event(event,'task_complete',task=key,status=r['status'],seconds=time.time()-start)
    except Exception as exc:
        append_event(event,'worker_failed',error_type=type(exc).__name__,error=str(exc));raise


def run_stage(challenges_path: str,anchor_path: str,model_path: str,output: str,infer_cfg: dict,
              end_time: float,nprocs: int=4,apply_if_promoted: bool=True) -> dict:
    from .modeling import audit_program_model
    root=Path(output);root.mkdir(parents=True,exist_ok=True); (root/'records').mkdir(exist_ok=True)
    tasks={k:public_task(v) for k,v in json.loads(Path(challenges_path).read_text()).items()}
    anchor=json.loads(Path(anchor_path).read_text());validate_submission(tasks,anchor)
    manifest=json.loads((Path(model_path)/'model_manifest.json').read_text())
    if manifest.get('architecture')!='qwen3-program-dsl' or manifest.get('dsl_version')!=DSL_VERSION or not manifest.get('reload_verified'):
        raise ValueError('Only a reload-verified trained P1 export is accepted')
    # Check immutable file contents once in parent, not once per GPU worker.
    for name,digest in manifest['files'].items():
        if file_hash(Path(model_path)/name)!=digest: raise ValueError(f'P1 export checksum mismatch: {name}')
    audit_program_model(model_path)
    conf={**DEFAULT_INFER,**infer_cfg}; chash=json_hash(conf)
    identity={"tasks_hash":json_hash(tasks),"weights_id":manifest['weights_id'],"config_hash":chash,
              "implementation_id":implementation_id(),"anchor_submission_hash":json_hash(anchor)}
    state=root/'run_identity.json'
    if state.exists() and json.loads(state.read_text())!=identity: raise ValueError('Output directory contains incompatible previous run')
    atomic_json(state,identity)
    gate_path=Path(model_path)/'hybrid_promotion.json'
    gate=json.loads(gate_path.read_text()) if gate_path.exists() else None
    approved=gate_valid(gate,manifest['weights_id'],chash)
    if apply_if_promoted and not approved:
        # A serious submission must not waste its specialist reservation on an unvalidated lane.
        report={"status":"NOT_RUN_NO_HYBRID_PROMOTION","approved":False,"weights_id":manifest['weights_id']}
        atomic_json(root/'stage_report.json',report);return report
    todo={k:v for k,v in tasks.items() if not (root/'records'/f'{k}.json').exists()}
    shards=balanced_shards(todo,nprocs)
    # A pure inference view written here prevents worker access to any inline test answers.
    clean=root/'challenges_only.json';atomic_json(clean,tasks)
    work={"model":str(Path(model_path).resolve()),"challenges":str(clean.resolve()),"output":str(root.resolve()),
          "infer":conf,"end_time":end_time,"shards":shards}
    atomic_json(root/'work.json',work);ps=[];handles=[]
    try:
        for rank in range(nprocs):
            if not shards[rank] or time.time()>=end_time: continue
            log=(root/f'worker-{rank}.log').open('w');handles.append(log)
            env=dict(os.environ);env.update(CUDA_VISIBLE_DEVICES=str(rank),HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',
                             TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='2')
            ps.append(subprocess.Popen([sys.executable,'-m','arc_exec_rl.hybrid','worker','--work',str((root/'work.json').resolve()),'--rank',str(rank)],
                           env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True))
        while any(p.poll() is None for p in ps) and time.time()<end_time: time.sleep(1)
    finally:
        for p in ps:
            if p.poll() is None:
                try: os.killpg(p.pid,signal.SIGTERM)
                except ProcessLookupError: pass
        for p in ps:
            try: p.wait(timeout=15)
            except subprocess.TimeoutExpired:
                try: os.killpg(p.pid,signal.SIGKILL)
                except ProcessLookupError: pass
                p.wait()
        for f in handles: f.close()
    records={}
    for key in tasks:
        path=root/'records'/f'{key}.json'
        if path.exists():
            try: records[key]=json.loads(path.read_text())
            except (ValueError,OSError): pass
    proposed,merge=merge_proposal(tasks,anchor,records)
    atomic_json(root/'proposed_submission.json',proposed)
    final=proposed if apply_if_promoted and approved else anchor
    atomic_json(root/'final_submission.json',final)
    report={"status":"COMPLETE_WITH_PARTIALS_ALLOWED","approved":approved,"applied":apply_if_promoted and approved,
            "tasks_completed":len(records),"task_count":len(tasks),"worker_exit_codes":[p.returncode for p in ps],
            "weights_id":manifest['weights_id'],"infer_config_hash":chash,**merge}
    atomic_json(root/'stage_report.json',report);return report

if __name__=='__main__':
    p=argparse.ArgumentParser();sub=p.add_subparsers(dest='command',required=True)
    w=sub.add_parser('worker');w.add_argument('--work',required=True);w.add_argument('--rank',type=int,required=True)
    s=sub.add_parser('run');s.add_argument('--config',required=True)
    a=p.parse_args()
    if a.command=='worker': run_worker(a.work,a.rank)
    else:
        cfg=json.loads(Path(a.config).read_text());print(json.dumps(run_stage(**cfg),indent=2))


In [ ]:
%%writefile arc_exec_rl/infer.py
from __future__ import annotations
import argparse
import json
import os
import random
import time
from collections import Counter
from pathlib import Path
from .common import atomic_json,append_event,json_hash,public_task,stable_seed,score_submission,score_banks
from .dsl import parse_program,execute,replay,program_id,DSL_VERSION
from .prompts import render_prompt

DEFAULT_INFER={"max_prompt_tokens":7000,"max_new_tokens":512,"samples":8,
               "temperature":.7,"top_p":.9,"per_task_seconds":120.,"relational":True,"repair_rounds":1,
               "microbatch":2,"stop_on_complete_json":True,"json_check_interval":8}


def rank_programs(task: dict, programs: list[dict | str]) -> tuple[list[list[dict]], dict]:
    """Canonical output groups. Demonstration fit is evidence, NOT proof of hidden correctness."""
    task=public_task(task); accepted={}; failed=0
    for text in programs:
        try:
            p=parse_program(text); pid=program_id(p)
            if pid in accepted: continue
            if not all(replay(p,task["train"])): failed+=1; continue
            grids=[execute(p,t["input"]) for t in task["test"]]
            accepted[pid]=(p,grids)
        except (ValueError,KeyError,TypeError,IndexError,RecursionError): failed+=1
    banks=[]
    for ti in range(len(task["test"])):
        grouped={}
        for pid,(p,ys) in accepted.items():
            h=json_hash(ys[ti]); rec=grouped.setdefault(h,{"grid":ys[ti],"program_ids":[],"programs":[],"min_steps":99})
            rec["program_ids"].append(pid); rec["programs"].append(p); rec["min_steps"]=min(rec["min_steps"],len(p["steps"]))
        order=sorted(grouped.values(),key=lambda r:(-len(r["program_ids"]),r["min_steps"],json_hash(r["grid"])))
        for r in order:
            r["demo_exact"]=len(task["train"]);r["selection_evidence"]="demo_fit_and_program_agreement_not_calibrated_probability"
        banks.append(order)
    return banks,{"unique_programs_fit":len(accepted),"rejected":failed}


def repair_message(text: str,task: dict) -> str:
    # Only the public demonstration outputs can appear in repair feedback.
    try:
        p=parse_program(text)
        for i,pair in enumerate(task["train"]):
            try: got=execute(p,pair["input"])
            except Exception as exc: return f"Program failed on demonstration {i}: {type(exc).__name__}. Return corrected JSON only."
            if got!=pair["output"]:
                return f"Program failed demonstration {i}. Produced {json.dumps(got)}. Expected {json.dumps(pair['output'])}. Return corrected JSON only."
        return "All demonstrations fit. Propose one different compact rule, or repeat the same program. Return JSON only."
    except Exception as exc:
        return f"Invalid JSON DSL: {type(exc).__name__}. Return only a valid object with a steps array using the allowed operations."


def infer_task(model,tokenizer,task: dict,cfg: dict,deadline: float,task_id: str) -> dict:
    import torch
    from .efficient import generate_programs
    from .prompts import messages
    unknown=set(cfg)-set(DEFAULT_INFER)
    if unknown: raise ValueError(f'Unused inference settings: {sorted(unknown)}')
    c={**DEFAULT_INFER,**cfg};task=public_task(task);begin=time.monotonic();texts=[];events=[];stats=[]
    deadline=min(deadline,begin+c['per_task_seconds'])
    try: text,n,rel=render_prompt(tokenizer,task,c['relational'],c['max_prompt_tokens'])
    except ValueError as exc:
        return {'task_id':task_id,'status':'SKIPPED_CONTEXT_LIMIT','banks':[[] for _ in task['test']],'error':str(exc)}
    prompt=tokenizer.encode(text,add_special_tokens=False)
    torch.manual_seed(stable_seed('infer-v2',task_id));model.eval()
    device=next(model.parameters()).device
    def generate(ids,count,sample):
        got,diag=generate_programs(model,tokenizer,ids,count,deadline,device,
            c['max_new_tokens'],c['microbatch'],sample,
            c['temperature'] if sample else 1.,c['top_p'] if sample else 1.,
            c['stop_on_complete_json'],c['json_check_interval'])
        stats.append(diag);return [x['text'] for x in got]
    try:
        if time.monotonic()<deadline: texts.extend(generate(prompt,1,False))
        if c['samples']>1 and time.monotonic()<deadline: texts.extend(generate(prompt,c['samples']-1,True))
    except torch.cuda.OutOfMemoryError:
        events.append('OOM_AFTER_BATCH_BACKOFF');torch.cuda.empty_cache()
    except (RuntimeError,ValueError) as exc: events.append(type(exc).__name__+':'+str(exc)[:180])
    for _ in range(c['repair_rounds']):
        if time.monotonic()>=deadline or not texts: break
        banks,_=rank_programs(task,texts)
        if all(banks): break
        ms=messages(task,rel)+[{'role':'assistant','content':texts[-1][:4096]},
                               {'role':'user','content':repair_message(texts[-1],task)}]
        rep=tokenizer.apply_chat_template(ms,tokenize=False,add_generation_prompt=True)
        rp=tokenizer.encode(rep,add_special_tokens=False)
        if len(rp)>c['max_prompt_tokens']: break
        try: texts.extend(generate(rp,1,False))
        except (RuntimeError,ValueError) as exc:
            events.append('repair_failed:'+type(exc).__name__);break
    banks,summary=rank_programs(task,texts)
    return {'task_id':task_id,'status':'CANDIDATES' if any(banks) else 'NO_DEMO_FITTING_PROGRAM',
            'banks':banks,'program_texts':texts,'prompt_tokens':n,'relational_used':rel,
            'seconds':time.monotonic()-begin,'events':events,'generation_stats':stats,**summary}


def evaluated_records_fingerprint(rows: list[dict]) -> str:
    """Bind evaluation to every prompt, reference answer and provenance field."""
    return json_hash(sorted((r['record_id'],json_hash(r)) for r in rows))


def evaluate_records(model_path: str,records_path: str,output: str,cfg: dict | None=None,limit: int=0,device="cuda:0") -> dict:
    from .corpus import read_records
    from .modeling import load_program_model
    root=Path(output); root.mkdir(parents=True,exist_ok=True)
    rows=read_records(records_path)
    # Stable prefix, not a cherry-picked subset. Explicit dataset/config hashes travel with results.
    rows=sorted(rows,key=lambda r:r["record_id"])
    if limit: rows=rows[:limit]
    if not rows: raise ValueError("No evaluation records")
    if len({r['split'] for r in rows})!=1: raise ValueError('Mixed evaluation splits')
    from .hybrid import implementation_id
    model,tok=load_program_model(model_path,device)
    conf={**DEFAULT_INFER,**(cfg or {})}
    tasks={r["record_id"]:public_task(r["task"]) for r in rows}; sols={r["record_id"]:r["query_outputs"] for r in rows}
    pred={}; candidates={}; status={}; families={r["record_id"]:r["family_id"] for r in rows}
    for r in rows:
        key=r["record_id"]
        result=infer_task(model,tok,r["task"],conf,time.monotonic()+conf["per_task_seconds"],key)
        atomic_json(root/f"{key}.json",result); status[result["status"]]=status.get(result["status"],0)+1
        pairs=[]
        for i,bank in enumerate(result["banks"]):
            grids=[p["grid"] for p in bank]
            candidates[f"{key}:{i}"]=grids
            # An empty result is scored as WRONG, not rescued using the reference program.
            pairs.append({"attempt_1":grids[0] if grids else [[0]],"attempt_2":grids[1] if len(grids)>1 else grids[0] if grids else [[0]]})
        pred[key]=pairs
    result=score_banks(tasks,sols,candidates)
    m=json.loads((Path(model_path)/"model_manifest.json").read_text())
    result.update({"weights_id":m["weights_id"],"dsl_version":DSL_VERSION,"config":conf,"config_hash":json_hash(conf),
                   "evaluated_records_hash":evaluated_records_fingerprint(rows),
                   "implementation_id":implementation_id(),
                   "families":families,"split":rows[0]["split"],"status_counts":status,
                   "protocol":"program_only_no_oracle_injection", "bootstrap_only":all(r['source']=='independent-bootstrap-v1' for r in rows)})
    atomic_json(root/"submission.json",pred); atomic_json(root/"report.json",result)
    return result


def compare_reports(base: dict,candidate: dict) -> dict:
    if not base.get('implementation_id') or not candidate.get('implementation_id'):
        raise ValueError('Legacy evaluation lacks implementation fingerprint; rerun evaluation')
    for k in ("evaluated_records_hash","config_hash","dsl_version","protocol","split","implementation_id"):
        if base.get(k)!=candidate.get(k): raise ValueError(f"Incomparable evaluations: {k}")
    keys=set(base["per_task"])
    if keys!=set(candidate["per_task"]): raise ValueError("Task set mismatch")
    wins=[]; losses=[]; gain_families=set()
    for k in sorted(keys):
        delta=candidate["per_task"][k]["fraction"]-base["per_task"][k]["fraction"]
        if delta>0: wins.append(k); gain_families.add(candidate["families"][k])
        if delta<0: losses.append(k)
    pair_gain=candidate["pass2_exact"]-base["pass2_exact"]
    admitted=len(keys)>=32 and pair_gain>=3 and len(gain_families)>=3 and candidate["task_macro_pass2"]>base["task_macro_pass2"]
    return {"candidate_weights_id":candidate["weights_id"],"baseline_weights_id":base["weights_id"],
            "passed_pilot_gate":admitted,"pair_gain":pair_gain,"task_wins":wins,"task_losses":losses,
            "gain_families":sorted(gain_families),"scope":"P1 improvement only, NOT proof of hybrid/Kaggle gain",
            "bootstrap_only":candidate["bootstrap_only"],"dataset_hash":candidate["evaluated_records_hash"]}

if __name__=="__main__":
    p=argparse.ArgumentParser(); p.add_argument("--model",required=True); p.add_argument("--records",required=True); p.add_argument("--output",required=True)
    p.add_argument("--limit",type=int,default=0); p.add_argument("--config",default="")
    a=p.parse_args(); cfg=json.loads(Path(a.config).read_text()) if a.config else {}
    print(json.dumps(evaluate_records(a.model,a.records,a.output,cfg,a.limit),indent=2))


In [ ]:
%%writefile arc_exec_rl/kaggle_inputs.py
"""Bounded offline discovery for Kaggle mounts. A URL is not a filesystem path.
Does not download models or silently choose among ambiguous checkpoints.
"""
from __future__ import annotations
import importlib.metadata
import json
import os
import struct
import subprocess
import sys
import zipfile
from pathlib import Path
from .common import atomic_json, file_hash, public_task

DATA_FILES = ('arc-agi_training_challenges.json','arc-agi_training_solutions.json',
              'arc-agi_evaluation_challenges.json','arc-agi_evaluation_solutions.json',
              'arc-agi_test_challenges.json','sample_submission.json')
PACKAGES = {'transformers':'4.55.4','peft':'0.17.1','accelerate':'1.10.1'}


def scan(root: str | Path, names: set[str], limit: int=100000) -> list[Path]:
    root=Path(root); found=[]; count=0
    if not root.is_dir(): raise FileNotFoundError(f'Input root does not exist: {root}')
    for here,dirs,files in os.walk(root, followlinks=False):
        if len(Path(here).relative_to(root).parts)>=10: dirs[:]=[]
        dirs[:]=sorted(d for d in dirs if d not in ('.git','__pycache__','.cache'))
        for name in sorted(files):
            count+=1
            if count>limit: raise RuntimeError('Input scan cap reached; specify the exact mounted input')
            if name in names: found.append(Path(here)/name)
    return found


def tensor_headers(path: str | Path) -> dict:
    """Read only safetensors headers; detect missing/truncated tensors without loading GBs."""
    path=Path(path); size=path.stat().st_size
    with path.open('rb') as f:
        b=f.read(8)
        if len(b)!=8: raise ValueError(f'Truncated safetensors header: {path.name}')
        n=struct.unpack('<Q',b)[0]
        if not 2<=n<=16*1024*1024 or 8+n>size: raise ValueError('Invalid safetensors header size')
        h=json.loads(f.read(n))
    data_size=size-8-n
    widths={'F64':8,'F32':4,'F16':2,'BF16':2,'I64':8,'I32':4,'I16':2,'I8':1,'U8':1,'BOOL':1}
    intervals=[]; result={}
    for key,v in h.items():
        if key=='__metadata__': continue
        shape=v.get('shape'); off=v.get('data_offsets'); dtype=v.get('dtype')
        if not isinstance(shape,list) or any(type(x)is not int or x<0 for x in shape): raise ValueError('Bad tensor shape')
        if not isinstance(off,list) or len(off)!=2 or any(type(x)is not int for x in off) or not 0<=off[0]<=off[1]<=data_size:
            raise ValueError(f'Truncated/invalid tensor offsets: {key}')
        if dtype in widths:
            count=1
            for x in shape: count*=x
            if count*widths[dtype]!=off[1]-off[0]: raise ValueError(f'Tensor byte length mismatch: {key}')
        intervals.append((off[0],off[1],key));result[key]={'shape':shape,'dtype':dtype}
    end=0
    for begin,finish,key in sorted(intervals):
        if begin<end and finish>begin: raise ValueError(f'Overlapping tensor bytes: {key}')
        end=max(end,finish)
    if not result: raise ValueError('No tensors')
    return result


def inspect_weights(root: str | Path, cfg: dict) -> dict:
    root=Path(root); index=root/'model.safetensors.index.json'; mapping=None
    if index.exists():
        mapping=json.loads(index.read_text())['weight_map']; names=sorted(set(mapping.values()))
    else: names=[p.name for p in sorted(root.glob('model*.safetensors'))]
    if not names: raise ValueError('No base/merged safetensors weights (adapter alone is insufficient)')
    headers={}
    for n in names:
        if Path(n).name!=n: raise ValueError('Unsafe shard path')
        if not(root/n).is_file(): raise ValueError(f'Missing shard: {n}')
        h=tensor_headers(root/n)
        if set(headers)&set(h): raise ValueError('Duplicate tensors across shards')
        if mapping:
            for key in h:
                if mapping.get(key)!=n: raise ValueError('Index/shard contents disagree')
        headers.update(h)
    if mapping and set(headers)!=set(mapping): raise ValueError('Incomplete weight index')
    emb=headers.get('model.embed_tokens.weight'); head=headers.get('lm_head.weight')
    expected=[cfg['vocab_size'],cfg['hidden_size']]
    if not emb or emb['shape']!=expected: raise ValueError('Embedding shape does not match config')
    if head is None and not cfg.get('tie_word_embeddings',False): raise ValueError('Missing untied LM head')
    if head is not None and head['shape']!=expected: raise ValueError('LM head shape mismatch')
    layers=cfg['num_hidden_layers'];d=cfg['hidden_size'];heads=cfg['num_attention_heads']
    kv=cfg['num_key_value_heads'];hd=cfg.get('head_dim',d//heads);ff=cfg['intermediate_size']
    shapes={'model.norm.weight':[d]}
    for i in range(layers):
        base=f'model.layers.{i}.'
        shapes.update({base+'self_attn.q_proj.weight':[heads*hd,d],
            base+'self_attn.k_proj.weight':[kv*hd,d],base+'self_attn.v_proj.weight':[kv*hd,d],
            base+'self_attn.o_proj.weight':[d,heads*hd],
            base+'self_attn.q_norm.weight':[hd],base+'self_attn.k_norm.weight':[hd],
            base+'mlp.gate_proj.weight':[ff,d],base+'mlp.up_proj.weight':[ff,d],base+'mlp.down_proj.weight':[d,ff],
            base+'input_layernorm.weight':[d],base+'post_attention_layernorm.weight':[d]})
    for name,shape in shapes.items():
        if name not in headers or headers[name]['shape']!=shape:
            raise ValueError(f'Missing/wrong Qwen tensor: {name}; expected {shape}')
    return {'shards':names,'bytes':sum((root/n).stat().st_size for n in names),'tensor_count':len(headers),
            'header_integrity':'PASS','full_tensor_numerics':'NOT_CHECKED'}


def model_contract(root: str | Path, role: str, check_headers: bool=True) -> dict:
    root=Path(root).resolve();cfg=json.loads((root/'config.json').read_text())
    if role not in ('grid','program'): raise ValueError('Role must be grid or program')
    if cfg.get('model_type')!='qwen3' or cfg.get('hidden_size')!=2560 or cfg.get('num_hidden_layers')!=36:
        raise ValueError('Expected the Qwen3-4B architecture')
    vocab=cfg.get('vocab_size',0)
    if role=='grid' and vocab!=16: raise ValueError('G0 requires the exact 16-token ARC model')
    if role=='program' and vocab<100000: raise ValueError('P1 requires the full vocabulary, never the 16-token ARC model')
    tj=json.loads((root/'tokenizer.json').read_text());tv=tj.get('model',{}).get('vocab',{})
    if role=='grid':
        for tok,num in {**{str(i):i for i in range(10)},'Ċ':10,'user':11,'assistant':12}.items():
            if tv.get(tok)!=num: raise ValueError(f'ARC token mapping mismatch: {tok}')
        full=dict(tv)
        for t in tj.get('added_tokens',[]): full[t['content']]=t['id']
        for t,n in {'<|endoftext|>':13,'<|im_start|>':14,'<|im_end|>':15}.items():
            if full.get(t)!=n: raise ValueError(f'ARC control mismatch: {t}')
    elif len(tv)<100000: raise ValueError('P1 tokenizer vocabulary is incomplete')
    tc=root/'tokenizer_config.json'
    if not tc.exists(): raise ValueError('Missing tokenizer_config.json')
    info={'role':role,'path':str(root),'config_sha256':file_hash(root/'config.json'),
          'tokenizer_sha256':file_hash(root/'tokenizer.json'),'architecture':'qwen3-4b'}
    if check_headers: info.update(inspect_weights(root,cfg))
    return info


def resolve_model(input_root: str | Path, role: str, override: str='', prefer: str='') -> tuple[str,dict]:
    if override:
        info=model_contract(override,role);return info['path'],info
    possible=[]; rejected=[]
    for p in scan(input_root,{'config.json'}):
        try:
            cfg=json.loads(p.read_text())
            if cfg.get('model_type')!='qwen3' or cfg.get('hidden_size')!=2560 or cfg.get('num_hidden_layers')!=36: continue
            if (cfg.get('vocab_size')==16)!=(role=='grid'): continue
            info=model_contract(p.parent,role)
            possible.append(info)
        except (ValueError,OSError,KeyError) as e: rejected.append({'path':str(p.parent),'error':str(e)})
    preferred=[i for i in possible if prefer.lower() in i['path'].lower()] if prefer else []
    pool=preferred or possible
    if len(pool)!=1:
        raise ValueError(f'{role} model: expected one unambiguous mounted checkpoint, found {len(pool)}. '
                         f'Set an explicit override. Candidates={[p["path"] for p in pool]}; rejected={rejected[:5]}')
    result=dict(pool[0]);result['rejected_candidates']=rejected
    return result['path'],result


def resolve_data(input_root: str | Path, output_dir: str | Path, override: str='') -> tuple[str,dict]:
    """Accept an ordinary competition mount or an uploaded official ZIP; normalize JSON only."""
    if override: paths=[Path(override)]
    else:
        paths=list({p.parent for p in scan(input_root,{'arc-agi_training_challenges.json'})})
        if not paths:
            paths=[p for p in Path(input_root).rglob('*.zip') if 'arc' in p.name.lower()]
    good=[]
    for p in sorted(paths):
        try:
            if p.is_dir():
                if all((p/n).is_file() for n in DATA_FILES[:2]): good.append(p)
            else:
                with zipfile.ZipFile(p) as z:
                    names=[Path(n).name for n in z.namelist()]
                    if all(n in names for n in DATA_FILES[:2]): good.append(p)
        except (OSError,zipfile.BadZipFile): continue
    if len(good)!=1: raise ValueError(f'Need one unambiguous ARC data input. Set DATA_SOURCE. Candidates={list(map(str,good))}')
    source=good[0]
    if source.is_file():
        root=Path(output_dir);root.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(source) as z:
            for name in DATA_FILES:
                matches=[i for i in z.infolist() if Path(i.filename).name==name and not i.is_dir()]
                if len(matches)>1: raise ValueError('Duplicate canonical JSON filenames in archive')
                if not matches: continue
                info=matches[0]
                if info.file_size>64*1024*1024: raise ValueError('ARC JSON unexpectedly large')
                data=json.loads(z.read(info));atomic_json(root/name,data)
    else: root=source
    train=json.loads((root/DATA_FILES[0]).read_text());sol=json.loads((root/DATA_FILES[1]).read_text())
    if set(train)!=set(sol): raise ValueError('Training challenge/solution IDs differ')
    report={'source':str(source),'directory':str(root),'training_tasks':len(train),
            'files':{n:file_hash(root/n) for n in DATA_FILES if (root/n).exists()}}
    for prefix in ('evaluation','test'):
        q=root/f'arc-agi_{prefix}_challenges.json'
        if q.exists():
            tasks=json.loads(q.read_text());report[prefix+'_tasks']=len(tasks)
            report[prefix+'_outputs']=sum(len(t['test']) for t in tasks.values())
            report[prefix+'_exact_training_overlap']=sum(k in train and public_task(t)==public_task(train[k]) for k,t in tasks.items())
    report['test_is_public_training_placeholder']=bool(report.get('test_tasks',0) and report.get('test_tasks')==report.get('test_exact_training_overlap'))
    report['score_claim']=None
    return str(root),report


def install_offline(wheelhouse: str, packages: dict | None=None) -> None:
    if not wheelhouse: raise ValueError('Attach a wheel dataset and specify WHEELHOUSE; no online installation')
    p=Path(wheelhouse)
    if not p.is_dir() or not list(p.glob('*.whl')): raise ValueError('WHEELHOUSE must directly contain compatible wheels')
    wanted=packages or PACKAGES
    if any(x.lower().startswith(('torch','nvidia')) for x in wanted): raise ValueError('Do not replace the working CUDA/PyTorch stack')
    subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--find-links',str(p),
                   *[f'{k}=={v}' for k,v in wanted.items()]],check=True,timeout=240)


def dependency_report() -> dict:
    versions={}
    for n in ('torch','transformers','peft','accelerate','safetensors','tokenizers','huggingface-hub','numpy'):
        try: versions[n]=importlib.metadata.version(n)
        except importlib.metadata.PackageNotFoundError: versions[n]=None
    return {'versions':versions,'reference_versions':PACKAGES,
            'missing':[k for k in versions if versions[k] is None],
            'cuda_kernels':'NOT_TESTED_BY_VERSION_CHECK'}


In [ ]:
%%writefile arc_exec_rl/learning.py
"""Auditable group-relative policy loss; used by the real trainer and CPU tests.
One on-policy update per generated group; no critic and no reward std scaling.
Fixed maximum-completion normalization follows the Dr.GRPO-style length correction.
"""
from __future__ import annotations
import torch
import torch.nn.functional as F


def group_advantages(rewards: torch.Tensor) -> torch.Tensor:
    if rewards.ndim!=1 or rewards.numel()<2 or not torch.isfinite(rewards).all():
        raise ValueError("Need >=2 finite rewards for the same prompt")
    return rewards - rewards.mean()


def selected_logps(logits: torch.Tensor, targets: torch.Tensor, chunk_tokens: int = 32) -> torch.Tensor:
    if logits.ndim!=2 or targets.ndim!=1 or logits.shape[0]!=targets.shape[0]:
        raise ValueError("Token-logit alignment mismatch")
    if chunk_tokens<1: raise ValueError("chunk_tokens must be positive")
    from torch.utils.checkpoint import checkpoint
    pieces=[]
    def ce(x,y):
        # Keep doubles for numerical gradient tests; otherwise stable FP32 CE.
        return -F.cross_entropy(x if x.dtype==torch.float64 else x.float(), y, reduction="none")
    for start in range(0,len(targets),chunk_tokens):
        x=logits[start:start+chunk_tokens];y=targets[start:start+chunk_tokens]
        if torch.is_grad_enabled() and x.requires_grad and len(targets)>chunk_tokens:
            pieces.append(checkpoint(ce,x,y,use_reentrant=False,preserve_rng_state=False))
        else: pieces.append(ce(x,y))
    if not pieces: raise ValueError("Empty completion")
    return torch.cat(pieces)


def clipped_policy_loss(logp: torch.Tensor, old_logp: torch.Tensor, ref_logp: torch.Tensor,
                        advantage: torch.Tensor | float, group_size: int, max_completion: int,
                        beta: float=.02, clip_eps: float=.2) -> tuple[torch.Tensor, dict]:
    if not(logp.shape==old_logp.shape==ref_logp.shape) or logp.ndim!=1 or not logp.numel():
        raise ValueError("Mismatched or empty completion scores")
    if group_size<2 or max_completion<logp.numel(): raise ValueError("Invalid normalization")
    if not all(torch.isfinite(t).all() for t in (logp,old_logp,ref_logp)):
        raise FloatingPointError("Nonfinite policy scores")
    logratio=logp-old_logp.detach(); ref_delta=ref_logp.detach()-logp
    # Extreme ratios indicate numerical/model failure; do not silently clip the KL estimator.
    if logratio.detach().abs().max()>20 or ref_delta.detach().abs().max()>20:
        raise FloatingPointError("Policy drift exceeded safe log-ratio range")
    ratio=logratio.exp()
    a=torch.as_tensor(advantage,dtype=logp.dtype,device=logp.device).detach()
    objective=torch.minimum(ratio*a, ratio.clamp(1-clip_eps,1+clip_eps)*a)
    kl=ref_delta.exp()-ref_delta-1
    loss=(-objective+beta*kl).sum()/(group_size*max_completion)
    return loss,{"kl":float(kl.detach().mean()),"ratio_mean":float(ratio.detach().mean()),
                 "tokens":int(logp.numel()),"loss":float(loss.detach())}


def suffix_logps(model, prompt_ids: list[int], completion_ids: list[int], device) -> torch.Tensor:
    if not prompt_ids or not completion_ids: raise ValueError("Empty prompt/completion")
    ids=torch.tensor([prompt_ids+completion_ids],device=device,dtype=torch.long)
    c=len(completion_ids)
    # Qwen3 accepts logits_to_keep; only the answer suffix is projected to its large vocabulary.
    out=model(input_ids=ids,attention_mask=torch.ones_like(ids),use_cache=False,logits_to_keep=c+1)
    logits=out.logits[0,-(c+1):-1,:]
    if logits.shape[0]!=c: raise RuntimeError("Model does not support suffix-only logits correctly")
    return selected_logps(logits,ids[0,-c:])


def suffix_logps_batch(model, prompt_ids: list[int], completions: list[list[int]], device,
                       pad_id: int, chunk_tokens: int=32) -> list[torch.Tensor]:
    """Same-prompt completion microbatch, exact right-pad masking and suffix alignment."""
    if not prompt_ids or not completions or any(not x for x in completions):
        raise ValueError("Empty prompt/completion batch")
    m=max(map(len,completions));p=len(prompt_ids);b=len(completions)
    ids=torch.full((b,p+m),pad_id,dtype=torch.long,device=device)
    mask=torch.zeros_like(ids)
    for i,c in enumerate(completions):
        ids[i,:p+len(c)]=torch.tensor(prompt_ids+c,dtype=torch.long,device=device)
        mask[i,:p+len(c)]=1
    out=model(input_ids=ids,attention_mask=mask,use_cache=False,logits_to_keep=m+1)
    logits=out.logits[:,-(m+1):-1,:]
    if logits.shape[:2]!=(b,m): raise RuntimeError("Wrong batched suffix alignment")
    return [selected_logps(logits[i,:len(c)],ids[i,p:p+len(c)],chunk_tokens) for i,c in enumerate(completions)]


In [ ]:
%%writefile arc_exec_rl/modeling.py
from __future__ import annotations
import inspect
import json
import os
from pathlib import Path
from .common import atomic_json, file_hash, json_hash
from .dsl import DSL_VERSION

TARGET_MODULES=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
RECOMMENDED_VERSIONS={"transformers":"4.55.4","peft":"0.17.1","accelerate":"1.10.1"}


def checkpoint_files(path: str | Path) -> list[Path]:
    path=Path(path)
    files=sorted(path.glob("*.safetensors"))
    index=path/"model.safetensors.index.json"
    if index.exists():
        names=set(json.loads(index.read_text())["weight_map"].values())
        missing=[n for n in names if not (path/n).is_file()]
        if missing: raise ValueError(f"Missing weight shards: {missing}")
        files=[path/n for n in sorted(names)]
    if not files: raise ValueError(f"No safetensors weights: {path}")
    return files


def audit_program_model(path: str | Path, full_hash: bool = False) -> dict:
    path=Path(path).resolve()
    cfg=json.loads((path/"config.json").read_text())
    if cfg.get("model_type")!="qwen3" or cfg.get("num_hidden_layers")!=36 or cfg.get("vocab_size",0)<100000:
        raise ValueError("P1 requires full-vocabulary Qwen3-4B, NOT the 16-token grid model")
    if cfg.get("hidden_size")!=2560: raise ValueError("Expected Qwen3-4B hidden size")
    weights=checkpoint_files(path)
    from .kaggle_inputs import inspect_weights
    header_audit=inspect_weights(path,cfg)
    tok_files=[p for n in ("tokenizer.json","tokenizer_config.json","special_tokens_map.json","chat_template.jinja","chat_template.j2") if (p:=path/n).exists()]
    if not (path/"tokenizer.json").exists(): raise ValueError("Missing tokenizer.json")
    result={"path":str(path),"config_hash":file_hash(path/"config.json"),
            "tokenizer_files":{p.name:file_hash(p) for p in tok_files},
            "weights":{p.name:{"bytes":p.stat().st_size,"sha256":file_hash(p) if full_hash else None} for p in weights},
            "full_weight_hashes":full_hash,"dsl_version":DSL_VERSION,"header_audit":header_audit}
    return result


def load_program_model(path: str, device: str = "cuda:0", dtype: str = "bfloat16", training: bool = False):
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer
    audit_program_model(path)
    if device.startswith("cuda") and not torch.cuda.is_available(): raise RuntimeError("CUDA is unavailable; no Qwen4B run occurred")
    tokenizer=AutoTokenizer.from_pretrained(path,local_files_only=True,trust_remote_code=False)
    if len(tokenizer)<100000: raise ValueError("Wrong tokenizer family")
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    tokenizer.padding_side="left"
    model=AutoModelForCausalLM.from_pretrained(path,local_files_only=True,trust_remote_code=False,
                 torch_dtype=getattr(torch,dtype),attn_implementation="sdpa",low_cpu_mem_usage=True)
    if "logits_to_keep" not in inspect.signature(model.forward).parameters:
        raise RuntimeError("Qwen forward lacks logits_to_keep. Use the documented compatible Transformers environment")
    model.to(device); model.eval()
    model.config.use_cache=not training
    return model,tokenizer


def attach_lora(model, rank: int=64, alpha: int=128):
    from peft import LoraConfig,get_peft_model
    cfg=LoraConfig(r=rank,lora_alpha=alpha,lora_dropout=0.,bias="none",target_modules=TARGET_MODULES,task_type="CAUSAL_LM")
    model=get_peft_model(model,cfg)
    model.enable_input_require_grads()
    # Non-reentrant checkpointing works when inputs themselves need not require grad.
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
    # Training mode activates Transformers gradient checkpointing. Disable stochastic
    # dropout explicitly so on-policy rollout and recomputed scores still match.
    import torch
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout): module.p = 0.0
    if hasattr(model.config, "attention_dropout"): model.config.attention_dropout = 0.0
    model.train()
    for n,p in model.named_parameters():
        if p.requires_grad: p.data=p.data.float()
    return model


def tokenizer_fingerprint(tokenizer) -> str:
    return json_hash({"vocab":tokenizer.get_vocab(),"template":tokenizer.chat_template,
                      "eos":tokenizer.eos_token_id,"pad":tokenizer.pad_token_id})


def merged_export(model,tokenizer,output: str | Path,stage: str,metadata: dict,probe_ids: list[int],
                  merge_atol: float=.15) -> dict:
    """Export only at a stage boundary. Adapter checkpoints are saved before this call.
    Check merge logits in memory, write checksummed staging, mark COMPLETE last.
    A separate reload verification must run before promotion.
    """
    import shutil
    import torch
    out=Path(output)
    if out.exists(): raise FileExistsError(f"Refusing to overwrite export: {out}")
    stage_dir=out.with_name(out.name+".staging")
    if stage_dir.exists(): raise FileExistsError(f"Preserved incomplete export exists: {stage_dir}")
    stage_dir.mkdir(parents=True)
    model.eval(); model.gradient_checkpointing_disable()
    device=next(model.parameters()).device
    inp=torch.tensor([probe_ids[:64]],dtype=torch.long,device=device)
    with torch.no_grad(): before=model(input_ids=inp,logits_to_keep=1).logits[0,-1].float().cpu()
    if hasattr(model,"merge_and_unload"): model=model.merge_and_unload(safe_merge=True)
    with torch.no_grad(): after=model(input_ids=inp,logits_to_keep=1).logits[0,-1].float().cpu()
    error=float((before-after).abs().max())
    if not torch.isfinite(after).all() or error>merge_atol:
        atomic_json(stage_dir/"FAILED.json",{"reason":"merge_logit_mismatch","max_abs":error,"limit":merge_atol})
        raise RuntimeError(f"Merge parity failed: max_abs={error}; original adapter checkpoint remains")
    model.save_pretrained(stage_dir,safe_serialization=True,max_shard_size="2GB")
    tokenizer.save_pretrained(stage_dir)
    torch.save({"ids":probe_ids[:64],"logits":after},stage_dir/"reload_probe.pt")
    hashes={p.name:file_hash(p) for p in stage_dir.iterdir() if p.is_file()}
    manifest={"format":"arc-exec-rl-model-1","stage":stage,"architecture":"qwen3-program-dsl",
              "dsl_version":DSL_VERSION,"tokenizer_fingerprint":tokenizer_fingerprint(tokenizer),
              "weights_id":json_hash({n:h for n,h in hashes.items() if n.endswith('.safetensors')}),
              "files":hashes,"merge_max_abs_error":error,"reload_verified":False,
              "trained_weights_in_this_release":True,"metadata":metadata}
    atomic_json(stage_dir/"model_manifest.json",manifest)
    (stage_dir/"COMPLETE").write_text("exported; reload parity and independent evaluation are still required\n")
    os.replace(stage_dir,out)
    return manifest


def verify_export(path: str, device="cuda:0", atol=.15) -> dict:
    import torch
    root=Path(path); manifest=json.loads((root/"model_manifest.json").read_text())
    if not (root/"COMPLETE").exists(): raise ValueError("Export incomplete")
    if manifest["dsl_version"]!=DSL_VERSION: raise ValueError("Executor version mismatch")
    for n,h in manifest["files"].items():
        if file_hash(root/n)!=h: raise ValueError(f"Export checksum mismatch: {n}")
    model,tok=load_program_model(path,device)
    if tokenizer_fingerprint(tok)!=manifest["tokenizer_fingerprint"]: raise ValueError("Tokenizer changed")
    probe=torch.load(root/"reload_probe.pt",map_location="cpu",weights_only=True)
    with torch.no_grad():
        actual=model(input_ids=torch.tensor([probe["ids"]],device=device),logits_to_keep=1).logits[0,-1].float().cpu()
    error=float((actual-probe["logits"]).abs().max())
    report={"passed":bool(torch.isfinite(actual).all() and error<=atol),"max_abs_error":error,
            "weights_id":manifest["weights_id"],"tokenizer_exact":True}
    atomic_json(root/"reload_verification.json",report)
    if not report["passed"]: raise RuntimeError("Reload parity failed")
    manifest["reload_verified"]=True; atomic_json(root/"model_manifest.json",manifest)
    return report


In [ ]:
%%writefile arc_exec_rl/preflight.py
"""Cheap real-model GPU contract test before a long run. Not an ARC score."""
from __future__ import annotations
import argparse,json,time
from pathlib import Path
from .common import atomic_json,append_event


def run_probe(config: dict) -> dict:
    import torch
    from .train import configure_seed,make_optimizer,tokenize_record
    from .modeling import load_program_model,attach_lora,audit_program_model
    from .corpus import read_records,verify_corpus_manifest
    from .learning import suffix_logps,suffix_logps_batch
    root=Path(config['output_dir']);root.mkdir(parents=True,exist_ok=True)
    if not torch.cuda.is_available():
        result={'status':'NOT_RUN_NO_CUDA','qwen4b_score':None};atomic_json(root/'probe.json',result)
        raise RuntimeError('Real Qwen4B probe requires CUDA. No model execution occurred.')
    configure_seed(config['seed']);verify_corpus_manifest(config['corpus_dir'])
    audit=audit_program_model(config['base_model_path']);atomic_json(root/'probe_model_audit.json',audit)
    model,tok=load_program_model(config['base_model_path'],config['device'],training=True)
    model=attach_lora(model,config['lora_rank'],config['lora_alpha'])
    rows=read_records(Path(config['corpus_dir'])/'train.jsonl','train');r=rows[0]
    prompt,reply,_=tokenize_record(tok,r,config)
    # The full training prompt is used. Only the supervised program suffix is capped for this numerical probe.
    reply=reply[:min(len(reply),32)]
    torch.cuda.reset_peak_memory_stats();start=time.monotonic()
    with torch.no_grad():
        serial=suffix_logps(model,prompt,reply,config['device'])
        batch=suffix_logps_batch(model,prompt,[reply,reply],config['device'],tok.pad_token_id,config['loss_chunk_tokens'])
    delta=max(float((serial-x).abs().max()) for x in batch)
    if delta>.15: raise RuntimeError(f'Batch/serial numerical parity failed: max logp delta={delta}')
    opt=make_optimizer(model,config['sft_lr'],config)
    before={n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}
    loss=-suffix_logps(model,prompt,reply,config['device']).mean();loss.backward()
    params=[p for p in model.parameters() if p.requires_grad]
    norm=torch.nn.utils.clip_grad_norm_(params,config['max_grad_norm'],error_if_nonfinite=True)
    if not torch.isfinite(loss) or float(norm)<=0: raise RuntimeError('No finite nonzero learning signal')
    opt.step()
    changed=sum(not torch.equal(p.detach().cpu(),before[n]) for n,p in model.named_parameters() if p.requires_grad)
    if not changed: raise RuntimeError('Optimizer did not change any adapter tensor')
    result={'status':'REAL_QWEN4B_GPU_SMOKE_PASS','loss':float(loss.detach()),'grad_norm':float(norm),
            'adapter_tensors_changed':changed,'batch_serial_max_logp_delta':delta,
            'prompt_tokens':len(prompt),'supervised_probe_tokens':len(reply),
            'wall_seconds':time.monotonic()-start,'peak_allocated_gib':torch.cuda.max_memory_allocated()/1024**3,
            'qwen4b_arc_score':None,'scope':'API/memory/gradient smoke only; modified adapter is discarded, no trained export'}
    atomic_json(root/'probe.json',result);return result

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);a=p.parse_args()
    from .train import load_config
    print(json.dumps(run_probe(load_config(a.config)),indent=2))


In [ ]:
%%writefile arc_exec_rl/prompts.py
from __future__ import annotations
import json
import numpy as np
from .common import public_task
from .dsl import SCHEMA, components, bbox, background, object_holes

SYSTEM = '''Infer ONE transformation shared by every demonstration. Return ONLY a JSON program
{"steps":[{"op":"...", ...}]} with at most 8 steps. No prose, Python, file access, literal answer grids,
or task-ID lookups. The same program is run independently on each test input.
The executor supports only the following operations (defaults in parentheses):
identity; rotate(k=1, 0..3 counterclockwise); reflect(axis=h|v|d);
crop(bg=mode); crop_color(color); recolor(mapping={"old":new});
scale(row=2,col=2); tile(row=2,col=2); reduce(row=2,col=2,uniform blocks only);
translate(dr=0,dc=0,bg=mode,no foreground clipping);
fill_holes(color=1,bg=mode,4-connected enclosed background);
crop_object / keep_object(selector=largest|smallest|top_left|most_holes|least_holes,
 bg=mode,connectivity=4|8,monochrome=true);
move_to_edge(direction=left|right|up|down,bg=mode,whole foreground);
pack_objects(axis=h|v,order=size_asc|size_desc|position|holes,gap=1,bg=mode);
connect(axis=h|v|both,color=1,bg=mode,join extrema without changing obstacles);
mirror_concat(axis=h|v); draw_bbox(color=1,bg=mode,fill=false);
hole_recolor(mapping={"hole_count":color},bg=mode);
legend_holes(axis=h|v,side=before|after,target=5,bg=mode,one full separator).
Colors are integers 0..9. Grids must stay between 1x1 and 30x30 at every step.
An object summary is a fallible alternate parse; exact numeric grids are authoritative.'''


def grid_text(g: list[list[int]]) -> str:
    return f"{len(g)}x{len(g[0])}\n" + "\n".join("".join(map(str,row)) for row in g)


def object_digest(g: list[list[int]], max_objects: int = 12, max_relations: int = 24) -> dict:
    a=np.asarray(g,dtype=np.int8); bg=background(a); objects=[]
    for index,obj in enumerate(components(a,bg)[:max_objects]):
        r0,c0,r1,c1=bbox(obj)
        objects.append({"id":index,"color":int(a[obj[0]]),"area":len(obj),
                        "box":[r0,c0,r1,c1],"holes":object_holes(obj)})
    rel=[]
    for i,x in enumerate(objects):
        for y in objects[i+1:]:
            b=x["box"]; c=y["box"]; types=[]
            if b[0]+b[2]==c[0]+c[2]: types.append("row_aligned")
            if b[1]+b[3]==c[1]+c[3]: types.append("col_aligned")
            if b[0]<=c[0] and b[1]<=c[1] and b[2]>=c[2] and b[3]>=c[3]: types.append("box_contains")
            if x["area"]==y["area"]: types.append("same_area")
            if types and len(rel)<max_relations: rel.append([x["id"],y["id"],types])
    return {"background_hypothesis":bg,"objects":objects,"relations":rel,
            "object_list_may_be_truncated": len(components(a,bg))>max_objects}


def messages(task: dict, relational: bool = True) -> list[dict[str,str]]:
    t=public_task(task); blocks=[]
    for i,p in enumerate(t["train"]):
        blocks.append(f"DEMONSTRATION {i}\nINPUT\n{grid_text(p['input'])}\nOUTPUT\n{grid_text(p['output'])}")
        if relational:
            blocks.append("INPUT_OBJECTS "+json.dumps(object_digest(p["input"]),separators=(",",":")))
            blocks.append("OUTPUT_OBJECTS "+json.dumps(object_digest(p["output"]),separators=(",",":")))
    for i,p in enumerate(t["test"]):
        blocks.append(f"TEST {i} INPUT\n{grid_text(p['input'])}")
        if relational: blocks.append("INPUT_OBJECTS "+json.dumps(object_digest(p["input"]),separators=(",",":")))
    blocks.append("Return the shared JSON program. Do not return final grids.")
    return [{"role":"system","content":SYSTEM},{"role":"user","content":"\n\n".join(blocks)}]


def render_prompt(tokenizer, task: dict, relational: bool, max_prompt_tokens: int) -> tuple[str, int, bool]:
    """Drop optional object summaries, NEVER demonstrations or target-independent inputs."""
    for with_objects in ([True,False] if relational else [False]):
        text=tokenizer.apply_chat_template(messages(task,with_objects),tokenize=False,add_generation_prompt=True)
        n=len(tokenizer.encode(text,add_special_tokens=False))
        if n<=max_prompt_tokens: return text,n,with_objects
    raise ValueError(f"Prompt {n} tokens exceeds {max_prompt_tokens}; no silent truncation")


In [ ]:
%%writefile arc_exec_rl/rewards.py
from __future__ import annotations
import json
import os
from pathlib import Path
from .common import append_event
from .dsl import execute, parse_program, replay, program_id


def completion_text(completion) -> str:
    if isinstance(completion,str): return completion
    if isinstance(completion,list) and len(completion)==1 and completion[0].get("role")=="assistant":
        return completion[0]["content"]
    raise ValueError("Expected one assistant completion")


def score_program(text: str, task: dict, query_outputs: list) -> dict:
    """Exact reward on training-only query targets, not pixels or injected candidates.
    Demonstration-perfect execution is necessary. Test query labels never enter prompts.
    """
    if len(task["test"])!=len(query_outputs) or not query_outputs: raise ValueError("Missing reward targets")
    try:
        p=parse_program(text)
        demo=replay(p,task["train"])
        if not all(demo): return {"reward":0.,"reason":"demo_mismatch","demo_exact":sum(demo)}
        predictions=[execute(p,x["input"]) for x in task["test"]]
        wins=[x==y for x,y in zip(predictions,query_outputs)]
        return {"reward":sum(wins)/len(wins),"reason":"query_scored","demo_exact":sum(demo),
                "query_exact":sum(wins),"query_count":len(wins),"program_id":program_id(p)}
    except (ValueError,KeyError,IndexError,TypeError,RecursionError,OverflowError) as exc:
        return {"reward":0.,"reason":"invalid_or_failed_program","error_type":type(exc).__name__}


def execution_reward(completions, task_json, targets_json, record_id=None, **kwargs):
    """TRL 0.22.2 custom reward signature; side columns do not get serialized into prompt."""
    rewards=[]; reasons={}
    for i,completion in enumerate(completions):
        try:
            text=completion_text(completion)
            result=score_program(text,json.loads(task_json[i]),json.loads(targets_json[i]))
        except (ValueError,TypeError,KeyError): result={"reward":0.,"reason":"malformed_completion"}
        rewards.append(float(result["reward"]))
        reasons[result["reason"]]=reasons.get(result["reason"],0)+1
    event_path=os.getenv("ARC_EXEC_RL_REWARD_LOG")
    if event_path:
        append_event(event_path,"reward_batch",n=len(rewards),mean=sum(rewards)/max(1,len(rewards)),
                     nonzero=sum(r>0 for r in rewards), reasons=reasons)
    return rewards


In [ ]:
%%writefile arc_exec_rl/train.py
"""Single-device Qwen4B SFT -> merged reference -> execution-reward LoRA GRPO.
Offline local weights only. Run outside the final competition notebook.
No value network, no duplicate 4B reference, no public-evaluation reward labels.
"""
from __future__ import annotations
import argparse
import gc
import json
import math
import os
import random
import shutil
import time
from pathlib import Path
from .common import atomic_json,append_event,file_hash,json_hash,environment_report,stable_seed
from .corpus import read_records,validate_record,scene,reference,verify_corpus_manifest
from .efficient import RecordTokenCache,generate_programs
from .curriculum import FamilySampler
from .dsl import parse_program
from .prompts import render_prompt
from .rewards import score_program

DEFAULTS={
 "base_model_path":"", "corpus_dir":"", "output_dir":"", "device":"cuda:0",
 "seed":20260907, "lora_rank":64,"lora_alpha":128,
 "sft_lr":2e-5,"rl_lr":2e-6,"weight_decay":.01,"max_grad_norm":1.,
 "sft_steps":400,"sft_accumulation":8,"sft_max_hours":4.,
 "rl_steps":128,"rl_max_hours":2.,"group_size":4,"kl_beta":.02,"clip_eps":.2,
 "max_prompt_tokens":6144,"max_completion_tokens":512,"max_sequence_tokens":8192,
 "gradient_checkpointing":True,"relational":True,
 "replay_every":4,"replay_weight":.1,"max_no_signal_groups":48,
 "save_every":25,"keep_checkpoints":2,"resume_from":"",
 "min_sft_demo_fit_for_rl":.02,"rl_probe_tasks":16,
 "rollout_microbatch":2,"rl_score_microbatch":1,"loss_chunk_tokens":32,
 "stop_on_complete_json":True,"json_check_interval":8,
 "family_sampling":True,"family_exploration":.25,"token_cache_entries":8192,
 "optimizer_kind":"adamw","lora_plus_ratio":16.,"max_query_zero_probe":True,
}


def load_config(path: str) -> dict:
    given=json.loads(Path(path).read_text())
    unknown=set(given)-set(DEFAULTS)
    if unknown: raise ValueError(f"Unused/unknown settings rejected: {sorted(unknown)}")
    c={**DEFAULTS,**given}
    if c["group_size"]<2: raise ValueError("RL groups need at least two samples")
    if c["max_prompt_tokens"]+c["max_completion_tokens"]>c["max_sequence_tokens"]: raise ValueError("Context budget exceeds limit")
    if not c["gradient_checkpointing"]: raise ValueError("This single-device recipe requires checkpointing; ablate separately")
    if c['optimizer_kind'] not in ('adamw','loraplus'): raise ValueError('Unknown optimizer')
    if not 0<c['family_exploration']<=1: raise ValueError('family_exploration must be in (0,1]')
    if any(c[k]<1 for k in ('rollout_microbatch','rl_score_microbatch','loss_chunk_tokens','json_check_interval','token_cache_entries','save_every','keep_checkpoints')):
        raise ValueError('Positive budgets required')
    return c


def configure_seed(seed: int):
    import torch,numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def make_optimizer(model,lr: float,c: dict):
    import torch
    params=[p for p in model.parameters() if p.requires_grad]
    if c.get('optimizer_kind')=='loraplus':
        # Optional official PEFT implementation; never enabled silently.
        from peft.optimizers import create_loraplus_optimizer
        return create_loraplus_optimizer(model=model,optimizer_cls=torch.optim.AdamW,
            lr=lr,loraplus_lr_ratio=c['lora_plus_ratio'],weight_decay=c['weight_decay'])
    if not params: raise RuntimeError("No trainable model parameters; RL would be a no-op")
    kwargs={"lr":lr,"betas":(.9,.95),"eps":1e-8,"weight_decay":c["weight_decay"]}
    # Fused AdamW only when supported on actual CUDA tensors; no bitsandbytes requirement.
    if all(p.is_cuda for p in params): kwargs["fused"]=True
    try: return torch.optim.AdamW(params,**kwargs)
    except (TypeError,RuntimeError):
        kwargs.pop("fused",None); return torch.optim.AdamW(params,**kwargs)


def save_checkpoint(model,tokenizer,optimizer,root: Path,stage: str,step: int,cursor: int,c: dict,corpus_hash: str,aux: dict | None=None):
    import torch,numpy as np
    root.mkdir(parents=True,exist_ok=True); dst=root/f"{stage}-{step:06d}-c{cursor:08d}"; tmp=root/(dst.name+".staging")
    if dst.exists(): return dst
    if tmp.exists(): shutil.rmtree(tmp)
    model.save_pretrained(tmp,safe_serialization=True); tokenizer.save_pretrained(tmp)
    torch.save({"optimizer":optimizer.state_dict(),"torch_rng":torch.get_rng_state(),
                "cuda_rng":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "python_rng":random.getstate(),"numpy_rng":np.random.get_state()},tmp/"training_state.pt")
    atomic_json(tmp/"state.json",{"stage":stage,"step":step,"cursor":cursor,"config":c,"corpus_hash":corpus_hash,"base_fingerprint":json_hash(json.loads((root.parent/"base_model_audit.json").read_text())),"aux":aux or {}})
    (tmp/"COMPLETE").write_text("trusted local training checkpoint\n"); os.replace(tmp,dst)
    atomic_json(root/"latest.json",{"path":str(dst.resolve())})
    complete=sorted(p for p in root.glob(stage+"-*") if (p/"COMPLETE").exists())
    for p in complete[:-c["keep_checkpoints"]]: shutil.rmtree(p)
    return dst


def restore_checkpoint(model,optimizer,path: str,stage: str,corpus_hash: str,c: dict) -> tuple[int,int]:
    import torch,numpy as np
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    p=Path(path); s=json.loads((p/"state.json").read_text())
    if not (p/"COMPLETE").exists() or s["stage"]!=stage or s["corpus_hash"]!=corpus_hash: raise ValueError("Resume stage/data mismatch")
    # Budget/path may change, but optimization/data/model settings must not.
    mutable={"resume_from","sft_max_hours","rl_max_hours","output_dir"}
    if s["base_fingerprint"]!=json_hash(json.loads((Path(c["output_dir"])/"base_model_audit.json").read_text())):
        raise ValueError("Resume base weights/tokenizer fingerprint changed")
    for k,v in s["config"].items():
        if k not in mutable and c.get(k)!=v: raise ValueError(f"Resume config changed: {k}")
    set_peft_model_state_dict(model,load_file(str(p/"adapter_model.safetensors")))
    # Only load state that THIS tool created in a trusted local training directory.
    state=torch.load(p/"training_state.pt",map_location="cpu",weights_only=False)
    optimizer.load_state_dict(state["optimizer"])
    torch.set_rng_state(state["torch_rng"]); random.setstate(state["python_rng"]); np.random.set_state(state["numpy_rng"])
    if torch.cuda.is_available() and state["cuda_rng"]: torch.cuda.set_rng_state_all(state["cuda_rng"])
    return int(s["step"]),int(s["cursor"])


def tokenize_record(tokenizer,r: dict,c: dict) -> tuple[list[int],list[int],bool]:
    text,n,rel=render_prompt(tokenizer,r["task"],c["relational"],c["max_prompt_tokens"])
    prompt=tokenizer.encode(text,add_special_tokens=False)
    reply=tokenizer.encode(json.dumps(r["program"],separators=(",",":")),add_special_tokens=False)+[tokenizer.eos_token_id]
    if len(reply)>c["max_completion_tokens"] or n+len(reply)>c["max_sequence_tokens"]:
        raise ValueError("Supervised program too long; record is not silently truncated")
    return prompt,reply,rel


def prepared_training(c: dict):
    from .modeling import load_program_model,attach_lora,audit_program_model
    import torch
    out=Path(c["output_dir"]); out.mkdir(parents=True,exist_ok=True)
    atomic_json(out/"environment.json",environment_report())
    if not torch.cuda.is_available(): raise RuntimeError("Qwen4B GPU training NOT_RUN: CUDA is unavailable")
    manifest=verify_corpus_manifest(c["corpus_dir"])
    rows=read_records(Path(c["corpus_dir"])/"train.jsonl","train")
    if len(rows)<16: raise ValueError("Training corpus too small")
    rows=[validate_record(r) for r in rows]
    random.Random(c["seed"]).shuffle(rows)
    audit=audit_program_model(c["base_model_path"],full_hash=True); atomic_json(out/"base_model_audit.json",audit)
    model,tok=load_program_model(c["base_model_path"],c["device"],training=True)
    model=attach_lora(model,c["lora_rank"],c["lora_alpha"])
    return out,manifest,rows,model,tok


def sft(c: dict) -> dict:
    import torch
    from .learning import suffix_logps
    from .modeling import merged_export
    configure_seed(c["seed"])
    out,manifest,rows,model,tok=prepared_training(c)
    opt=make_optimizer(model,c["sft_lr"],c); start_step=cursor=0
    if c["resume_from"]: start_step,cursor=restore_checkpoint(model,opt,c["resume_from"],"sft",manifest["dataset_hash"],c)
    deadline=time.monotonic()+c["sft_max_hours"]*3600; useful=0; completions=0; invalid=0; last_step=start_step; probe=None
    cache=RecordTokenCache(tok,c["token_cache_entries"])
    log=out/"sft_events.jsonl"
    begin=time.monotonic()
    for step in range(start_step,c["sft_steps"]):
        if time.monotonic()>=deadline: break
        opt.zero_grad(set_to_none=True); losses=[]; attempts=0
        while len(losses)<c["sft_accumulation"] and attempts<4*c["sft_accumulation"]:
            r=rows[cursor%len(rows)]; cursor+=1; attempts+=1
            try: prompt,reply,rel=cache.get(r,c)
            except ValueError: invalid+=1; continue
            if probe is None: probe=prompt
            lp=suffix_logps(model,prompt,reply,c["device"])
            loss=-lp.mean()/c["sft_accumulation"]
            if not torch.isfinite(loss): raise FloatingPointError("Nonfinite SFT loss")
            loss.backward(); losses.append(float(loss.detach())*c["sft_accumulation"])
            useful+=len(prompt)+len(reply); completions+=len(reply)
            del loss,lp
            if time.monotonic()>=deadline: break
        if not losses: raise RuntimeError("No usable supervised examples")
        # Scale a partial accumulated step to preserve the intended mean gradient.
        if len(losses)!=c["sft_accumulation"]:
            for p in model.parameters():
                if p.grad is not None: p.grad.mul_(c["sft_accumulation"]/len(losses))
        norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],c["max_grad_norm"],error_if_nonfinite=True)
        warm=max(1,int(.03*c["sft_steps"])); progress=step/max(c["sft_steps"],1)
        lr=c["sft_lr"]*min(1,(step+1)/warm)*(.1+.9*.5*(1+math.cos(math.pi*progress)))
        for g in opt.param_groups:
            g.setdefault('arc_lr_multiplier',g['lr']/c['sft_lr'])
            g["lr"]=lr*g['arc_lr_multiplier']
        opt.step(); last_step=step+1
        append_event(log,"sft_step",step=last_step,loss=sum(losses)/len(losses),grad_norm=float(norm),
                     useful_tokens=useful,answer_tokens=completions,skipped_length=invalid,lr=lr,
                     useful_tokens_per_second=useful/max(time.monotonic()-begin,1e-6),
                     token_cache_hits=cache.hits,token_cache_misses=cache.misses,
                     peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else None)
        print(f"SFT step={last_step} loss={sum(losses)/len(losses):.5f} tokens={useful} grad={float(norm):.5f}",flush=True)
        if last_step%c["save_every"]==0: save_checkpoint(model,tok,opt,out/"checkpoints","sft",last_step,cursor,c,manifest["dataset_hash"])
    if last_step==start_step: raise RuntimeError("No SFT optimizer steps completed")
    ckpt=save_checkpoint(model,tok,opt,out/"checkpoints","sft",last_step,cursor,c,manifest["dataset_hash"])
    report={"status":"TRAINED_CANDIDATE_NOT_SCORED","optimizer_steps":last_step,"useful_tokens_this_run":useful,
            "answer_tokens_this_run":completions,"skipped_length":invalid,"corpus_hash":manifest["dataset_hash"],
            "bootstrap_only":manifest["bootstrap_only"],"checkpoint":str(ckpt)}
    atomic_json(out/"sft_report.json",report)
    del opt; gc.collect(); torch.cuda.empty_cache()
    merged_export(model,tok,out/"sft_merged","sft",report,probe)
    return report


def generated_group(model,tokenizer,prompt: list[int],c: dict,deadline: float) -> list[dict]:
    result,stats=generate_programs(model,tokenizer,prompt,c['group_size'],deadline,c['device'],
        max_new_tokens=c['max_completion_tokens'],microbatch=c['rollout_microbatch'],
        sample=True,temperature=1.,top_p=1.,stop_on_json=c['stop_on_complete_json'],
        json_check_interval=c['json_check_interval'])
    for x in result: x['generation_stats']=stats
    return result


def refresh_training_queries(r: dict,cursor: int) -> dict:
    if r["source"]!="independent-bootstrap-v1": return r
    r=json.loads(json.dumps(r)); rng=random.Random(stable_seed("fresh-queries",r["record_id"],cursor)); queries=[]
    for _ in range(80):
        g=scene(rng)
        palette=r.get("generator_metadata",{}).get("palette",list(range(10)))
        g=[[palette[v] for v in row] for row in g]
        try: y=reference(r["program"],g)
        except ValueError: continue
        queries.append((g,y))
        if len(queries)==3: break
    if len(queries)==3:
        r["task"]["test"]=[{"input":g} for g,y in queries]; r["query_outputs"]=[y for g,y in queries]
    return r


def rl(c: dict) -> dict:
    import torch
    from .learning import suffix_logps,suffix_logps_batch,group_advantages,clipped_policy_loss
    from .modeling import merged_export
    configure_seed(c["seed"])
    base_manifest=Path(c["base_model_path"])/"model_manifest.json"
    if not base_manifest.exists() or json.loads(base_manifest.read_text()).get("stage")!="sft":
        raise ValueError("RL must start from a MERGED SFT export, not a bare generic Qwen or unmerged adapter")
    bm=json.loads(base_manifest.read_text())
    if not bm.get("reload_verified"):
        raise ValueError("Verify SFT export reload before RL")
    for name,digest in bm["files"].items():
        if file_hash(Path(c["base_model_path"])/name)!=digest:
            raise ValueError(f"SFT reference export changed: {name}")
    out,manifest,rows,model,tok=prepared_training(c)
    opt=make_optimizer(model,c["rl_lr"],c); step=cursor=0
    if c["resume_from"]: step,cursor=restore_checkpoint(model,opt,c["resume_from"],"rl",manifest["dataset_hash"],c)
    deadline=time.monotonic()+c["rl_max_hours"]*3600; start_step=step; nosignal=0; groups=0; probe=None; prompt_skips=0
    log=out/"rl_events.jsonl"; all_rewards=[]; probe_groups=0; probe_demo_valid=0; probe_total=0
    sampler=FamilySampler(rows,c['seed']+1,c['family_exploration']) if c['family_sampling'] else None
    cache=RecordTokenCache(tok,c['token_cache_entries'])
    if c['resume_from']:
        aux=json.loads((Path(c['resume_from'])/'state.json').read_text()).get('aux',{})
        if sampler and aux.get('sampler'): sampler.restore(aux['sampler'])
        nosignal=aux.get('no_signal',0)
    def auxiliary(): return {'sampler':sampler.state() if sampler else None,'no_signal':nosignal}

    while step<c["rl_steps"] and time.monotonic()<deadline:
        selected=sampler.choose() if sampler else rows[cursor%len(rows)]
        r=refresh_training_queries(selected,cursor); cursor+=1
        rollout_start=time.monotonic()
        try:
            prompt_text,_,rel=render_prompt(tok,r["task"],c["relational"],c["max_prompt_tokens"])
            prompt=tok.encode(prompt_text,add_special_tokens=False)
        except ValueError:
            append_event(log,"prompt_too_long",record_id=r["record_id"])
            prompt_skips+=1
            if prompt_skips>=len(rows): raise RuntimeError("All training prompts exceeded the context limit")
            continue
        if probe is None: probe=prompt
        group=generated_group(model,tok,prompt,c,deadline)
        if len(group)!=c["group_size"] or time.monotonic()>=deadline: break  # Do not update on a partial deadline-interrupted group.
        groups+=1
        details=[score_program(x["text"],r["task"],r["query_outputs"]) if x["terminated"] else {"reward":0.,"reason":"truncated"} for x in group]
        rewards=torch.tensor([d["reward"] for d in details],dtype=torch.float32)
        all_rewards.extend(rewards.tolist()); probe_groups+=1
        if sampler: sampler.update(r['family_id'],rewards.tolist(),time.monotonic()-rollout_start)
        probe_total+=len(group); probe_demo_valid+=sum(d["reason"]=="query_scored" for d in details)
        append_event(log,"rollout_group",group=groups,step=step,record_id=r["record_id"],family=r["family_id"],
                     rewards=rewards.tolist(),reasons=[d["reason"] for d in details],
                     completion_lengths=[len(x["ids"]) for x in group],
                     generation=group[0].get('generation_stats'),
                     stop_reasons=[x.get('stop_reason') for x in group],
                     rollout_seconds=time.monotonic()-rollout_start)
        if probe_groups==c["rl_probe_tasks"] and probe_demo_valid/max(probe_total,1)<c["min_sft_demo_fit_for_rl"]:
            append_event(log,"rl_stopped_insufficient_warmstart",demo_fit=probe_demo_valid/max(probe_total,1)); break
        if probe_groups==c['rl_probe_tasks'] and c['max_query_zero_probe'] and not any(x>0 for x in all_rewards):
            append_event(log,'rl_stopped_no_query_success');break
        print(f"RL group={groups} updates={step} rewards={rewards.tolist()}",flush=True)
        adv=group_advantages(rewards)
        if float(adv.abs().max())==0:
            nosignal+=1
            if nosignal>=c["max_no_signal_groups"]:
                append_event(log,"rl_stopped_no_reward_variance",consecutive=nosignal); break
            continue
        nosignal=0; opt.zero_grad(set_to_none=True); statistics=[]
        valid=[i for i,x in enumerate(group) if x['terminated'] and x['ids']]
        # Right-padded completion batches share a prompt; labels/pads are masked exactly.
        for start in range(0,len(valid),c['rl_score_microbatch']):
            indexes=valid[start:start+c['rl_score_microbatch']]
            completions_batch=[group[i]['ids'] for i in indexes]
            with torch.no_grad():
                with model.disable_adapter():
                    refs=suffix_logps_batch(model,prompt,completions_batch,c['device'],tok.pad_token_id,c['loss_chunk_tokens'])
            currents=suffix_logps_batch(model,prompt,completions_batch,c['device'],tok.pad_token_id,c['loss_chunk_tokens'])
            batch_loss=None
            for j,i in enumerate(indexes):
                current=currents[j];ref=refs[j].detach();old=current.detach()
                loss,stats=clipped_policy_loss(current,old,ref,adv[i],c['group_size'],c['max_completion_tokens'],c['kl_beta'],c['clip_eps'])
                batch_loss=loss if batch_loss is None else batch_loss+loss
                statistics.append(stats)
            batch_loss.backward()
            del batch_loss,currents,refs,current,ref,old,loss
        if not statistics: continue
        replay_loss=None
        if c["replay_every"] and (step+1)%c["replay_every"]==0:
            rr=rows[(cursor+17)%len(rows)]
            try:
                rp,ry,_=cache.get(rr,c)
                lp=suffix_logps(model,rp,ry,c["device"])
                loss=-c["replay_weight"]*lp.mean(); loss.backward(); replay_loss=float(loss.detach()); del lp,loss
            except ValueError: pass
        norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],c["max_grad_norm"],error_if_nonfinite=True)
        opt.step(); step+=1
        append_event(log,"rl_optimizer_step",step=step,reward=float(rewards.mean()),
                     grad_norm=float(norm),mean_kl=sum(s["kl"] for s in statistics)/len(statistics),replay_loss=replay_loss,
                     gradient_batches=len(statistics),peak_allocated_gib=torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else None)
        if step%c["save_every"]==0: save_checkpoint(model,tok,opt,out/"checkpoints","rl",step,cursor,c,manifest["dataset_hash"],auxiliary())
    report={"status":"TRAINED_CANDIDATE_NOT_SCORED" if step>start_step else "NO_RL_UPDATE_COMPLETED",
            "optimizer_steps_total":step,"optimizer_steps_this_run":step-start_step,"rollout_groups_this_run":groups,
            "mean_training_reward":sum(all_rewards)/max(len(all_rewards),1),
            "corpus_hash":manifest["dataset_hash"],"bootstrap_only":manifest["bootstrap_only"],
            "loss":"group-relative, no std scaling, clipped token surrogate, fixed-length denominator, reference KL",
            "sampling":"temperature=1, top_p=1, top_k=0, no policy-distorting sampling warpers",
            "score_claim":None}
    atomic_json(out/"rl_report.json",report)
    if step==start_step: return report
    save_checkpoint(model,tok,opt,out/"checkpoints","rl",step,cursor,c,manifest["dataset_hash"],auxiliary())
    del opt; gc.collect(); torch.cuda.empty_cache()
    merged_export(model,tok,out/"rl_merged","rl",report,probe)
    return report

if __name__=="__main__":
    p=argparse.ArgumentParser(); p.add_argument("stage",choices=["sft","rl","verify-export"]); p.add_argument("--config"); p.add_argument("--model")
    a=p.parse_args()
    if a.stage=="verify-export":
        from .modeling import verify_export
        print(json.dumps(verify_export(a.model),indent=2))
    else:
        c=load_config(a.config); print(json.dumps(sft(c) if a.stage=="sft" else rl(c),indent=2))


In [ ]:
%%writefile arc_dataset_guard.py
from __future__ import annotations

import hashlib
import json
import math
import os
import time
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

from arc_symbolic import SymbolicSolver, as_grid, infer_output_shapes, valid_grid


def env_truthy(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "1" if default else "0").strip().lower()
    return raw not in ("", "0", "false", "no", "none")


def atomic_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    json.loads(tmp.read_text(encoding="utf-8"))
    os.replace(tmp, path)


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def find_competition_dir() -> Path:
    explicit = os.getenv("ARC_COMPETITION_DIR", "").strip()
    if explicit and Path(explicit).exists():
        return Path(explicit)
    direct = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    if direct.exists():
        return direct
    root = Path("/kaggle/input")
    for name in ("arc-agi_evaluation_challenges.json", "arc-agi_test_challenges.json"):
        hit = next(root.rglob(name), None) if root.exists() else None
        if hit is not None:
            return hit.parent
    raise FileNotFoundError("ARC-AGI-2 competition directory was not found")


def _load(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def _count_outputs(challenges: dict[str, Any]) -> int:
    return sum(len(task.get("test", [])) for task in challenges.values())


def dataset_audit(output_dir: str | Path, run_cpu_public_audit: bool = True) -> dict[str, Any]:
    start = time.time()
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    comp = find_competition_dir()
    files = {
        name: comp / name
        for name in (
            "arc-agi_training_challenges.json",
            "arc-agi_training_solutions.json",
            "arc-agi_evaluation_challenges.json",
            "arc-agi_evaluation_solutions.json",
            "arc-agi_test_challenges.json",
            "sample_submission.json",
        )
        if (comp / name).exists()
    }
    hashes = {name: sha256_file(path) for name, path in files.items()}
    training = _load(files["arc-agi_training_challenges.json"]) if "arc-agi_training_challenges.json" in files else {}
    evaluation = _load(files["arc-agi_evaluation_challenges.json"]) if "arc-agi_evaluation_challenges.json" in files else {}
    test = _load(files["arc-agi_test_challenges.json"]) if "arc-agi_test_challenges.json" in files else {}

    shared_test_train_ids = sorted(set(test) & set(training))
    exact_test_train = sum(test[k] == training[k] for k in shared_test_train_ids)
    placeholder = bool(test and len(shared_test_train_ids) == len(test) and exact_test_train == len(test))
    report: dict[str, Any] = {
        "competition_dir": str(comp),
        "file_sha256": hashes,
        "training_tasks": len(training),
        "training_outputs": _count_outputs(training),
        "evaluation_tasks": len(evaluation),
        "evaluation_outputs": _count_outputs(evaluation),
        "test_tasks": len(test),
        "test_outputs": _count_outputs(test),
        "test_training_shared_ids": len(shared_test_train_ids),
        "test_training_exact_content": exact_test_train,
        "test_is_training_placeholder": placeholder,
        "warning": (
            "The local test file is a 240-task public-training placeholder. It is not hidden-score evidence; "
            "Kaggle replaces it during the competition rerun."
            if placeholder else "The test file is not the known public-training placeholder."
        ),
    }

    # Public diagnostic only. It never writes predictions or routes from solutions.
    if run_cpu_public_audit and evaluation and "arc-agi_evaluation_solutions.json" in files and not env_truthy("KAGGLE_IS_COMPETITION_RERUN"):
        solutions = _load(files["arc-agi_evaluation_solutions.json"])
        shape_hits = Counter()
        symbolic_top1 = 0
        symbolic_oracle = 0
        total = 0
        solver = SymbolicSolver(max_programs=int(os.getenv("ARC_PUBLIC_AUDIT_SYMBOLIC_PROGRAMS", "500")), max_outputs=12)
        for task_id, task in evaluation.items():
            for test_idx, test_ex in enumerate(task.get("test", [])):
                total += 1
                target = as_grid(solutions[task_id][test_idx])
                hypotheses = infer_output_shapes(task, as_grid(test_ex["input"]), max_shapes=12)
                shapes = [(h, w) for h, w, _, _ in hypotheses]
                for k in (1, 2, 3, 5, 8, 12):
                    shape_hits[k] += int(tuple(target.shape) in shapes[:k])
                candidates = solver.synthesize(task, test_idx)
                matches = [i for i, cand in enumerate(candidates) if np.array_equal(cand.grid, target)]
                symbolic_top1 += int(bool(matches) and matches[0] == 0)
                symbolic_oracle += int(bool(matches))
        report["public_cpu_audit"] = {
            "outputs": total,
            "shape_recall": {f"top{k}": int(shape_hits[k]) for k in (1, 2, 3, 5, 8, 12)},
            "shape_recall_percent": {f"top{k}": 100.0 * shape_hits[k] / max(1, total) for k in (1, 2, 3, 5, 8, 12)},
            "symbolic_top1_exact": symbolic_top1,
            "symbolic_union_exact": symbolic_oracle,
            "symbolic_promotion_allowed": False,
            "interpretation": (
                "Shape inference is useful as a constrained-decoding prior. The exact symbolic lane produced no "
                "public-evaluation saves in this audit, so it is not allowed to displace a neural attempt by default."
            ),
        }
    report["runtime_seconds"] = time.time() - start
    atomic_json(out / "dataset_audit.json", report)
    return report


def _model_candidates(explicit: str = "") -> list[Path]:
    values: list[Path] = []
    if explicit:
        values.append(Path(explicit))
    values.extend([
        Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        Path("/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
    ])
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            text = str(cfg).lower()
            if "qwen3_4b_grids15_sft139" in text:
                values.append(cfg.parent)
    unique: list[Path] = []
    seen: set[str] = set()
    for value in values:
        key = str(value.resolve()) if value.exists() else str(value)
        if key not in seen:
            seen.add(key)
            unique.append(value)
    return unique


def audit_grid16_model(path: str | Path, require_exact_ids: bool = True) -> dict[str, Any]:
    path = Path(path)
    report: dict[str, Any] = {"path": str(path), "passed": False, "errors": []}
    if not path.is_dir():
        report["errors"].append("directory_missing")
        return report
    config_path = path / "config.json"
    if not config_path.exists():
        report["errors"].append("config_missing")
        return report
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as exc:
        report["errors"].append(f"config_parse:{type(exc).__name__}")
        return report
    report["vocab_size"] = config.get("vocab_size")
    report["model_type"] = config.get("model_type")
    report["hidden_size"] = config.get("hidden_size")
    report["layers"] = config.get("num_hidden_layers")
    if int(config.get("vocab_size", -1)) != 16:
        report["errors"].append("vocab_size_not_16")
    weights = sorted(path.glob("*.safetensors")) + sorted(path.glob("pytorch_model*.bin"))
    report["weight_files"] = [p.name for p in weights]
    if not weights:
        report["errors"].append("weights_missing")
    try:
        from transformers import AutoTokenizer
        from arc_loader import ArcTokenSpec
        tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
        spec = ArcTokenSpec.from_tokenizer(tokenizer)
        report["tokenizer_length"] = len(tokenizer)
        report["token_spec"] = {
            "digits": list(spec.digit_ids),
            "newline": spec.newline_id,
            "eos": spec.eos_id,
            "pad": spec.pad_id,
            "user_marker": list(spec.user_marker),
            "assistant_marker": list(spec.assistant_marker),
        }
        if len(tokenizer) != 16:
            report["errors"].append("tokenizer_length_not_16")
        if require_exact_ids:
            expected = {
                "digits": list(range(10)), "newline": 10, "eos": 15,
                "user_marker": [14, 11, 10], "assistant_marker": [14, 12, 10],
            }
            actual = report["token_spec"]
            for name, value in expected.items():
                if actual.get(name) != value:
                    report["errors"].append(f"token_semantics:{name}:{actual.get(name)}!={value}")
    except Exception as exc:
        report["errors"].append(f"tokenizer_contract:{type(exc).__name__}:{exc}")
    report["passed"] = not report["errors"]
    return report


def resolve_primary_model(output_dir: str | Path, override: str = "", allow_custom: bool = False, require_exact_ids: bool = True) -> tuple[str, dict[str, Any]]:
    if override and not allow_custom:
        raise RuntimeError("PRIMARY_MODEL_OVERRIDE was set while ALLOW_CUSTOM_PRIMARY=False")
    audits = []
    for candidate in _model_candidates(override if allow_custom else ""):
        audit = audit_grid16_model(candidate, require_exact_ids=require_exact_ids)
        audits.append(audit)
        if audit["passed"]:
            payload = {"selected": str(candidate), "audits": audits}
            atomic_json(Path(output_dir) / "model_audit.json", payload)
            return str(candidate), payload
    payload = {"selected": None, "audits": audits}
    atomic_json(Path(output_dir) / "model_audit.json", payload)
    raise RuntimeError("No exact 16-token Qwen3-4B sft139 checkpoint passed the model contract")


def _task_complexity(task: dict[str, Any]) -> dict[str, float]:
    train = task.get("train", [])
    tests = task.get("test", [])
    input_cells = [np.asarray(ex["input"]).size for ex in train + tests]
    output_cells = [np.asarray(ex["output"]).size for ex in train]
    shape_change = np.mean([
        tuple(np.asarray(ex["input"]).shape) != tuple(np.asarray(ex["output"]).shape)
        for ex in train
    ]) if train else 1.0
    colors = [len(np.unique(np.asarray(ex["input"]))) for ex in train + tests]
    return {
        "demos": float(len(train)),
        "tests": float(len(tests)),
        "mean_input_cells": float(np.mean(input_cells or [1])),
        "max_input_cells": float(max(input_cells or [1])),
        "mean_output_cells": float(np.mean(output_cells or [1])),
        "shape_change": float(shape_change),
        "mean_colors": float(np.mean(colors or [1])),
    }


def build_safe_routes(output_dir: str | Path, profile: str = "DATASET_SAFE_MAX") -> dict[str, Any]:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    comp = find_competition_dir()
    rerun = env_truthy("KAGGLE_IS_COMPETITION_RERUN")
    challenge_name = "arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json"
    challenges = _load(comp / challenge_name)
    train_q = comp / "arc-agi_training_challenges.json"
    training = _load(train_q) if train_q.exists() else {}
    # Routing may skip TTT only for byte-equivalent public-training tasks. The
    # more expensive D4/color-canonical retrieval remains available inside
    # starter.py, but never controls the task budget before it has produced an
    # actual exact answer.
    raw_training_index = {json.dumps(task, sort_keys=True, separators=(",", ":")): key for key, task in training.items()}
    routes: dict[str, Any] = {}
    action_counts: Counter[str] = Counter()
    for task_id, task in challenges.items():
        exact_source = raw_training_index.get(json.dumps(task, sort_keys=True, separators=(",", ":")))
        exact_all = exact_source is not None
        stats = _task_complexity(task)
        test_grid = as_grid(task["test"][0]["input"]) if task.get("test") else np.zeros((1, 1), dtype=np.int8)
        shapes = infer_output_shapes(task, test_grid, max_shapes=8)
        top_shape_score = float(shapes[0][2]) if shapes else 0.0
        high_shape_confidence = top_shape_score >= 5.0

        if exact_all:
            augments, action = 0, "skip_exact_canonical_retrieval"
        elif profile.upper() == "PARITY_FULL":
            augments, action = 15, "full_128_parity"
        elif profile.upper() == "COVERAGE_FAST":
            if stats["shape_change"] == 0.0 and stats["max_input_cells"] <= 225 and high_shape_confidence:
                augments, action = 3, "light_32_simple"
            else:
                augments, action = 7, "standard_64_coverage"
        else:
            hard = (
                stats["shape_change"] > 0.0
                or stats["max_input_cells"] > 400
                or stats["mean_colors"] >= 5.0
                or not high_shape_confidence
            )
            if hard:
                augments, action = 15, "full_128_hard"
            else:
                augments, action = 7, "standard_64_simple"
        action_counts[action] += 1
        routes[task_id] = {
            "train_augments": int(augments),
            "skip_ttt": bool(exact_all),
            "exact_retrieval": bool(exact_all),
            "exact_training_source": exact_source,
            "action": action,
            "stats": stats,
            "shape_top_score": top_shape_score,
            "shape_hypotheses": [[int(h), int(w), float(score), reason] for h, w, score, reason in shapes[:5]],
            "reason": "Dataset-aware static prior; final cap is recomputed from remaining wall time inside each GPU worker.",
        }
    atomic_json(out / "safe_routes.json", routes)
    report = {
        "profile": profile,
        "challenge_file": challenge_name,
        "tasks": len(routes),
        "outputs": _count_outputs(challenges),
        "action_counts": dict(action_counts),
        "route_path": str(out / "safe_routes.json"),
        "uses_evaluation_solutions": False,
        "symbolic_confidence_reduces_ttt": False,
    }
    atomic_json(out / "safe_route_report.json", report)
    return report


def make_initial_submission(output_path: str | Path) -> dict[str, Any]:
    comp = find_competition_dir()
    rerun = env_truthy("KAGGLE_IS_COMPETITION_RERUN")
    challenge_name = "arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json"
    challenges = _load(comp / challenge_name)
    submission: dict[str, Any] = {}
    for task_id, task in challenges.items():
        rows = []
        for test_ex in task.get("test", []):
            grid = as_grid(test_ex["input"])
            second = grid.T.copy()
            if np.array_equal(grid, second):
                second = np.full(grid.shape, (int(grid[0, 0]) + 1) % 10, dtype=np.int8)
            rows.append({"attempt_1": grid.tolist(), "attempt_2": second.tolist()})
        submission[task_id] = rows
    atomic_json(output_path, submission)
    return submission


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", default="/kaggle/working/arc60_dataset_safe")
    parser.add_argument("--profile", default="DATASET_SAFE_MAX")
    parser.add_argument("--audit", action="store_true")
    parser.add_argument("--routes", action="store_true")
    parser.add_argument("--initial-submission", default="")
    args = parser.parse_args()
    payload: dict[str, Any] = {}
    if args.audit:
        payload["audit"] = dataset_audit(args.output_dir, run_cpu_public_audit=True)
    if args.routes:
        payload["routes"] = build_safe_routes(args.output_dir, args.profile)
    if args.initial_submission:
        payload["initial_submission_tasks"] = len(make_initial_submission(args.initial_submission))
    print(json.dumps(payload, indent=2))


In [ ]:
%%writefile arc_decoder.py
from __future__ import annotations

import bz2
import math
import os
import pickle
from collections import Counter, defaultdict
from typing import Any, Callable

import numpy as np




def _env_truthy(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "1" if default else "0").strip().lower()
    return raw not in ("", "0", "false", "no", "none")


def _allow_symbolic_attempt2() -> bool:
    return _env_truthy("ARC_ALLOW_SYMBOLIC_ATTEMPT2", False)

def hashable(guess: Any) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(map(int, row)) for row in np.asarray(guess))


def _answer_len(solution: Any) -> int:
    grid = np.asarray(solution)
    return int(grid.size + max(0, grid.shape[0] - 1) + 1)  # cells + newlines + EOS


def _beam_norm(sample: dict[str, Any]) -> float:
    if "beam_nll" in sample:
        return float(sample["beam_nll"])
    return float(sample.get("beam_score", 99.0)) / max(1, int(sample.get("beam_len", _answer_len(sample["solution"]))))


def _aug_norms(sample: dict[str, Any]) -> list[float]:
    if sample.get("score_aug_norm"):
        return [float(x) for x in sample["score_aug_norm"]]
    length = max(1, int(sample.get("answer_len", _answer_len(sample["solution"]))))
    return [float(x) / length for x in sample.get("score_aug", [])]


def group_outputs(guesses: dict[str, dict[str, Any]]) -> dict[tuple[tuple[int, ...], ...], dict[str, Any]]:
    groups: dict[tuple[tuple[int, ...], ...], dict[str, Any]] = {}
    for key, sample in guesses.items():
        grid = np.asarray(sample["solution"], dtype=np.int8)
        h = hashable(grid)
        record = groups.setdefault(h, {"solution": grid, "samples": [], "keys": []})
        record["samples"].append(sample)
        record["keys"].append(key)
    return groups


def score_sum(guesses: dict[str, dict[str, Any]], getter: Callable[[list[dict[str, Any]]], float]) -> list[np.ndarray]:
    scored = [(getter(record["samples"]), record["solution"]) for record in group_outputs(guesses).values()]
    scored.sort(key=lambda item: item[0], reverse=True)
    return [grid for _, grid in scored]


def getter_full_probmul_3(samples: list[dict[str, Any]], baseline: float = 3.0) -> float:
    inference = sum(baseline - float(sample.get("beam_score", baseline)) for sample in samples)
    augmented = [sum(baseline - float(score) for score in sample.get("score_aug", [])) for sample in samples]
    return float(inference + (np.mean(augmented) if augmented else 0.0))


def score_full_probmul_3(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    return score_sum(guesses, getter_full_probmul_3)


def getter_kgmon(samples: list[dict[str, Any]]) -> float:
    augmented = [np.mean(sample["score_aug"]) for sample in samples if sample.get("score_aug")]
    return float(len(samples) - (np.mean(augmented) if augmented else 50.0))


def score_kgmon(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    return score_sum(guesses, getter_kgmon)


def _geometry_view(key: str) -> str:
    ops = [part for part in key.split(".")[1:] if part in ("transpose", "rot90")]
    return ".".join(ops) or "id"


def normalized_group_score(record: dict[str, Any]) -> float:
    samples = record["samples"]
    support = len(samples)
    views = len({_geometry_view(key) for key in record.get("keys", [])})
    beam = np.asarray([_beam_norm(sample) for sample in samples], dtype=float)
    aug = [value for sample in samples for value in _aug_norms(sample)]
    aug_mean = float(np.median(aug)) if aug else 0.09
    structural = float(np.mean([sample.get("structural_score", 0.5) for sample in samples]))
    shape_score = float(max(sample.get("shape_score", 0.0) for sample in samples))
    symbolic = float(max(sample.get("symbolic_confidence", 0.0) for sample in samples))
    source_bonus = 8.0 if (_allow_symbolic_attempt2() and symbolic >= 0.999) else 0.0
    return float(
        1.45 * math.log1p(support)
        + 0.22 * views
        - 45.0 * float(np.median(beam))
        - 65.0 * aug_mean
        + 0.55 * structural
        + 0.08 * shape_score
        + source_bonus
    )


def best_likelihood_score(record: dict[str, Any]) -> float:
    samples = record["samples"]
    best_beam = min(_beam_norm(sample) for sample in samples)
    aug_means = [float(np.mean(_aug_norms(sample))) for sample in samples if _aug_norms(sample)]
    best_aug = min(aug_means) if aug_means else 0.09
    structural = max(float(sample.get("structural_score", 0.5)) for sample in samples)
    return float(-55.0 * best_beam - 55.0 * best_aug + 0.35 * structural + 0.15 * math.log1p(len(samples)))


def score_normalized(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    records = sorted(group_outputs(guesses).values(), key=normalized_group_score, reverse=True)
    return [record["solution"] for record in records]


def score_best_likelihood(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    records = sorted(group_outputs(guesses).values(), key=best_likelihood_score, reverse=True)
    return [record["solution"] for record in records]


def _strict_symbolic_grids(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    candidates: list[tuple[float, np.ndarray]] = []
    for record in group_outputs(guesses).values():
        confidence = max(float(sample.get("symbolic_confidence", 0.0)) for sample in record["samples"])
        if confidence >= 0.999:
            candidates.append((confidence, record["solution"]))
    candidates.sort(key=lambda item: item[0], reverse=True)
    return [grid for _, grid in candidates]


def _neural_pool(guesses: dict[str, dict[str, Any]]) -> dict[str, dict[str, Any]]:
    return {
        key: sample
        for key, sample in guesses.items()
        if sample.get("source", "llm") not in (("fallback", "symbolic_low") if _allow_symbolic_attempt2() else ("fallback", "symbolic_low", "symbolic"))
    }


def _fallback_order(guesses: dict[str, dict[str, Any]]) -> list[np.ndarray]:
    records = group_outputs(guesses).values()
    ordered = sorted(
        records,
        key=lambda rec: (
            max(float(sample.get("symbolic_confidence", 0.0)) for sample in rec["samples"]),
            max(float(sample.get("fallback_priority", -99.0)) for sample in rec["samples"]),
        ),
        reverse=True,
    )
    return [record["solution"] for record in ordered]


def score_portfolio(guesses: dict[str, dict[str, Any]], n_guesses: int = 2) -> list[np.ndarray]:
    """Conservative pass@2 portfolio.

    Attempt 1 preserves the proven consensus/NLL selector unless at least three
    independent selectors agree on another grid. Attempt 2 uses Borda evidence
    and reserves a slot for a uniquely verified symbolic program.
    """
    if not guesses:
        return []

    retrieval_groups = {}
    for key, sample in guesses.items():
        if sample.get("source") == "retrieval":
            retrieval_groups[hashable(sample["solution"])] = np.asarray(sample["solution"], dtype=np.int8)
    if len(retrieval_groups) == 1:
        exact = next(iter(retrieval_groups.values()))
        if n_guesses <= 1:
            return [exact]
        exact_hash = hashable(exact)
        remaining = {
            key: sample for key, sample in guesses.items()
            if hashable(sample["solution"]) != exact_hash and sample.get("source") != "retrieval"
        }
        return [exact] + score_portfolio(remaining, n_guesses - 1)

    neural = _neural_pool(guesses)
    if not neural:
        return _fallback_order(guesses)[:n_guesses]

    orders = [
        score_kgmon(neural),
        score_full_probmul_3(neural),
        score_normalized(neural),
        score_best_likelihood(neural),
    ]
    orders = [order for order in orders if order]
    if not orders:
        return _fallback_order(guesses)[:n_guesses]

    top_votes = Counter(hashable(order[0]) for order in orders)
    consensus_hash, votes = top_votes.most_common(1)[0]
    first = next(grid for order in orders for grid in order if hashable(grid) == consensus_hash) if votes >= 3 else orders[0][0]
    selected = [np.asarray(first, dtype=np.int8)]
    if n_guesses <= 1:
        return selected

    rank_points = (8.0, 4.0, 2.0, 1.0, 0.4, 0.2)
    weights = (1.15, 1.0, 1.0, 0.85)
    points: dict[tuple[tuple[int, ...], ...], float] = defaultdict(float)
    grids: dict[tuple[tuple[int, ...], ...], np.ndarray] = {}
    for weight, order in zip(weights, orders):
        for rank, grid in enumerate(order[: len(rank_points)]):
            h = hashable(grid)
            grids[h] = np.asarray(grid, dtype=np.int8)
            points[h] += weight * rank_points[rank]
    first_hash = hashable(first)
    points.pop(first_hash, None)

    strict_symbolic = [grid for grid in _strict_symbolic_grids(guesses) if hashable(grid) != first_hash] if _allow_symbolic_attempt2() else []
    if strict_symbolic:
        sym_hash = hashable(strict_symbolic[0])
        grids[sym_hash] = strict_symbolic[0]
        points[sym_hash] += 20.0

    if points:
        second_hash = max(points, key=points.get)
        selected.append(grids[second_hash])
    else:
        for grid in _fallback_order(guesses):
            if hashable(grid) != first_hash:
                selected.append(grid)
                break
    return selected[:n_guesses]


selection_algorithms = [score_full_probmul_3, score_kgmon, score_normalized, score_best_likelihood]


class ArcDecoder:
    def __init__(self, dataset: Any, n_guesses: int):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results: dict[str, dict[str, dict[str, Any]]] = {}

    def load_decoded_results(self, store: str, run_name: str = "") -> None:
        if not os.path.isdir(store):
            print(f"*** Result directory does not exist: {store}")
            return
        loaded = 0
        for filename in sorted(os.listdir(store)):
            path = os.path.join(store, filename)
            if not os.path.isfile(path):
                continue
            try:
                with bz2.BZ2File(path, "rb") as f:
                    outputs = pickle.load(f)
            except Exception as exc:
                print(f"*** Skip unreadable result {filename}: {exc}")
                continue
            if not isinstance(outputs, list):
                continue
            base_key = filename.split(".")[0]
            target = self.decoded_results.setdefault(base_key, {})
            for i, sample in enumerate(outputs):
                if isinstance(sample, dict) and "solution" in sample:
                    target[f"{filename}{run_name}.out{i}"] = sample
                    loaded += 1
        print(f"*** Loaded {loaded} candidate records for {len(self.decoded_results)} test outputs")

    def run_selection_algo(self, selection_algorithm: Callable | None = None) -> dict[str, list[np.ndarray]]:
        if selection_algorithm is None:
            return {base_key: score_portfolio(values, self.n_guesses) for base_key, values in self.decoded_results.items()}
        return {base_key: selection_algorithm(values)[: self.n_guesses] for base_key, values in self.decoded_results.items()}

    def benchmark_selection_algos(self) -> None:
        print("*** Benchmark selection algorithms...")
        labels: dict[str, np.ndarray] = {}
        tasks_per_puzzle: dict[str, int] = {}
        solved_subkeys = 0
        total_subkeys = 0

        for base_key, values in self.decoded_results.items():
            if base_key not in self.dataset.replies:
                continue
            puzzle, test_nr = base_key.rsplit("_", 1)
            tasks_per_puzzle[puzzle] = max(tasks_per_puzzle.get(puzzle, 0), int(test_nr) + 1)
            target = np.asarray(self.dataset.replies[base_key][0])
            labels[base_key] = target
            for subkey, sample in values.items():
                solution = np.asarray(sample["solution"])
                if solution.shape == target.shape and np.array_equal(solution, target):
                    solved_subkeys += 1
                    print(
                        f"ALL_CORRECT beam={float(sample.get('beam_score', np.nan)):8.5f} "
                        f"shape={solution.shape[0]}x{solution.shape[1]} [{subkey}]"
                    )
                total_subkeys += 1
        print(f" subkeys: {solved_subkeys}/{total_subkeys}")

        algorithms: list[tuple[str, Callable[[dict[str, dict[str, Any]]], list[np.ndarray]]]] = [
            (algo.__name__, lambda values, a=algo: a(values)[: self.n_guesses]) for algo in selection_algorithms
        ]
        algorithms.append(("score_portfolio", lambda values: score_portfolio(values, self.n_guesses)))
        for name, algorithm in algorithms:
            selected = {base_key: algorithm(values) for base_key, values in self.decoded_results.items()}
            correct = {
                key
                for key, grids in selected.items()
                if key in labels and any(np.array_equal(grid, labels[key]) for grid in grids)
            }
            score = sum(1.0 / tasks_per_puzzle[key.rsplit("_", 1)[0]] for key in correct)
            print(f" acc: {score:5.1f}/{len(tasks_per_puzzle):3d} ('{name}') solved={sorted(correct)}")


In [ ]:
%%writefile arc_loader.py
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from typing import Any, Callable

import numpy as np


def stable_seed(text: str, modulo: int = 2**31 - 1) -> int:
    """Stable cross-process seed (unlike Python's randomized hash())."""
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest(), "little") % modulo


def convert_grid_to_string(grid: Any) -> str:
    return "\n".join("".join(str(int(cell)) for cell in row) for row in grid)


def is_valid_solution(guess: Any) -> bool:
    return (
        isinstance(guess, np.ndarray)
        and guess.ndim == 2
        and all(0 < int(x) <= 30 for x in guess.shape)
        and bool(np.all((guess >= 0) & (guess <= 9)))
    )


def shuffled(data_list: list[Any]) -> list[Any]:
    return np.random.permutation(data_list).tolist()


def permute_mod(a: Any, descriptor: str, invert: bool = False) -> np.ndarray:
    permutation = [int(i) for i in descriptor if i.isdigit()]
    if sorted(permutation) != list(range(10)):
        raise ValueError(f"Bad color permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim == 3:
        if not invert:
            permutation = np.argsort(permutation)
        return a[..., permutation]
    if a.ndim != 2:
        raise ValueError(f"ARC grid must be 2-D, got {a.ndim}-D")
    if invert:
        permutation = np.argsort(permutation)
    return np.asarray(permutation)[a]


def permute_rnd_all_(_query: Any) -> str:
    return "permute" + "".join(map(str, np.random.permutation(10).tolist()))


def permute_rnd_keep0_(_query: Any) -> str:
    return "permute" + "".join(map(str, [0] + (np.random.permutation(9) + 1).tolist()))


@dataclass(frozen=True)
class ArcTokenSpec:
    digit_ids: tuple[int, ...]
    newline_id: int
    eos_id: int
    pad_id: int
    assistant_marker: tuple[int, ...]
    user_marker: tuple[int, ...]

    @classmethod
    def from_tokenizer(cls, tokenizer: Any) -> "ArcTokenSpec":
        def enc(text: str) -> list[int]:
            try:
                return list(map(int, tokenizer.encode(text, add_special_tokens=False)))
            except TypeError:
                return list(map(int, tokenizer.encode(text)))

        digits: list[int] = []
        for i in range(10):
            ids = enc(str(i))
            if len(ids) != 1:
                raise RuntimeError(f"ARC checkpoint must tokenize digit {i} as one token; got {ids}")
            digits.append(ids[0])
        newline = enc("\n")
        eos = enc("<|im_end|>")
        user = enc("<|im_start|>user\n")
        assistant = enc("<|im_start|>assistant\n")
        if len(newline) != 1 or len(eos) != 1 or not user or not assistant:
            raise RuntimeError(
                "Unexpected tokenizer layout: "
                f"newline={newline}, eos={eos}, user={user}, assistant={assistant}"
            )
        pad = tokenizer.pad_token_id
        if pad is None:
            pad = getattr(tokenizer, "eos_token_id", eos[0])
        return cls(
            digit_ids=tuple(digits),
            newline_id=newline[0],
            eos_id=eos[0],
            pad_id=int(pad),
            assistant_marker=tuple(assistant),
            user_marker=tuple(user),
        )

    def schedule(self, shape: tuple[int, int]) -> list[tuple[int, ...]]:
        """Allowed token IDs at each output position for an exact HxW grid."""
        h, w = map(int, shape)
        if not (1 <= h <= 30 and 1 <= w <= 30):
            raise ValueError(f"Invalid ARC output shape: {shape}")
        out: list[tuple[int, ...]] = []
        for y in range(h):
            out.extend([self.digit_ids] * w)
            if y + 1 < h:
                out.append((self.newline_id,))
        out.append((self.eos_id,))
        return out

    def tokens_to_grid(self, tokens: list[int], expected_shape: tuple[int, int] | None = None) -> np.ndarray | None:
        tokens = list(map(int, tokens))
        if tokens and tokens[-1] == self.eos_id:
            tokens = tokens[:-1]
        reverse = {tok: i for i, tok in enumerate(self.digit_ids)}
        rows: list[list[int]] = [[]]
        for tok in tokens:
            if tok == self.newline_id:
                rows.append([])
            elif tok in reverse:
                rows[-1].append(reverse[tok])
            else:
                return None
        if rows and not rows[-1]:
            rows.pop()
        if not rows or not rows[0] or len({len(row) for row in rows}) != 1:
            return None
        arr = np.asarray(rows, dtype=np.int8)
        if expected_shape is not None and arr.shape != tuple(expected_shape):
            return None
        return arr if is_valid_solution(arr) else None


class QwenFormatter:
    def __init__(self, tokenizer: Any):
        self.tokenizer = tokenizer
        self.tokens = ArcTokenSpec.from_tokenizer(tokenizer)

    def fmt_query(self, query: list[dict[str, Any]]) -> str:
        return "<|im_start|>user\n" + convert_grid_to_string(query[0]["input"]) + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply: list[Any]) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train: list[dict[str, Any]], last_is_challenge: bool = False) -> str:
        # In TTFT mode the shuffled last demonstration is the held-out completion.
        examples = list(train)
        if last_is_challenge and examples:
            test = examples[-1]
            examples = examples[:-1]
        else:
            test = None
        text = ""
        for ex in examples:
            text += self.fmt_query([ex]) + self.fmt_reply([ex["output"]])
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self) -> int:
        return len(self.tokens.schedule((30, 30))) + 1

    def convert_tokens_to_array(
        self,
        tokens: list[int],
        limit_rows: int = 30,
        expected_shape: tuple[int, int] | None = None,
    ) -> np.ndarray | None:
        arr = self.tokens.tokens_to_grid(tokens, expected_shape=expected_shape)
        if arr is not None and arr.shape[0] <= limit_rows:
            return arr
        # Compatibility fallback for tokenizers with surprising decode behavior.
        try:
            text_tokens = list(tokens)
            if text_tokens and int(text_tokens[-1]) == self.tokens.eos_id:
                text_tokens = text_tokens[:-1]
            lines = self.tokenizer.decode(text_tokens).strip().split("\n")
            rows = [[int(ch) for ch in line if ch.isdigit()] for line in lines]
            rows = [row for row in rows if row][:limit_rows]
            if not rows or len({len(row) for row in rows}) != 1:
                return None
            arr = np.asarray(rows, dtype=np.int8)
            if expected_shape is not None and arr.shape != tuple(expected_shape):
                return None
            return arr if is_valid_solution(arr) else None
        except Exception:
            return None


class ArcDataset:
    @staticmethod
    def forward_mod(a: Any, key: str, use_perm: bool = True) -> Any:
        if a is None:
            return None
        out = np.asarray(a)
        for op in key.split(".")[1:]:
            if op == "rot90":
                out = np.rot90(out)
            elif op == "transpose":
                out = np.swapaxes(out, 0, 1)
            elif op.startswith("permute"):
                out = permute_mod(out, op, invert=False) if use_perm else out
            elif op.startswith("copy"):
                out = np.copy(out)
            elif op.startswith(("out", "ex", "run")):
                pass
            else:
                raise NotImplementedError(f"Unknown forward operation '{op}'")
        return out

    @staticmethod
    def invert_mod(a: Any, key: str, inv_perm: bool = True) -> Any:
        if a is None:
            return None
        out = np.asarray(a)
        for op in key.split(".")[1:][::-1]:
            if op == "rot90":
                out = np.rot90(out, k=3)
            elif op == "transpose":
                out = np.swapaxes(out, 0, 1)
            elif op.startswith("permute"):
                out = permute_mod(out, op, invert=True) if inv_perm else out
            elif op.startswith("copy"):
                out = np.copy(out)
            elif op.startswith(("out", "ex", "run")):
                pass
            else:
                raise NotImplementedError(f"Unknown inverse operation '{op}'")
        return out

    def __init__(
        self,
        queries: dict[str, Any],
        replies: dict[str, Any] | None = None,
        keys: list[str] | None = None,
        is_orig: bool = False,
    ):
        replies = {} if replies is None else replies
        if keys is not None:
            keys = [key for key in keys if key is not None]
        self.queries = queries if keys is None else {key: queries[key] for key in keys}
        self.replies = replies if keys is None else {key: replies[key] for key in keys if key in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries) if keys is None else list(keys)
        self.transposed_dataset: ArcDataset | None = None

    def __len__(self) -> int:
        return len(self.keys)

    def change_keys(self, keys: list[str], keep_flags: bool = False) -> "ArcDataset":
        flags = {"is_orig": self.is_orig} if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file: str, keys: list[str] | None = None) -> "ArcDataset":
        with open(queries_file, "r", encoding="utf-8") as f:
            queries = json.load(f)
        return cls(queries=queries, is_orig=True, keys=keys)

    def load_replies(self, replies_file: str) -> "ArcDataset":
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file, "r", encoding="utf-8") as f:
            parsed = json.load(f)
        self.replies = {key: parsed[key] for key in self.keys}
        return self

    def split_multi_replies(self) -> "ArcDataset":
        indices = [(key, i) for key in self.keys for i in range(len(self.queries[key]["test"]))]
        return self.__class__(
            keys=[f"{key}_{i}" for key, i in indices],
            queries={
                f"{key}_{i}": {"train": self.queries[key]["train"], "test": [self.queries[key]["test"][i]]}
                for key, i in indices
            },
            replies={f"{key}_{i}": [self.replies[key][i]] for key, i in indices if key in self.replies},
        )

    def shuffled(self) -> "ArcDataset":
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    @classmethod
    def append(cls, *datasets: "ArcDataset | None") -> "ArcDataset":
        valid = [d for d in datasets if d is not None and d.keys]
        if not valid:
            return cls({}, {})
        return cls(
            queries={key: value for dataset in valid for key, value in dataset.queries.items()},
            replies={key: value for dataset in valid for key, value in dataset.replies.items()},
            keys=[key for dataset in valid for key in dataset.keys],
        )

    def mod_single(
        self,
        mod_func: Callable[..., Any],
        descriptor: str | Callable[[Any], str] | None,
        i: int,
        keep_key: bool,
        inputs_only: bool,
    ) -> "ArcDataset":
        queries: dict[str, Any] = {}
        replies: dict[str, Any] = {}
        keys: list[str] = []
        for key0 in self.keys:
            if descriptor is None:
                raw_desc = "copy{i}" if mod_func is np.copy else mod_func.__name__
            elif isinstance(descriptor, str):
                raw_desc = descriptor
            else:
                raw_desc = descriptor(self.queries[key0])
            desc = raw_desc.format(i=i)

            def func(a: Any, d: str) -> list[Any]:
                value = mod_func(a) if descriptor is None else mod_func(a, d)
                return np.asarray(value).tolist()

            key1 = key0 if keep_key else f"{key0}.{'I' if inputs_only else ''}{desc}"
            keys.append(key1)
            queries[key1] = {
                mode: [
                    {
                        field: (func(array, desc) if field == "input" or not inputs_only else array)
                        for field, array in example.items()
                    }
                    for example in examples
                ]
                for mode, examples in self.queries[key0].items()
            }
            if key0 in self.replies:
                replies[key1] = [func(array, desc) for array in self.replies[key0]]
        return self.__class__(queries=queries, replies=replies, keys=keys)

    def mod(
        self,
        mod_func: Callable[..., Any],
        descriptor: str | Callable[[Any], str] | None = None,
        n: int = 1,
        stack: bool | None = None,
        keep: bool = False,
        keep_key: bool = False,
        shuffle: bool = False,
        join: bool = True,
        inputs_only: bool = False,
    ) -> "ArcDataset | list[ArcDataset]":
        if keep and keep_key:
            raise ValueError("keep and keep_key are mutually exclusive")
        cur: ArcDataset = self
        out: list[ArcDataset] = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None:
            stack = mod_func.__name__.startswith("rot")
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            out.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*out) if join else out

    def geometric(self) -> "ArcDataset":
        data = self.mod(np.transpose, keep=True)
        assert isinstance(data, ArcDataset)
        data = data.mod(np.rot90, n=3, keep=True)
        assert isinstance(data, ArcDataset)
        return data

    def get(self, key: str, formatter: QwenFormatter) -> dict[str, str]:
        train = formatter.fmt_train(self.queries[key]["train"])
        query = formatter.fmt_query(self.queries[key]["test"])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ""
        # TTFT uses leave-one-demonstration-out completion text.
        text = train + query + reply if reply else formatter.fmt_train(self.queries[key]["train"], last_is_challenge=True)
        return {"key": key, "train": train, "query": query, "reply": reply, "input": train + query, "text": text}

    def as_list(self, formatter: QwenFormatter) -> list[dict[str, str]]:
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key: str, formatter: QwenFormatter | None, name: str, max_of_transposed: bool = False) -> int:
        if formatter is None:
            if name == "input":
                return int(sum(np.prod(np.shape(v)) for groups in self.queries[key].values() for ex in groups for v in ex.values()))
            if name == "reply":
                return int(sum(np.prod(np.shape(v)) for v in self.replies[key]))
            raise ValueError(name)
        datasets = [self]
        if max_of_transposed:
            if self.transposed_dataset is None:
                transformed = self.mod(np.transpose, keep=False, keep_key=True)
                assert isinstance(transformed, ArcDataset)
                self.transposed_dataset = transformed
            datasets.append(self.transposed_dataset)
        return max(len(formatter.tokenizer.encode(ds.get(key, formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter: QwenFormatter, name: str, max_len: int, from_end: bool = False) -> "ArcDataset":
        temp = self.change_keys(self.keys)
        new_keys: list[str] = []
        new_queries: dict[str, Any] = {}
        new_replies: dict[str, Any] = {}
        for original_key in self.keys:
            key = original_key
            reply = temp.replies.get(key)
            while temp.get_length(key, formatter, name) > max_len:
                query = temp.queries[key]
                if len(query["train"]) <= 1:
                    break
                if not key.split(".")[-1].startswith("ex"):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                parts = key.split(".")
                payload = parts[-1][2:]
                if not payload:
                    break
                payload = payload[:-1] if from_end else payload[1:]
                key = ".".join(parts[:-1] + [f"ex{payload}"])
                temp.queries[key] = {
                    mode: ((examples[:-1] if from_end else examples[1:]) if mode == "train" else examples)
                    for mode, examples in query.items()
                }
                if reply is not None:
                    temp.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp.queries[key]
            if reply is not None:
                new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def shuffle_ex(self, perm: Any = None, keep_max: int | None = None) -> "ArcDataset":
        new_keys: list[str] = []
        new_queries: dict[str, Any] = {}
        new_replies: dict[str, Any] = {}
        for key in self.keys:
            n = len(self.queries[key]["train"])
            p = np.random.permutation(n) if perm is None else np.asarray(perm)
            if keep_max is not None:
                p = p[:keep_max]
            separator = "-" if len(p) and int(p.max()) > 9 else ""
            new_key = f"{key}.ex" + separator.join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {
                mode: (np.asarray(examples, dtype=object)[p].tolist() if mode == "train" else examples)
                for mode, examples in self.queries[key].items()
            }
            if key in self.replies:
                new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(
        self,
        n: int = 1,
        shfl_keys: bool = False,
        seed: int = 42,
        descriptor: Callable[[Any], str] = permute_rnd_all_,
        include_identity: bool = False,
    ) -> "ArcDataset":
        np.random.seed(seed)
        geom = self.geometric()
        perm = geom.mod(permute_mod, descriptor, n=n, shuffle=shfl_keys, keep=False)
        assert isinstance(perm, ArcDataset)
        data = self.__class__.append(geom, perm) if include_identity else perm
        return data.shuffle_ex()

    def augment_train_mixed(self, n_total: int = 15, seed: int = 42) -> "ArcDataset":
        """128 TTFT examples at n_total=15: 8 identity + 64 full + 56 zero-preserving."""
        np.random.seed(seed)
        geom = self.geometric().shuffle_ex()
        n_full = (n_total + 1) // 2
        n_keep0 = n_total - n_full
        full = self.geometric().mod(permute_mod, permute_rnd_all_, n=n_full, shuffle=True)
        assert isinstance(full, ArcDataset)
        full = full.shuffle_ex()
        keep0: ArcDataset | None = None
        if n_keep0:
            keep0_data = self.geometric().mod(permute_mod, permute_rnd_keep0_, n=n_keep0, shuffle=True)
            assert isinstance(keep0_data, ArcDataset)
            keep0 = keep0_data.shuffle_ex()
        return self.__class__.append(geom, full, keep0).shuffled()

    @staticmethod
    def geometry_group(key: str) -> tuple[bool, int]:
        ops = key.split(".")[1:]
        transposed = "transpose" in ops
        rotations = sum(op == "rot90" for op in ops) % 4
        # This tuple uniquely determines whether H/W are swapped for all grids.
        return transposed, rotations % 2

    @staticmethod
    def transformed_shape(shape: tuple[int, int], key: str) -> tuple[int, int]:
        return tuple(ArcDataset.forward_mod(np.zeros(shape, dtype=np.int8), key, use_perm=False).shape)

    def get_submission(self, results: dict[str, list[np.ndarray]] | None = None) -> dict[str, Any]:
        if not self.is_orig:
            raise AssertionError("Must be run on original dataset")
        submission = {
            key: [{f"attempt_{i + 1}": [[0]] for i in range(2)} for _ in range(len(self.queries[key]["test"]))]
            for key in self.keys
        }
        if results is not None:
            self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results: dict[str, list[np.ndarray]], submission: dict[str, Any]) -> None:
        print(f"*** Generating submission for {len(results)} outputs...")
        for key, grids in results.items():
            base_id, base_nr = key.rsplit("_", 1)
            if base_id not in submission or int(base_nr) >= len(submission[base_id]):
                continue
            target = submission[base_id][int(base_nr)]
            for i, grid in enumerate(grids[: len(target)]):
                target[f"attempt_{i + 1}"] = np.asarray(grid, dtype=int).tolist()

    def validate_submission(self, submission: dict[str, Any]) -> float:
        if not self.is_orig:
            raise AssertionError("Must be run on original dataset")
        score = 0.0
        for key, replies in self.replies.items():
            for i, target in enumerate(replies):
                if any(np.array_equal(target, submission[key][i][attempt]) for attempt in ("attempt_1", "attempt_2")):
                    score += 1.0 / len(replies)
        return score


In [ ]:
%%writefile arc_retrieval.py
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from typing import Any, Iterable

import numpy as np

from arc_symbolic import D4_NAMES, as_grid, d4, grid_key, valid_grid


INVERSE_D4 = {
    "id": "id",
    "r1": "r3",
    "r2": "r2",
    "r3": "r1",
    "t": "t",
    "tr1": "tr1",
    "tr2": "tr2",
    "tr3": "tr3",
}


@dataclass(frozen=True)
class CanonicalTask:
    fingerprint: str
    representation: str
    transform: str
    color_to_canonical: dict[int, int]
    canonical_to_color: dict[int, int]


@dataclass(frozen=True)
class ReferenceTask:
    task: dict[str, Any]
    solutions: list[Any]
    canonical: CanonicalTask
    source_key: str


def _normalize_colors(arrays: list[np.ndarray]) -> tuple[list[np.ndarray], dict[int, int]]:
    mapping: dict[int, int] = {}
    next_color = 0
    normalized: list[np.ndarray] = []
    for array in arrays:
        out = np.empty_like(array, dtype=np.int8)
        for index, value in np.ndenumerate(array):
            color = int(value)
            if color not in mapping:
                mapping[color] = next_color
                next_color += 1
            out[index] = mapping[color]
        normalized.append(out)
    return normalized, mapping


def _local_pair_key(inp: np.ndarray, out: np.ndarray) -> str:
    normalized, _ = _normalize_colors([inp, out])
    payload = {
        "in_shape": list(inp.shape),
        "out_shape": list(out.shape),
        "in": normalized[0].tolist(),
        "out": normalized[1].tolist(),
    }
    return json.dumps(payload, separators=(",", ":"), sort_keys=True)


def _canonical_for_transform(task: dict[str, Any], transform: str) -> tuple[str, dict[int, int]]:
    transformed_train: list[tuple[np.ndarray, np.ndarray]] = []
    for example in task["train"]:
        inp = d4(as_grid(example["input"]), transform)
        out = d4(as_grid(example["output"]), transform)
        transformed_train.append((inp, out))
    # Demonstration order has no semantic meaning. This key is itself invariant
    # to color labels, making sorting robust to globally permuted duplicates.
    transformed_train.sort(key=lambda pair: _local_pair_key(pair[0], pair[1]))
    transformed_test = [d4(as_grid(example["input"]), transform) for example in task["test"]]

    arrays: list[np.ndarray] = []
    for inp, out in transformed_train:
        arrays.extend([inp, out])
    arrays.extend(transformed_test)
    normalized, mapping = _normalize_colors(arrays)

    cursor = 0
    train_payload = []
    for _ in transformed_train:
        train_payload.append({"input": normalized[cursor].tolist(), "output": normalized[cursor + 1].tolist()})
        cursor += 2
    test_payload = [{"input": normalized[cursor + i].tolist()} for i in range(len(transformed_test))]
    representation = json.dumps({"train": train_payload, "test": test_payload}, separators=(",", ":"), sort_keys=True)
    return representation, mapping


def canonicalize_task(task: dict[str, Any]) -> CanonicalTask:
    variants: list[tuple[str, str, dict[int, int]]] = []
    for transform in D4_NAMES:
        representation, mapping = _canonical_for_transform(task, transform)
        variants.append((representation, transform, mapping))
    representation, transform, mapping = min(variants, key=lambda item: (item[0], item[1]))
    fingerprint = hashlib.sha256(representation.encode("utf-8")).hexdigest()
    inverse = {canonical: original for original, canonical in mapping.items()}
    return CanonicalTask(
        fingerprint=fingerprint,
        representation=representation,
        transform=transform,
        color_to_canonical=dict(mapping),
        canonical_to_color=inverse,
    )


def _solution_to_target(
    reference_solution: Any,
    reference_canonical: CanonicalTask,
    target_canonical: CanonicalTask,
) -> np.ndarray | None:
    transformed = d4(as_grid(reference_solution), reference_canonical.transform)
    canonical = np.empty_like(transformed, dtype=np.int8)
    for index, value in np.ndenumerate(transformed):
        color = int(value)
        if color not in reference_canonical.color_to_canonical:
            return None
        canonical[index] = reference_canonical.color_to_canonical[color]

    target_transformed = np.empty_like(canonical, dtype=np.int8)
    for index, value in np.ndenumerate(canonical):
        canonical_color = int(value)
        if canonical_color not in target_canonical.canonical_to_color:
            return None
        target_transformed[index] = target_canonical.canonical_to_color[canonical_color]
    result = d4(target_transformed, INVERSE_D4[target_canonical.transform])
    return np.asarray(result, dtype=np.int8) if valid_grid(result) else None


class RetrievalIndex:
    """Exact/canonical challenge retrieval with transformation-safe output inversion."""

    def __init__(self):
        self._records: dict[str, list[ReferenceTask]] = {}

    @classmethod
    def from_files(cls, challenge_solution_files: Iterable[tuple[str, str]]) -> "RetrievalIndex":
        index = cls()
        for challenge_path, solution_path in challenge_solution_files:
            try:
                with open(challenge_path, "r", encoding="utf-8") as f:
                    challenges = json.load(f)
                with open(solution_path, "r", encoding="utf-8") as f:
                    solutions = json.load(f)
            except FileNotFoundError:
                continue
            for key, task in challenges.items():
                if key not in solutions:
                    continue
                canonical = canonicalize_task(task)
                index._records.setdefault(canonical.fingerprint, []).append(
                    ReferenceTask(task=task, solutions=solutions[key], canonical=canonical, source_key=key)
                )
        return index

    def __len__(self) -> int:
        return sum(len(values) for values in self._records.values())

    def solve(self, task: dict[str, Any], test_idx: int) -> list[tuple[np.ndarray, str]]:
        target = canonicalize_task(task)
        references = self._records.get(target.fingerprint, [])
        outputs: dict[tuple[tuple[int, ...], ...], tuple[np.ndarray, str]] = {}
        for reference in references:
            if test_idx >= len(reference.solutions):
                continue
            solution = _solution_to_target(reference.solutions[test_idx], reference.canonical, target)
            if solution is None:
                continue
            outputs[grid_key(solution)] = (solution, reference.source_key)
        # A fingerprint collision with incompatible outputs is ambiguous. Returning
        # no answer is safer than asserting a false exact retrieval.
        if len(outputs) != 1:
            return []
        return list(outputs.values())


In [ ]:
%%writefile arc_solver.py
from __future__ import annotations

import bz2
import gc
import io
import json
import logging
import math
import os
import pickle
import sys
import time
import traceback
from collections import defaultdict
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass
from typing import Any, Union

# Unsloth must patch Transformers/PEFT before those packages are imported.
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments

import numpy as np
import torch
from datasets import Dataset
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from transformers import DataCollatorForLanguageModeling

from arc_loader import ArcDataset, QwenFormatter, stable_seed
from arc_symbolic import SymbolicSolver, as_grid, grid_key, infer_output_shapes, structural_consistency

logging.disable(logging.WARNING)
sys.setrecursionlimit(5000)


@dataclass(frozen=True)
class SolverConfig:
    # Environment overrides are deliberate: Kaggle model mounts differ between
    # interactive notebooks and competition reruns. Every resolved checkpoint is
    # still validated by ArcTokenSpec at runtime.
    model_path: str = os.getenv("ARC_MODEL_PATH", "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
    competition_dir: str = os.getenv("ARC_COMPETITION_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    output_dir: str = os.getenv("ARC_OUTPUT_DIR", "/kaggle/inference_outputs")
    max_seq_length: int = int(os.getenv("ARC_MAX_SEQ_LENGTH", "8192"))
    train_augments: int = int(os.getenv("ARC_MAX_TRAIN_AUGMENTS", "15"))
    symbolic_programs: int = int(os.getenv("ARC_SYMBOLIC_PROGRAMS", "500"))
    max_puzzle_seconds: int = int(os.getenv("ARC_MAX_PUZZLE_SECONDS", "900"))
    min_puzzle_seconds: int = int(os.getenv("ARC_MIN_PUZZLE_SECONDS", "240"))
    reserve_scoring_seconds: int = int(os.getenv("ARC_RESERVE_SCORING_SECONDS", "55"))
    constrained_max_score: float = float(os.getenv("ARC_CONSTRAINED_MAX_SCORE", "2.20"))
    unconstrained_max_score: float = float(os.getenv("ARC_UNCONSTRAINED_MAX_SCORE", str(float(-np.log(0.18)))))
    constrained_branch_cap: int = int(os.getenv("ARC_CONSTRAINED_BRANCH_CAP", "4"))
    unconstrained_branch_cap: int = int(os.getenv("ARC_UNCONSTRAINED_BRANCH_CAP", "6"))
    beam_cap: int = int(os.getenv("ARC_BEAM_CAP", "14"))
    max_candidates_per_view: int = int(os.getenv("ARC_MAX_CANDIDATES_PER_VIEW", "6"))
    max_shape_hypotheses: int = int(os.getenv("ARC_MAX_SHAPE_HYPOTHESES", "3"))
    tta_color_permutations: int = int(os.getenv("ARC_TTA_COLOR_PERMUTATIONS", "2"))


CFG = SolverConfig()


class UnslothFixedTrainer(UnslothTrainer):
    """Avoid the view-tensor loss mutation issue in recent torch/Unsloth builds."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped, "_get_name") and "unsloth" in unwrapped._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


def _subsequence_positions(sequence: list[int], pattern: tuple[int, ...]) -> list[int]:
    if not pattern:
        return []
    width = len(pattern)
    return [i for i in range(len(sequence) - width + 1) if tuple(sequence[i : i + width]) == pattern]


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    """Train only assistant grid spans; mask user text and padding."""

    def __init__(self, *args, assistant_marker: tuple[int, ...], eos_id: int, **kwargs):
        super().__init__(*args, **kwargs)
        self.assistant_marker = tuple(assistant_marker)
        self.eos_id = int(eos_id)

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            ids = batch["input_ids"][i].tolist()
            labels = torch.full_like(batch["input_ids"][i], -100)
            for marker_start in _subsequence_positions(ids, self.assistant_marker):
                start = marker_start + len(self.assistant_marker)
                try:
                    end = ids.index(self.eos_id, start) + 1
                except ValueError:
                    continue
                labels[start:end] = batch["input_ids"][i, start:end]
            if not torch.any(labels != -100):
                raise RuntimeError("Completion collator found no assistant answer span")
            batch["labels"][i] = labels
        return batch


def _candidate_tokens(
    logits: torch.Tensor,
    base_scores: list[float],
    allowed: list[tuple[int, ...] | list[int]],
    max_score: float,
    branch_cap: int,
) -> dict[int, list[tuple[float, int]]]:
    n = logits.size(0)
    nll = torch.tensor(base_scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    candidates: dict[int, list[tuple[float, int]]] = {}
    for i in range(n):
        row = [(float(nll[i, token]), int(token)) for token in allowed[i] if float(nll[i, token]) < max_score]
        row.sort(key=lambda item: item[0])
        candidates[i] = row[:branch_cap]
    return candidates


def turbo_dfs_constrained(
    model: Any,
    logits: torch.Tensor,
    schedule: list[tuple[int, ...]],
    step: int,
    max_score: float,
    scores: list[float],
    pos: int,
    cache: Any,
    search_deadline: float,
    pad_id: int,
    branch_cap: int,
    beam_cap: int,
) -> dict[int, list[tuple[float, list[int]]]]:
    n = logits.size(0)
    if step >= len(schedule) or time.time() >= search_deadline:
        return defaultdict(list)
    candidates = _candidate_tokens(logits, scores, [schedule[step]] * n, max_score, branch_cap)
    suffixes: dict[int, list[tuple[float, list[int]]]] = defaultdict(list)

    while time.time() < search_deadline:
        batch_tokens: list[int] = []
        batch_scores: list[float] = []
        alive = 0
        for i in range(n):
            if candidates[i]:
                score, token = candidates[i].pop(0)
                batch_tokens.append(token)
                batch_scores.append(score)
                alive += 1
            else:
                batch_tokens.append(pad_id)
                batch_scores.append(1e6)
        if alive == 0:
            break

        if step + 1 == len(schedule):
            for i, (token, score) in enumerate(zip(batch_tokens, batch_scores)):
                if score < max_score:
                    suffixes[i].append((score, [token]))
            continue

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device, dtype=torch.long),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        children = turbo_dfs_constrained(
            model=model,
            logits=outputs.logits[:, -1],
            schedule=schedule,
            step=step + 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=outputs.past_key_values,
            search_deadline=search_deadline,
            pad_id=pad_id,
            branch_cap=branch_cap,
            beam_cap=beam_cap,
        )
        for batch_id, beams in children.items():
            for score, suffix in beams:
                suffixes[batch_id].append((score, [batch_tokens[batch_id]] + suffix))
            suffixes[batch_id] = sorted(suffixes[batch_id], key=lambda item: item[0])[:beam_cap]
    return suffixes


@torch.no_grad()
def inference_constrained(
    model: Any,
    prefix_tokens: list[list[int]],
    schedule: list[tuple[int, ...]],
    max_score: float,
    search_deadline: float,
    pad_id: int,
    branch_cap: int,
    beam_cap: int,
) -> list[tuple[int, list[tuple[float, list[int]]]]]:
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs_constrained(
        model=model,
        logits=outputs.logits[:, -1],
        schedule=schedule,
        step=0,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        search_deadline=search_deadline,
        pad_id=pad_id,
        branch_cap=branch_cap,
        beam_cap=beam_cap,
    )
    return [(i, sorted(beams, key=lambda item: item[0])[:beam_cap]) for i, beams in suffixes.items()]


# Free-shape DFS still obeys the ARC rectangular-grid grammar.
# State = (completed_rows, current_row_width, fixed_width_or_zero).
def _free_allowed(state: tuple[int, int, int], token_spec: Any) -> list[int]:
    completed, current, width = state
    allowed: list[int] = []
    max_width = width if width else 30
    if current < max_width:
        allowed.extend(token_spec.digit_ids)
    if current > 0 and completed < 29 and (width == 0 or current == width):
        allowed.append(token_spec.newline_id)
    if current > 0 and (width == 0 or current == width):
        allowed.append(token_spec.eos_id)
    return allowed


def _free_next_state(state: tuple[int, int, int], token: int, token_spec: Any) -> tuple[int, int, int]:
    completed, current, width = state
    if token in token_spec.digit_ids:
        return completed, current + 1, width
    if token == token_spec.newline_id:
        return completed + 1, 0, current if width == 0 else width
    return state


def turbo_dfs_unconstrained(
    model: Any,
    logits: torch.Tensor,
    max_new_tokens: int,
    max_score: float,
    scores: list[float],
    states: list[tuple[int, int, int]],
    pos: int,
    cache: Any,
    search_deadline: float,
    token_spec: Any,
    branch_cap: int,
    beam_cap: int,
) -> dict[int, list[tuple[float, list[int]]]]:
    n = logits.size(0)
    if max_new_tokens <= 0 or time.time() >= search_deadline:
        return defaultdict(list)
    allowed = [_free_allowed(state, token_spec) for state in states]
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes: dict[int, list[tuple[float, list[int]]]] = defaultdict(list)
    candidates: dict[int, list[tuple[float, int]]] = {}
    for i in range(n):
        row: list[tuple[float, int]] = []
        for token in allowed[i]:
            score = float(nll[i, token])
            if score >= max_score:
                continue
            if token == token_spec.eos_id:
                suffixes[i].append((score, [token]))
            elif max_new_tokens > 1:
                row.append((score, token))
        row.sort(key=lambda item: item[0])
        candidates[i] = row[:branch_cap]

    while time.time() < search_deadline:
        batch_tokens: list[int] = []
        batch_scores: list[float] = []
        batch_states: list[tuple[int, int, int]] = []
        alive = 0
        for i in range(n):
            if candidates[i]:
                score, token = candidates[i].pop(0)
                batch_tokens.append(token)
                batch_scores.append(score)
                batch_states.append(_free_next_state(states[i], token, token_spec))
                alive += 1
            else:
                batch_tokens.append(token_spec.pad_id)
                batch_scores.append(1e6)
                batch_states.append(states[i])
        if alive == 0:
            break
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device, dtype=torch.long),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )
        children = turbo_dfs_unconstrained(
            model=model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            states=batch_states,
            pos=pos + 1,
            cache=outputs.past_key_values,
            search_deadline=search_deadline,
            token_spec=token_spec,
            branch_cap=branch_cap,
            beam_cap=beam_cap,
        )
        for batch_id, beams in children.items():
            for score, suffix in beams:
                suffixes[batch_id].append((score, [batch_tokens[batch_id]] + suffix))
            suffixes[batch_id] = sorted(suffixes[batch_id], key=lambda item: item[0])[:beam_cap]
    return suffixes


@torch.no_grad()
def inference_unconstrained(
    model: Any,
    prefix_tokens: list[list[int]],
    max_new_tokens: int,
    max_score: float,
    search_deadline: float,
    token_spec: Any,
    branch_cap: int,
    beam_cap: int,
) -> list[tuple[int, list[tuple[float, list[int]]]]]:
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs_unconstrained(
        model=model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        states=[(0, 0, 0)] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        search_deadline=search_deadline,
        token_spec=token_spec,
        branch_cap=branch_cap,
        beam_cap=beam_cap,
    )
    return [(i, sorted(beams, key=lambda item: item[0])[:beam_cap]) for i, beams in suffixes.items()]


@torch.no_grad()
def calc_scores(
    queries: list[str],
    answers: list[str],
    tokenizer: Any,
    model: Any,
    pad_id: int,
) -> tuple[list[float], list[float], list[int]]:
    query_tokens: list[list[int]] = []
    answer_tokens: list[list[int]] = []
    combined: list[list[int]] = []
    for query, answer in zip(queries, answers):
        q = list(map(int, tokenizer.encode(query)))
        a = list(map(int, tokenizer.encode(answer)))
        query_tokens.append(q)
        answer_tokens.append(a)
        combined.append(q + a)
    max_len = max(map(len, combined))
    rows: list[list[int]] = []
    masks: list[list[int]] = []
    for row in combined:
        padding = max_len - len(row)
        rows.append(row + [pad_id] * padding)
        masks.append([1] * len(row) + [0] * padding)
    input_ids = torch.tensor(rows, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(masks, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    logp = outputs.logits.float().cpu().log_softmax(-1)
    totals: list[float] = []
    normalized: list[float] = []
    lengths: list[int] = []
    for logits, q, a in zip(logp, query_tokens, answer_tokens):
        start = len(q) - 1
        answer_logits = logits[start : start + len(a)]
        score = -answer_logits[torch.arange(len(a)), a].sum().item()
        totals.append(float(score))
        normalized.append(float(score / max(1, len(a))))
        lengths.append(len(a))
    return totals, normalized, lengths


def _strict_symbolic_confidence(candidate: Any, candidates: list[Any]) -> float:
    if candidate is None or candidate.confidence < 0.995 or candidate.min_cost > 0.45:
        return 0.0
    margin = candidate.score - (candidates[1].score if len(candidates) > 1 else -99.0)
    simple_d4 = any(name.startswith("d4:") for name in candidate.programs)
    if simple_d4 and (len(candidates) == 1 or margin >= 0.65):
        return 0.999
    return 0.0


def save_symbolic_candidates(puzzle_ds: ArcDataset, puzzle_key: str, output_dir: str) -> dict[str, list[Any]]:
    solver = SymbolicSolver(max_programs=CFG.symbolic_programs, max_outputs=12)
    task = puzzle_ds.queries[puzzle_key]
    hints: dict[str, list[Any]] = {}
    for test_idx in range(len(task["test"])):
        base_key = f"{puzzle_key}_{test_idx}"
        candidates = solver.synthesize(task, test_idx)
        hints[base_key] = candidates
        records: list[dict[str, Any]] = []
        for candidate in candidates[:2]:
            confidence = _strict_symbolic_confidence(candidate, candidates)
            if confidence < 0.999:
                continue
            records.append(
                {
                    "beam_score": 0.0,
                    "beam_nll": 0.0,
                    "beam_len": int(candidate.grid.size + candidate.grid.shape[0]),
                    "score_aug": [],
                    "score_aug_norm": [],
                    "answer_len": int(candidate.grid.size + candidate.grid.shape[0]),
                    "solution": candidate.grid,
                    "source": "symbolic",
                    "symbolic_confidence": confidence,
                    "structural_score": 1.0,
                    "shape_score": 6.0,
                    "programs": candidate.programs,
                }
            )
        if records:
            with bz2.BZ2File(os.path.join(output_dir, f"{base_key}.symbolic"), "wb") as f:
                pickle.dump(records, f)
    return hints


def _shape_plan(task: dict[str, Any], test_idx: int, symbolic_candidates: list[Any]) -> tuple[list[tuple[int, int, float, str]], bool]:
    test_grid = as_grid(task["test"][test_idx]["input"])
    hypotheses = infer_output_shapes(task, test_grid, max_shapes=12)
    by_shape = {(h, w): (score, reason) for h, w, score, reason in hypotheses}
    for candidate in symbolic_candidates or []:
        shape = tuple(candidate.grid.shape)
        old = by_shape.get(shape, (-99.0, ""))
        symbolic_score = 4.5 + 0.2 * float(candidate.confidence)
        if symbolic_score > old[0]:
            by_shape[shape] = (symbolic_score, "symbolic-program")
    ordered = [(shape[0], shape[1], value[0], value[1]) for shape, value in by_shape.items()]
    ordered.sort(key=lambda item: item[2], reverse=True)
    top_score = ordered[0][2] if ordered else 0.0
    if top_score >= 5.0:
        return ordered[:1], False
    if top_score >= 3.0:
        return ordered[:2], True
    return ordered[: CFG.max_shape_hypotheses], True


def _group_eval_subkeys(eval_ds: ArcDataset) -> list[list[str]]:
    grouped: dict[tuple[int, tuple[bool, int]], list[str]] = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        base_key = subkey.split(".")[0]
        test_idx = int(base_key.rsplit("_", 1)[1])
        grouped[(test_idx, ArcDataset.geometry_group(subkey))].append(subkey)
    batches: list[list[str]] = []
    for _, subkeys in sorted(grouped.items()):
        for i in range(0, len(subkeys), 4):
            batches.append(subkeys[i : i + 4])
    return batches


def _adaptive_puzzle_budget(queue: Any, end_time: float, n_workers: int) -> float:
    seconds_left = max(1.0, end_time - time.time())
    try:
        queued = max(0, int(queue.qsize()))
    except Exception:
        queued = n_workers * 8
    # Include currently active workers in the denominator. This converges to the
    # average budget required to cover every queued puzzle before the deadline.
    fair_share = 0.92 * seconds_left * n_workers / max(n_workers, queued + n_workers)
    return float(np.clip(fair_share, CFG.min_puzzle_seconds, CFG.max_puzzle_seconds))


def worker(rank: int, queue: Any, end_time: float, n_workers: int = 4) -> None:
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() not in ("", "0", "false", "no")
    peft_params = dict(
        r=int(os.getenv("ARC60_TTT_LORA_R", "256")),
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=int(os.getenv("ARC60_TTT_LORA_ALPHA", "32")),
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )
    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        max_grad_norm=1.0,
        learning_rate=float(os.getenv("ARC60_TTT_LR", "5e-5")),
        optim=os.getenv("ARC60_TTT_OPTIMIZER", "adamw_torch"),
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CFG.model_path,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=CFG.max_seq_length,
    )
    model = FastLanguageModel.get_peft_model(model, **peft_params)
    for _, parameter in model.named_parameters():
        if parameter.dtype == torch.float32:
            parameter.data = parameter.data.to(torch.bfloat16)

    default_weights = {key: value.clone().detach() for key, value in get_peft_model_state_dict(model, adapter_name="default").items()}
    formatter = QwenFormatter(tokenizer)
    token_spec = formatter.tokens
    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
        assistant_marker=token_spec.assistant_marker,
        eos_id=token_spec.eos_id,
    )
    max_new_tokens = formatter.max_new_tokens()

    filename = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    arc_test_set = ArcDataset.from_file(os.path.join(CFG.competition_dir, filename))
    os.makedirs(CFG.output_dir, exist_ok=True)

    route_path = os.getenv("ARC_SAFE_ROUTE_PATH", "/kaggle/working/arc60_dataset_safe/safe_routes.json")
    try:
        with open(route_path, "r", encoding="utf-8") as route_file:
            route_table = json.load(route_file)
        print(f"[Rank {rank}] loaded dataset-safe route table with {len(route_table)} tasks")
    except Exception as exc:
        route_table = {}
        print(f"[Rank {rank}] safe route table unavailable; using full TTT: {type(exc).__name__}")

    while True:
        if time.time() >= end_time:
            print(f"[Rank {rank}] global deadline reached")
            break
        key = queue.get()
        if key is None:
            break

        try:
            start_time = time.time()
            puzzle_budget = _adaptive_puzzle_budget(queue, end_time, n_workers)
            puzzle_deadline = min(end_time, start_time + puzzle_budget)
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass

            set_peft_model_state_dict(model, default_weights, adapter_name="default")
            puzzle_ds = arc_test_set.change_keys([key])
            symbolic_hints = save_symbolic_candidates(puzzle_ds, key, CFG.output_dir)

            route = route_table.get(key, {}) if isinstance(route_table, dict) else {}
            preferred_augments = int(route.get("train_augments", CFG.train_augments))
            preferred_augments = max(0, min(CFG.train_augments, preferred_augments))

            # The public symbolic lane solved 0/172 evaluation outputs in the
            # supplied data audit. It therefore never earns a reduced TTT budget.
            # Only exact canonical training retrieval may bypass adaptation.
            exact_retrieval = bool(route.get("exact_retrieval", False))
            allow_skip = os.getenv("ARC_ALLOW_EXACT_RETRIEVAL_SKIP", "1").strip().lower() not in ("0", "false", "no")

            # Deadline-safe cap: 15 -> ~128 views, 7 -> ~64, 3 -> ~32.
            # This is based on remaining wall time, not an unvalidated reward model.
            force_full = os.getenv("ARC_FORCE_FULL_TTT", "0").strip().lower() not in ("", "0", "false", "no")
            if force_full:
                deadline_cap = CFG.train_augments
            elif puzzle_budget < 420:
                deadline_cap = min(CFG.train_augments, 3)
            elif puzzle_budget < 690:
                deadline_cap = min(CFG.train_augments, 7)
            else:
                deadline_cap = CFG.train_augments
            task_train_augments = min(preferred_augments, deadline_cap)
            skip_ttt = bool(exact_retrieval and allow_skip) or task_train_augments == 0
            train_runtime = 0.0
            train_loss = float("nan")
            ttt_reverted = False
            if skip_ttt:
                model = FastLanguageModel.for_inference(model)
            else:
                model = FastLanguageModel.for_training(model)
                train_ds = puzzle_ds.augment_train_mixed(
                    n_total=task_train_augments,
                    seed=stable_seed(f"train:{key}:{task_train_augments}"),
                )
                train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=CFG.max_seq_length)
                train_rows = Dataset.from_list(train_ds.as_list(formatter))
                with io.StringIO() as buffer, redirect_stdout(buffer), redirect_stderr(buffer):
                    active_args = dict(train_args)
                    try:
                        trainer = UnslothFixedTrainer(
                            model=model,
                            tokenizer=tokenizer,
                            data_collator=collator,
                            train_dataset=train_rows,
                            dataset_text_field="text",
                            max_seq_length=CFG.max_seq_length,
                            args=UnslothTrainingArguments(**active_args),
                        )
                        stats = trainer.train()
                    except Exception:
                        if active_args.get("optim") == "adamw_torch":
                            raise
                        # Some Kaggle images expose bitsandbytes but not a working
                        # optimizer kernel. Reset the adapter and retry the proven
                        # torch AdamW path rather than losing the task.
                        try:
                            del trainer
                        except Exception:
                            pass
                        set_peft_model_state_dict(model, default_weights, adapter_name="default")
                        active_args["optim"] = "adamw_torch"
                        trainer = UnslothFixedTrainer(
                            model=model,
                            tokenizer=tokenizer,
                            data_collator=collator,
                            train_dataset=train_rows,
                            dataset_text_field="text",
                            max_seq_length=CFG.max_seq_length,
                            args=UnslothTrainingArguments(**active_args),
                        )
                        stats = trainer.train()
                    train_runtime = float(stats.metrics.get("train_runtime", 0.0))
                    train_loss = float(stats.metrics.get("train_loss", float("nan")))
                    model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
                    del trainer
                max_ttt_loss = float(os.getenv("ARC_MAX_ACCEPTED_TTT_LOSS", "5.0"))
                if (not math.isfinite(train_loss)) or train_loss > max_ttt_loss:
                    set_peft_model_state_dict(model, default_weights, adapter_name="default")
                    ttt_reverted = True
                    print(f"[Rank {rank}] {key} reverted pathological TTT loss={train_loss}")
                model = FastLanguageModel.for_inference(model)
            gc.collect()
            torch.cuda.empty_cache()
            print(
                f"[Rank {rank}] {key} budget={puzzle_budget:.0f}s route={route.get('action', 'full_default')} "
                f"augments={task_train_augments} skip_ttt={skip_ttt} "
                f"train={train_runtime:.1f}s loss={train_loss:.6f} reverted={ttt_reverted}"
            )

            puzzle_ds_multi = puzzle_ds.split_multi_replies()
            eval_ds = puzzle_ds_multi.augment(n=CFG.tta_color_permutations, seed=stable_seed(f"eval:{key}"))
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=CFG.max_seq_length - max_new_tokens)
            original_task = puzzle_ds.queries[key]
            known_scores: dict[tuple[str, tuple[tuple[int, ...], ...]], tuple[list[float], list[float], int]] = {}

            # Split geometry groups again by actual prefix length as a defensive guard.
            batches: list[tuple[list[str], list[list[int]]]] = []
            for subkeys in _group_eval_subkeys(eval_ds):
                buckets: dict[int, list[tuple[str, list[int]]]] = defaultdict(list)
                for subkey in subkeys:
                    tokens = list(map(int, tokenizer.encode(eval_ds.get(subkey, formatter)["input"])))
                    buckets[len(tokens)].append((subkey, tokens))
                for values in buckets.values():
                    batches.append(([item[0] for item in values], [item[1] for item in values]))

            with torch.inference_mode():
                for batch_index, (subkeys, prefix_tokens) in enumerate(batches):
                    hard_decode_deadline = puzzle_deadline - CFG.reserve_scoring_seconds
                    if time.time() >= hard_decode_deadline:
                        print(f"[Rank {rank}] {key} decoding deadline")
                        break
                    batches_left = max(1, len(batches) - batch_index)
                    fair_batch_seconds = max(18.0, (hard_decode_deadline - time.time()) / batches_left)
                    batch_deadline = min(hard_decode_deadline, time.time() + fair_batch_seconds)
                    batch_score_deadline = min(
                        puzzle_deadline - 5.0,
                        batch_deadline + CFG.reserve_scoring_seconds / max(1, len(batches)),
                    )
                    base_key = subkeys[0].split(".")[0]
                    test_idx = int(base_key.rsplit("_", 1)[1])
                    shape_plan, needs_free = _shape_plan(original_task, test_idx, symbolic_hints.get(base_key, []))
                    beam_records: dict[int, list[tuple[float, list[int], tuple[int, int] | None, float, str, str]]] = defaultdict(list)

                    for shape_index, (h, w, shape_score, shape_reason) in enumerate(shape_plan):
                        remaining = batch_deadline - time.time()
                        if remaining <= 5:
                            break
                        transformed_shape = ArcDataset.transformed_shape((h, w), subkeys[0])
                        schedule = token_spec.schedule(transformed_shape)
                        calls_left = max(1, len(shape_plan) - shape_index + (1 if needs_free else 0))
                        call_deadline = min(batch_deadline, time.time() + max(8.0, remaining / calls_left))
                        results = inference_constrained(
                            model=model,
                            prefix_tokens=prefix_tokens,
                            schedule=schedule,
                            max_score=CFG.constrained_max_score,
                            search_deadline=call_deadline,
                            pad_id=token_spec.pad_id,
                            branch_cap=CFG.constrained_branch_cap,
                            beam_cap=CFG.beam_cap,
                        )
                        for subkey_id, beams in results:
                            for beam_score, out_tokens in beams:
                                beam_records[subkey_id].append(
                                    (beam_score, out_tokens, transformed_shape, shape_score, shape_reason, "shape-grammar")
                                )
                        # Confident shape is calibrated as exact on all supplied public
                        # evaluation outputs; do not waste time on redundant shapes.
                        if shape_score >= 5.0 and sum(bool(beam_records[i]) for i in range(len(subkeys))) >= max(1, len(subkeys) // 2):
                            break

                    if needs_free or not any(beam_records.values()):
                        remaining = batch_deadline - time.time()
                        if remaining > 8:
                            call_deadline = batch_deadline
                            results = inference_unconstrained(
                                model=model,
                                prefix_tokens=prefix_tokens,
                                max_new_tokens=max_new_tokens,
                                max_score=CFG.unconstrained_max_score,
                                search_deadline=call_deadline,
                                token_spec=token_spec,
                                branch_cap=CFG.unconstrained_branch_cap,
                                beam_cap=CFG.beam_cap,
                            )
                            for subkey_id, beams in results:
                                for beam_score, out_tokens in beams:
                                    beam_records[subkey_id].append((beam_score, out_tokens, None, 0.0, "free", "rect-grammar"))

                    for subkey_id, records in beam_records.items():
                        subkey = subkeys[subkey_id]
                        base_key = subkey.split(".")[0]
                        test_idx = int(base_key.rsplit("_", 1)[1])
                        test_input = as_grid(original_task["test"][test_idx]["input"])
                        by_grid: dict[tuple[tuple[int, ...], ...], tuple[Any, ...]] = {}
                        for record in sorted(records, key=lambda item: item[0]):
                            beam_score, out_tokens, expected_shape, shape_score, shape_reason, decoder_name = record
                            array = formatter.convert_tokens_to_array(out_tokens, expected_shape=expected_shape)
                            if array is None:
                                continue
                            solution = np.asarray(puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True), dtype=np.int8)
                            hkey = grid_key(solution)
                            previous = by_grid.get(hkey)
                            if previous is None or beam_score < previous[0]:
                                by_grid[hkey] = (beam_score, out_tokens, solution, shape_score, shape_reason, decoder_name)

                        decoded: list[dict[str, Any]] = []
                        for beam_score, out_tokens, solution, shape_score, shape_reason, decoder_name in sorted(by_grid.values(), key=lambda item: item[0])[: CFG.max_candidates_per_view]:
                            cache_key = (base_key, grid_key(solution))
                            if cache_key in known_scores:
                                totals, norms, answer_len = known_scores[cache_key]
                            else:
                                totals, norms, lengths = [], [], []
                                if time.time() < batch_score_deadline:
                                    score_dataset = ArcDataset(
                                        keys=[base_key],
                                        queries={base_key: puzzle_ds_multi.queries[base_key]},
                                        replies={base_key: [solution.tolist()]},
                                    )
                                    augmented = score_dataset.augment(n=1, seed=stable_seed(f"score:{base_key}:{grid_key(solution)}"))
                                    augmented = augmented.cut_to_len(
                                        formatter=formatter,
                                        name="input",
                                        max_len=CFG.max_seq_length - max_new_tokens,
                                    )
                                    samples = [score_dataset.get(base_key, formatter)] + augmented.as_list(formatter)
                                    if batch_score_deadline - time.time() < 18:
                                        samples = samples[:1]
                                    for start in range(0, len(samples), 4):
                                        if time.time() >= batch_score_deadline:
                                            break
                                        chunk = samples[start : start + 4]
                                        chunk_totals, chunk_norms, chunk_lengths = calc_scores(
                                            [sample["input"] for sample in chunk],
                                            [sample["reply"] for sample in chunk],
                                            tokenizer,
                                            model,
                                            token_spec.pad_id,
                                        )
                                        totals.extend(chunk_totals)
                                        norms.extend(chunk_norms)
                                        lengths.extend(chunk_lengths)
                                answer_len = lengths[0] if lengths else int(solution.size + solution.shape[0])
                                known_scores[cache_key] = (totals, norms, answer_len)

                            decoded.append(
                                {
                                    "beam_score": float(beam_score),
                                    "beam_nll": float(beam_score / max(1, len(out_tokens))),
                                    "beam_len": len(out_tokens),
                                    "score_aug": totals,
                                    "score_aug_norm": norms,
                                    "answer_len": answer_len,
                                    "solution": solution,
                                    "source": "llm",
                                    "decoder": decoder_name,
                                    "shape_score": float(shape_score),
                                    "shape_reason": shape_reason,
                                    "structural_score": structural_consistency(original_task, test_input, solution),
                                    "rl_route": route.get("action", "full_default"),
                                    "ttt_augments": int(task_train_augments),
                                    "ttt_skipped": bool(skip_ttt),
                                    "ttt_reverted": bool(ttt_reverted),
                                    "route_action": str(route.get("action", "full_default")),
                                    "puzzle_budget_seconds": float(puzzle_budget),
                                }
                            )
                        if decoded:
                            from arc_exec_rl.common import atomic_candidate_pickle
                            atomic_candidate_pickle(os.path.join(CFG.output_dir, subkey), decoded)

            try:
                memory = torch.cuda.max_memory_allocated() // 1024**2
            except Exception:
                memory = -1
            print(f"[Rank {rank}] finished {key} in {time.time() - start_time:.1f}s peak={memory}MB")
        except Exception as exc:
            print(f"[Rank {rank}] ERROR on {key}: {type(exc).__name__}: {exc}")
            traceback.print_exc()
            gc.collect()
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            continue


In [ ]:
%%writefile arc_symbolic.py
from __future__ import annotations

import math
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from typing import Any, Callable, Iterable

import numpy as np

Grid = np.ndarray


def as_grid(value: Any) -> Grid:
    arr = np.asarray(value, dtype=np.int8)
    if arr.ndim != 2:
        raise ValueError(f"ARC grid must be 2-D, got shape {arr.shape}")
    return arr


def grid_key(grid: Any) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(map(int, row)) for row in np.asarray(grid))


def valid_grid(grid: Any) -> bool:
    a = np.asarray(grid)
    return a.ndim == 2 and 1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30 and np.all((0 <= a) & (a <= 9))


def mode_color(grid: Grid) -> int:
    values, counts = np.unique(grid, return_counts=True)
    return int(values[int(np.argmax(counts))])


def bbox_of_mask(mask: np.ndarray) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask)
    if len(ys) == 0:
        return None
    return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1


def crop_nonbackground(grid: Grid, bg: int) -> Grid | None:
    box = bbox_of_mask(grid != bg)
    if box is None:
        return None
    y0, y1, x0, x1 = box
    return np.array(grid[y0:y1, x0:x1], copy=True)


D4_NAMES = ("id", "r1", "r2", "r3", "t", "tr1", "tr2", "tr3")


def d4(grid: Grid, name: str) -> Grid:
    if name == "id":
        return np.array(grid, copy=True)
    if name == "r1":
        return np.rot90(grid, 1).copy()
    if name == "r2":
        return np.rot90(grid, 2).copy()
    if name == "r3":
        return np.rot90(grid, 3).copy()
    if name == "t":
        return grid.T.copy()
    if name == "tr1":
        return np.rot90(grid.T, 1).copy()
    if name == "tr2":
        return np.rot90(grid.T, 2).copy()
    if name == "tr3":
        return np.rot90(grid.T, 3).copy()
    raise KeyError(name)


@dataclass(frozen=True)
class Component:
    cells: tuple[tuple[int, int], ...]
    color: int | None
    y0: int
    y1: int
    x0: int
    x1: int

    @property
    def area(self) -> int:
        return len(self.cells)

    @property
    def height(self) -> int:
        return self.y1 - self.y0

    @property
    def width(self) -> int:
        return self.x1 - self.x0


def components(grid: Grid, bg: int, diagonal: bool = False, same_color: bool = False) -> list[Component]:
    h, w = grid.shape
    seen = np.zeros((h, w), dtype=bool)
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    if diagonal:
        dirs += [(-1, -1), (-1, 1), (1, -1), (1, 1)]
    out: list[Component] = []
    for sy in range(h):
        for sx in range(w):
            if seen[sy, sx] or int(grid[sy, sx]) == bg:
                continue
            seed_color = int(grid[sy, sx])
            q = deque([(sy, sx)])
            seen[sy, sx] = True
            cells: list[tuple[int, int]] = []
            while q:
                y, x = q.popleft()
                cells.append((y, x))
                for dy, dx in dirs:
                    yy, xx = y + dy, x + dx
                    if not (0 <= yy < h and 0 <= xx < w) or seen[yy, xx] or int(grid[yy, xx]) == bg:
                        continue
                    if same_color and int(grid[yy, xx]) != seed_color:
                        continue
                    seen[yy, xx] = True
                    q.append((yy, xx))
            ys = [p[0] for p in cells]
            xs = [p[1] for p in cells]
            colors = {int(grid[y, x]) for y, x in cells}
            out.append(
                Component(
                    cells=tuple(cells),
                    color=next(iter(colors)) if len(colors) == 1 else None,
                    y0=min(ys),
                    y1=max(ys) + 1,
                    x0=min(xs),
                    x1=max(xs) + 1,
                )
            )
    return out


def select_component(items: list[Component], selector: str) -> Component | None:
    if not items:
        return None
    if selector == "largest":
        best = max(c.area for c in items)
        winners = [c for c in items if c.area == best]
    elif selector == "smallest":
        best = min(c.area for c in items)
        winners = [c for c in items if c.area == best]
    elif selector == "tallest":
        best = max(c.height for c in items)
        winners = [c for c in items if c.height == best]
    elif selector == "widest":
        best = max(c.width for c in items)
        winners = [c for c in items if c.width == best]
    elif selector == "unique_color":
        counts = Counter(c.color for c in items if c.color is not None)
        winners = [c for c in items if c.color is not None and counts[c.color] == 1]
    else:
        raise KeyError(selector)
    if len(winners) != 1:
        return None
    return winners[0]


def component_patch(
    grid: Grid,
    bg: int,
    diagonal: bool,
    same_color: bool,
    selector: str,
    mask_only: bool,
) -> Grid | None:
    comp = select_component(components(grid, bg, diagonal, same_color), selector)
    if comp is None:
        return None
    patch = np.array(grid[comp.y0 : comp.y1, comp.x0 : comp.x1], copy=True)
    if mask_only:
        keep = {(y - comp.y0, x - comp.x0) for y, x in comp.cells}
        for y in range(patch.shape[0]):
            for x in range(patch.shape[1]):
                if (y, x) not in keep:
                    patch[y, x] = bg
    return patch


def color_role(grid: Grid, role: str) -> int | None:
    counts = Counter(map(int, grid.ravel()))
    if role == "zero":
        return 0
    if role == "mode":
        return counts.most_common(1)[0][0]
    nonzero = [(count, color) for color, count in counts.items() if color != 0]
    if not nonzero:
        return None
    if role == "rarest_nonzero":
        value = min(count for count, _ in nonzero)
        winners = [color for count, color in nonzero if count == value]
    elif role == "common_nonzero":
        value = max(count for count, _ in nonzero)
        winners = [color for count, color in nonzero if count == value]
    else:
        raise KeyError(role)
    return winners[0] if len(winners) == 1 else None


def crop_color_role(grid: Grid, role: str, keep_other: bool) -> Grid | None:
    color = color_role(grid, role)
    if color is None:
        return None
    box = bbox_of_mask(grid == color)
    if box is None:
        return None
    y0, y1, x0, x1 = box
    patch = np.array(grid[y0:y1, x0:x1], copy=True)
    if not keep_other:
        bg = mode_color(grid)
        patch[patch != color] = bg
    return patch


def compress_nonbackground(grid: Grid, bg: int) -> Grid | None:
    row_mask = np.any(grid != bg, axis=1)
    col_mask = np.any(grid != bg, axis=0)
    if not row_mask.any() or not col_mask.any():
        return None
    return np.array(grid[np.ix_(row_mask, col_mask)], copy=True)


def trim_uniform_border(grid: Grid) -> Grid:
    out = np.array(grid, copy=True)
    changed = True
    while changed and out.shape[0] > 1 and out.shape[1] > 1:
        changed = False
        if np.all(out[0] == out[0, 0]) and np.all(out[0] == out[-1, 0]) and np.all(out[0] == out[:, 0]) and np.all(out[0] == out[:, -1]):
            if np.all(out[-1] == out[0, 0]) and np.all(out[:, -1] == out[0, 0]):
                out = out[1:-1, 1:-1]
                changed = True
                continue
        if np.all(out[0] == out[0, 0]) and np.all(out[0] == mode_color(out)):
            out = out[1:]
            changed = True
        if out.shape[0] > 1 and np.all(out[-1] == out[-1, 0]) and np.all(out[-1] == mode_color(out)):
            out = out[:-1]
            changed = True
        if out.shape[1] > 1 and np.all(out[:, 0] == out[0, 0]) and np.all(out[:, 0] == mode_color(out)):
            out = out[:, 1:]
            changed = True
        if out.shape[1] > 1 and np.all(out[:, -1] == out[0, -1]) and np.all(out[:, -1] == mode_color(out)):
            out = out[:, :-1]
            changed = True
    return out


def fill_holes(grid: Grid, bg: int, fill: int | None = None) -> Grid:
    h, w = grid.shape
    outside = np.zeros((h, w), dtype=bool)
    q: deque[tuple[int, int]] = deque()
    for y in range(h):
        for x in (0, w - 1):
            if int(grid[y, x]) == bg and not outside[y, x]:
                outside[y, x] = True
                q.append((y, x))
    for x in range(w):
        for y in (0, h - 1):
            if int(grid[y, x]) == bg and not outside[y, x]:
                outside[y, x] = True
                q.append((y, x))
    while q:
        y, x = q.popleft()
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            yy, xx = y + dy, x + dx
            if 0 <= yy < h and 0 <= xx < w and int(grid[yy, xx]) == bg and not outside[yy, xx]:
                outside[yy, xx] = True
                q.append((yy, xx))
    holes = (grid == bg) & ~outside
    out = np.array(grid, copy=True)
    if fill is None:
        neighbors: list[int] = []
        for y, x in zip(*np.where(holes)):
            for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                yy, xx = y + dy, x + dx
                if 0 <= yy < h and 0 <= xx < w and int(grid[yy, xx]) != bg:
                    neighbors.append(int(grid[yy, xx]))
        fill = Counter(neighbors).most_common(1)[0][0] if neighbors else bg
    out[holes] = int(fill)
    return out


def block_reduce(grid: Grid, fy: int, fx: int, mode: str, bg: int) -> Grid | None:
    h, w = grid.shape
    if h % fy or w % fx:
        return None
    blocks = grid.reshape(h // fy, fy, w // fx, fx).transpose(0, 2, 1, 3)
    out = np.empty((h // fy, w // fx), dtype=np.int8)
    for y in range(out.shape[0]):
        for x in range(out.shape[1]):
            vals = list(map(int, blocks[y, x].ravel()))
            if mode == "tl":
                out[y, x] = vals[0]
            elif mode == "majority":
                out[y, x] = Counter(vals).most_common(1)[0][0]
            elif mode == "uniform":
                if len(set(vals)) != 1:
                    return None
                out[y, x] = vals[0]
            elif mode == "nonbg":
                present = [v for v in vals if v != bg]
                if not present:
                    out[y, x] = bg
                elif len(set(present)) == 1:
                    out[y, x] = present[0]
                else:
                    return None
            else:
                raise KeyError(mode)
    return out


def _separator_split(grid: Grid, axis: int) -> tuple[Grid, Grid] | None:
    n = grid.shape[axis]
    uniform: list[tuple[int, int]] = []
    for i in range(n):
        line = grid[i, :] if axis == 0 else grid[:, i]
        if np.all(line == line.flat[0]):
            uniform.append((i, int(line.flat[0])))
    for start, color in uniform:
        end = start + 1
        while end < n:
            line = grid[end, :] if axis == 0 else grid[:, end]
            if not (np.all(line == color)):
                break
            end += 1
        if start == n - end and start > 0:
            if axis == 0:
                return np.array(grid[:start], copy=True), np.array(grid[end:], copy=True)
            return np.array(grid[:, :start], copy=True), np.array(grid[:, end:], copy=True)
    return None


def split_two_panels(grid: Grid, mode: str) -> tuple[Grid, Grid] | None:
    h, w = grid.shape
    if mode == "h_sep":
        return _separator_split(grid, 0)
    if mode == "v_sep":
        return _separator_split(grid, 1)
    if mode == "h_half" and h % 2 == 0:
        return np.array(grid[: h // 2], copy=True), np.array(grid[h // 2 :], copy=True)
    if mode == "v_half" and w % 2 == 0:
        return np.array(grid[:, : w // 2], copy=True), np.array(grid[:, w // 2 :], copy=True)
    return None


def combine_panels(a: Grid, b: Grid, op: str, bg: int) -> Grid | None:
    if a.shape != b.shape:
        return None
    ma, mb = a != bg, b != bg
    if op == "a":
        return a.copy()
    if op == "b":
        return b.copy()
    if op == "overlay_ab":
        return np.where(ma, a, b).astype(np.int8)
    if op == "overlay_ba":
        return np.where(mb, b, a).astype(np.int8)
    if op == "and_a":
        return np.where(ma & mb, a, bg).astype(np.int8)
    if op == "and_b":
        return np.where(ma & mb, b, bg).astype(np.int8)
    if op == "xor_values":
        return np.where(ma ^ mb, np.where(ma, a, b), bg).astype(np.int8)
    if op == "or_mask":
        return (ma | mb).astype(np.int8)
    if op == "and_mask":
        return (ma & mb).astype(np.int8)
    if op == "xor_mask":
        return (ma ^ mb).astype(np.int8)
    if op == "eq_mask":
        return (a == b).astype(np.int8)
    if op == "neq_mask":
        return (a != b).astype(np.int8)
    raise KeyError(op)


def infer_color_map(srcs: list[Grid], dsts: list[Grid], allow_many_to_one: bool = True) -> dict[int, int] | None:
    mapping: dict[int, int] = {}
    reverse: dict[int, int] = {}
    for src, dst in zip(srcs, dsts):
        if src.shape != dst.shape:
            return None
        for x, y in zip(map(int, src.ravel()), map(int, dst.ravel())):
            if x in mapping and mapping[x] != y:
                return None
            mapping[x] = y
            if not allow_many_to_one:
                if y in reverse and reverse[y] != x:
                    return None
                reverse[y] = x
    return mapping


def apply_color_map(grid: Grid, mapping: dict[int, int]) -> Grid:
    out = np.array(grid, copy=True)
    for src, dst in mapping.items():
        out[grid == src] = dst
    return out


@dataclass(frozen=True)
class Program:
    name: str
    cost: float
    family: str
    fn: Callable[[Grid], Grid | None]


@dataclass
class ProgramCandidate:
    grid: Grid
    score: float
    confidence: float
    support: int
    min_cost: float
    programs: list[str]
    families: list[str]


def _safe(program_fn: Callable[[Grid], Grid | None], grid: Grid) -> Grid | None:
    try:
        out = program_fn(grid)
        if out is None:
            return None
        out = np.asarray(out, dtype=np.int8)
        return out if valid_grid(out) else None
    except Exception:
        return None


def enumerate_programs() -> list[Program]:
    programs: list[Program] = []

    def add(name: str, cost: float, family: str, fn: Callable[[Grid], Grid | None]) -> None:
        programs.append(Program(name, cost, family, fn))

    for transform in D4_NAMES:
        base_cost = 0.0 if transform == "id" else 0.35
        add(f"d4:{transform}", base_cost, "d4", lambda g, t=transform: d4(g, t))
        for bg_role in ("zero", "mode"):
            add(
                f"crop:{transform}:{bg_role}",
                base_cost + 0.75,
                "crop",
                lambda g, t=transform, r=bg_role: crop_nonbackground(d4(g, t), 0 if r == "zero" else mode_color(d4(g, t))),
            )
            add(
                f"compress:{transform}:{bg_role}",
                base_cost + 1.0,
                "compress",
                lambda g, t=transform, r=bg_role: compress_nonbackground(d4(g, t), 0 if r == "zero" else mode_color(d4(g, t))),
            )
            add(
                f"fillholes:{transform}:{bg_role}",
                base_cost + 1.15,
                "fill",
                lambda g, t=transform, r=bg_role: fill_holes(d4(g, t), 0 if r == "zero" else mode_color(d4(g, t))),
            )
        add(f"trim:{transform}", base_cost + 0.9, "trim", lambda g, t=transform: trim_uniform_border(d4(g, t)))

        for role in ("rarest_nonzero", "common_nonzero"):
            for keep_other in (False, True):
                add(
                    f"colorcrop:{transform}:{role}:{int(keep_other)}",
                    base_cost + 1.15 + 0.15 * keep_other,
                    "colorcrop",
                    lambda g, t=transform, r=role, k=keep_other: crop_color_role(d4(g, t), r, k),
                )

        for bg_role in ("zero", "mode"):
            for diagonal in (False, True):
                for same_color in (False, True):
                    for selector in ("largest", "smallest", "tallest", "widest", "unique_color"):
                        for mask_only in (False, True):
                            add(
                                f"component:{transform}:{bg_role}:{int(diagonal)}:{int(same_color)}:{selector}:{int(mask_only)}",
                                base_cost + 1.15 + 0.15 * diagonal + 0.15 * same_color + 0.2 * mask_only,
                                "component",
                                lambda g, t=transform, br=bg_role, di=diagonal, sc=same_color, se=selector, mo=mask_only: component_patch(
                                    d4(g, t), 0 if br == "zero" else mode_color(d4(g, t)), di, sc, se, mo
                                ),
                            )

        for fy, fx in ((2, 2), (3, 3), (4, 4), (2, 1), (1, 2), (3, 1), (1, 3)):
            add(
                f"scale:{transform}:{fy}x{fx}",
                base_cost + 0.95,
                "scale",
                lambda g, t=transform, y=fy, x=fx: np.repeat(np.repeat(d4(g, t), y, axis=0), x, axis=1),
            )
        for fy, fx in ((2, 2), (3, 3), (4, 4), (2, 1), (1, 2), (3, 1), (1, 3)):
            for reduce_mode in ("uniform", "nonbg", "majority", "tl"):
                for bg_role in ("zero", "mode"):
                    add(
                        f"reduce:{transform}:{fy}x{fx}:{reduce_mode}:{bg_role}",
                        base_cost + 1.15 + (0.25 if reduce_mode in ("majority", "tl") else 0.0),
                        "reduce",
                        lambda g, t=transform, y=fy, x=fx, m=reduce_mode, br=bg_role: block_reduce(
                            d4(g, t), y, x, m, 0 if br == "zero" else mode_color(d4(g, t))
                        ),
                    )

        # Common self-compositions.
        add(f"concat_h:{transform}", base_cost + 1.2, "concat", lambda g, t=transform: np.concatenate([d4(g, t), d4(g, t)], axis=1))
        add(f"concat_v:{transform}", base_cost + 1.2, "concat", lambda g, t=transform: np.concatenate([d4(g, t), d4(g, t)], axis=0))
        add(f"mirror_h:{transform}", base_cost + 1.25, "concat", lambda g, t=transform: np.concatenate([d4(g, t), np.fliplr(d4(g, t))], axis=1))
        add(f"mirror_v:{transform}", base_cost + 1.25, "concat", lambda g, t=transform: np.concatenate([d4(g, t), np.flipud(d4(g, t))], axis=0))

    for split_mode in ("h_sep", "v_sep", "h_half", "v_half"):
        for b_transform in D4_NAMES:
            for op in ("a", "b", "overlay_ab", "overlay_ba", "and_a", "and_b", "xor_values", "or_mask", "and_mask", "xor_mask", "eq_mask", "neq_mask"):
                for bg_role in ("zero", "mode"):
                    def panel_fn(g: Grid, sm=split_mode, bt=b_transform, operation=op, br=bg_role) -> Grid | None:
                        pair = split_two_panels(g, sm)
                        if pair is None:
                            return None
                        a, b = pair
                        b = d4(b, bt)
                        bg = 0 if br == "zero" else mode_color(g)
                        return combine_panels(a, b, operation, bg)

                    add(
                        f"panel:{split_mode}:{b_transform}:{op}:{bg_role}",
                        1.25 + (0.0 if b_transform == "id" else 0.3),
                        "panel",
                        panel_fn,
                    )
    family_priority = {"d4": 0, "crop": 1, "trim": 2, "compress": 3, "colorcrop": 4, "panel": 5, "scale": 6, "reduce": 7, "component": 8, "fill": 9, "concat": 10}
    programs.sort(key=lambda p: (p.cost, family_priority.get(p.family, 99), p.name))
    return programs


_PROGRAMS: list[Program] | None = None


def get_programs() -> list[Program]:
    global _PROGRAMS
    if _PROGRAMS is None:
        _PROGRAMS = enumerate_programs()
    return _PROGRAMS


class SymbolicSolver:
    """Small exact program synthesizer used as a high-precision lane and shape hint."""

    def __init__(self, max_programs: int = 6000, max_outputs: int = 12):
        self.max_programs = max_programs
        self.max_outputs = max_outputs

    def synthesize(self, task: dict[str, Any], test_idx: int = 0) -> list[ProgramCandidate]:
        train_inputs = [as_grid(ex["input"]) for ex in task["train"]]
        train_outputs = [as_grid(ex["output"]) for ex in task["train"]]
        test_input = as_grid(task["test"][test_idx]["input"])
        grouped: dict[tuple[tuple[int, ...], ...], dict[str, Any]] = {}

        for program in get_programs()[: self.max_programs]:
            generated = [_safe(program.fn, grid) for grid in train_inputs]
            if any(value is None for value in generated):
                continue
            srcs = [value for value in generated if value is not None]
            if any(src.shape != dst.shape for src, dst in zip(srcs, train_outputs)):
                continue
            mapping: dict[int, int] | None
            if all(np.array_equal(src, dst) for src, dst in zip(srcs, train_outputs)):
                mapping = {}
            else:
                mapping = infer_color_map(srcs, train_outputs, allow_many_to_one=True)
                if mapping is None or not all(np.array_equal(apply_color_map(src, mapping), dst) for src, dst in zip(srcs, train_outputs)):
                    continue
            test_grid = _safe(program.fn, test_input)
            if test_grid is None:
                continue
            if mapping:
                test_grid = apply_color_map(test_grid, mapping)
            if not valid_grid(test_grid):
                continue
            key = grid_key(test_grid)
            rec = grouped.setdefault(
                key,
                {"grid": test_grid, "programs": [], "families": set(), "costs": []},
            )
            suffix = "" if not mapping else ":map=" + ",".join(f"{a}>{b}" for a, b in sorted(mapping.items()))
            rec["programs"].append(program.name + suffix)
            rec["families"].add(program.family)
            rec["costs"].append(program.cost + (0.35 if mapping else 0.0))

        candidates: list[ProgramCandidate] = []
        for rec in grouped.values():
            support = len(rec["programs"])
            family_count = len(rec["families"])
            min_cost = float(min(rec["costs"]))
            score = 4.0 - min_cost + 0.5 * math.log1p(support) + 0.35 * math.log1p(family_count)
            simple = min_cost <= 0.45 and any(name.startswith("d4:") for name in rec["programs"])
            consensus = family_count >= 2 and support >= 3 and min_cost <= 1.6
            confidence = 0.995 if simple else 0.975 if consensus else min(0.94, 0.60 + 0.06 * support + 0.05 * family_count - 0.05 * min_cost)
            candidates.append(
                ProgramCandidate(
                    grid=np.asarray(rec["grid"], dtype=np.int8),
                    score=float(score),
                    confidence=float(confidence),
                    support=support,
                    min_cost=min_cost,
                    programs=sorted(rec["programs"], key=len)[:16],
                    families=sorted(rec["families"]),
                )
            )
        candidates.sort(key=lambda c: (c.score, c.confidence, -c.min_cost), reverse=True)
        return candidates[: self.max_outputs]


# ------------------------- Output-shape inference -------------------------

def _shape_features(grid: Grid) -> dict[str, int]:
    h, w = grid.shape
    out: dict[str, int] = {
        "h": h,
        "w": w,
        "area": h * w,
        "ncolors": len(np.unique(grid)),
        "ncolors_nonzero": len([v for v in np.unique(grid) if int(v) != 0]),
    }
    for bg_name, bg in (("zero", 0), ("mode", mode_color(grid))):
        mask = grid != bg
        box = bbox_of_mask(mask)
        out[f"rows:{bg_name}"] = int(np.any(mask, axis=1).sum())
        out[f"cols:{bg_name}"] = int(np.any(mask, axis=0).sum())
        if box is not None:
            out[f"bbox_h:{bg_name}"] = box[1] - box[0]
            out[f"bbox_w:{bg_name}"] = box[3] - box[2]
        for diagonal in (False, True):
            for same_color in (False, True):
                comps = components(grid, bg, diagonal, same_color)
                tag = f"{bg_name}:{int(diagonal)}:{int(same_color)}"
                out[f"ncomp:{tag}"] = len(comps)
                if comps:
                    out[f"max_area:{tag}"] = max(c.area for c in comps)
                    out[f"min_area:{tag}"] = min(c.area for c in comps)
                    out[f"max_h:{tag}"] = max(c.height for c in comps)
                    out[f"max_w:{tag}"] = max(c.width for c in comps)
                    out[f"min_h:{tag}"] = min(c.height for c in comps)
                    out[f"min_w:{tag}"] = min(c.width for c in comps)
                    for selector in ("largest", "smallest", "tallest", "widest", "unique_color"):
                        comp = select_component(comps, selector)
                        if comp is not None:
                            out[f"comp_h:{tag}:{selector}"] = comp.height
                            out[f"comp_w:{tag}:{selector}"] = comp.width
                            out[f"comp_area:{tag}:{selector}"] = comp.area
    for color in range(10):
        mask = grid == color
        out[f"count:c{color}"] = int(mask.sum())
        out[f"rows:c{color}"] = int(np.any(mask, axis=1).sum())
        out[f"cols:c{color}"] = int(np.any(mask, axis=0).sum())
        box = bbox_of_mask(mask)
        if box is not None:
            out[f"bbox_h:c{color}"] = box[1] - box[0]
            out[f"bbox_w:c{color}"] = box[3] - box[2]
    for mode in ("h_sep", "v_sep", "h_half", "v_half"):
        pair = split_two_panels(grid, mode)
        if pair is not None:
            out[f"panel_h:{mode}"] = pair[0].shape[0]
            out[f"panel_w:{mode}"] = pair[0].shape[1]
    return {name: int(value) for name, value in out.items() if 0 <= int(value) <= 900}


def _expanded_dim_features(grid: Grid) -> dict[str, tuple[int, float]]:
    base = _shape_features(grid)
    out: dict[str, tuple[int, float]] = {}
    for name, value in base.items():
        if name in ("h", "w"):
            base_cost = 0.15
        elif name.startswith(("bbox_", "rows:", "cols:", "panel_")):
            base_cost = 0.45
        else:
            base_cost = 0.75
        out[name] = (value, base_cost)
        for delta in (-3, -2, -1, 1, 2, 3):
            candidate = value + delta
            if 1 <= candidate <= 30:
                out[f"{name}{delta:+d}"] = (candidate, base_cost + 0.55 + 0.08 * abs(delta))
        for factor in (2, 3, 4):
            candidate = value * factor
            if 1 <= candidate <= 30:
                out[f"{name}*{factor}"] = (candidate, base_cost + 0.5 + 0.12 * factor)
            if value % factor == 0 and 1 <= value // factor <= 30:
                out[f"{name}/{factor}"] = (value // factor, base_cost + 0.65 + 0.1 * factor)
    for constant in range(1, 31):
        out[f"const:{constant}"] = (constant, 1.25)
    return out


def infer_output_shapes(task: dict[str, Any], test_grid: Grid, max_shapes: int = 12) -> list[tuple[int, int, float, str]]:
    """Rank output dimensions fitted exactly on demonstrations.

    High confidence is deliberately withheld when a rule must extrapolate from a
    dimension/feature that never varied in demonstrations. This prevents a
    constant-shape shortcut from blocking the unconstrained decoder.
    """
    train_inputs = [as_grid(ex["input"]) for ex in task["train"]]
    train_outputs = [as_grid(ex["output"]) for ex in task["train"]]
    pairs = [(x.shape, y.shape) for x, y in zip(train_inputs, train_outputs)]
    th, tw = test_grid.shape
    scored: dict[tuple[int, int], tuple[float, str]] = {}

    def add(shape: tuple[int, int], score: float, reason: str) -> None:
        h, w = map(int, shape)
        if 1 <= h <= 30 and 1 <= w <= 30:
            old = scored.get((h, w))
            if old is None or score > old[0]:
                scored[(h, w)] = (float(score), reason)

    in_heights = {shape[0] for shape, _ in pairs}
    in_widths = {shape[1] for shape, _ in pairs}
    direct_safe = (th in in_heights or len(in_heights) >= 2) and (tw in in_widths or len(in_widths) >= 2)

    if len({out_shape for _, out_shape in pairs}) == 1:
        add(pairs[0][1], 5.0 if direct_safe else 2.65, "constant" if direct_safe else "constant-ambiguous")
    if all(inp == out for inp, out in pairs):
        add((th, tw), 6.0, "same")
    if all((inp[1], inp[0]) == out for inp, out in pairs):
        add((tw, th), 5.8, "transpose")

    deltas = {(out[0] - inp[0], out[1] - inp[1]) for inp, out in pairs}
    if len(deltas) == 1:
        dy, dx = next(iter(deltas))
        add((th + dy, tw + dx), 3.4 if direct_safe else 2.60, "delta" if direct_safe else "delta-ambiguous")

    ratios = [(out[0] / inp[0], out[1] / inp[1]) for inp, out in pairs]
    if len(set(ratios)) == 1:
        ry, rx = ratios[0]
        hh, ww = round(th * ry), round(tw * rx)
        if abs(hh - th * ry) < 1e-9 and abs(ww - tw * rx) < 1e-9:
            add((hh, ww), 3.8 if direct_safe else 2.75, "ratio" if direct_safe else "ratio-ambiguous")

    train_features = [_expanded_dim_features(grid) for grid in train_inputs]
    test_features = _expanded_dim_features(test_grid)
    common = set(test_features)
    for features in train_features:
        common &= set(features)
    h_rules: list[tuple[float, str, int]] = []
    w_rules: list[tuple[float, str, int]] = []
    for name in common:
        values = [features[name][0] for features in train_features]
        base_cost = max(features[name][1] for features in train_features)
        test_value = test_features[name][0]
        # A feature that was constant in all examples cannot safely be treated as
        # causal when it changes at test time; keep it as a low-confidence hint.
        extrapolation_penalty = 0.85 if len(set(values)) == 1 and test_value != values[0] else 0.0
        cost = base_cost + extrapolation_penalty
        if all(value == output.shape[0] for value, output in zip(values, train_outputs)):
            h_rules.append((cost, name, test_value))
        if all(value == output.shape[1] for value, output in zip(values, train_outputs)):
            w_rules.append((cost, name, test_value))
    h_rules.sort()
    w_rules.sort()
    for h_cost, h_name, h_value in h_rules[:24]:
        for w_cost, w_name, w_value in w_rules[:24]:
            add((h_value, w_value), 4.7 - h_cost - w_cost, f"features:{h_name}|{w_name}")

    add((th, tw), 0.9, "fallback-same")
    add((tw, th), 0.7, "fallback-transpose")
    for output in train_outputs:
        add(output.shape, 0.55, "observed-output-shape")

    ordered = sorted(scored.items(), key=lambda item: item[1][0], reverse=True)
    return [(h, w, score, reason) for (h, w), (score, reason) in ordered[:max_shapes]]


# ------------------------- Candidate verification -------------------------

def _shape_relation(inp: Grid, out: Grid) -> str:
    if out.shape == inp.shape:
        return "same"
    if out.shape == inp.T.shape:
        return "transpose"
    if out.shape[0] <= inp.shape[0] and out.shape[1] <= inp.shape[1]:
        return "crop"
    if out.shape[0] >= inp.shape[0] and out.shape[1] >= inp.shape[1]:
        return "expand"
    return "mixed"


def _relation_vector(inp: Grid, out: Grid) -> tuple[str, np.ndarray]:
    ivals = set(map(int, np.unique(inp)))
    ovals = set(map(int, np.unique(out)))
    vec = np.asarray(
        [
            out.shape[0] / inp.shape[0],
            out.shape[1] / inp.shape[1],
            len(ovals) / 10.0,
            (len(ovals - ivals) - len(ivals - ovals)) / 10.0,
            float(np.mean(out != mode_color(out))),
            float(np.mean(out == np.fliplr(out))),
            float(np.mean(out == np.flipud(out))),
        ],
        dtype=float,
    )
    return _shape_relation(inp, out), vec


def structural_consistency(task: dict[str, Any], test_input: Grid, candidate: Grid) -> float:
    candidate = as_grid(candidate)
    train_relations = [_relation_vector(as_grid(ex["input"]), as_grid(ex["output"])) for ex in task["train"]]
    test_relation, test_vec = _relation_vector(test_input, candidate)
    categories = [category for category, _ in train_relations]
    category_score = categories.count(test_relation) / max(1, len(categories))
    vectors = np.stack([vector for _, vector in train_relations])
    center = np.median(vectors, axis=0)
    scale = np.maximum(np.median(np.abs(vectors - center), axis=0), 0.08)
    distance = float(np.mean(np.minimum(4.0, np.abs(test_vec - center) / scale)))
    return float(np.clip(0.55 * category_score + 0.45 * math.exp(-distance), 0.0, 1.0))


def fallback_grids(task: dict[str, Any], test_idx: int, limit: int = 2) -> list[Grid]:
    """Cheap valid attempts used only when no neural result is available."""
    test = as_grid(task["test"][test_idx]["input"])
    candidates: list[Grid] = []
    symbolic = SymbolicSolver(max_outputs=4).synthesize(task, test_idx)
    candidates.extend(c.grid for c in symbolic)
    candidates.append(test.copy())
    candidates.append(test.T.copy())
    for bg in (0, mode_color(test)):
        cropped = crop_nonbackground(test, bg)
        if cropped is not None:
            candidates.append(cropped)
    unique: list[Grid] = []
    seen: set[tuple[tuple[int, ...], ...]] = set()
    for grid in candidates:
        if valid_grid(grid) and grid_key(grid) not in seen:
            seen.add(grid_key(grid))
            unique.append(np.asarray(grid, dtype=np.int8))
        if len(unique) >= limit:
            break
    return unique


In [ ]:
%%writefile starter.py
from __future__ import annotations

import argparse
import bz2
import json
import os
import pickle
import shutil
import tempfile
import time
from typing import Any

import numpy as np
import torch
import torch.multiprocessing as mp


COMPETITION_DIR = os.getenv("ARC_COMPETITION_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
OUTPUT_DIR = os.getenv("ARC_OUTPUT_DIR", "/kaggle/inference_outputs")
DEFAULT_DEBUG_KEYS = ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]


def _strict_symbolic(candidate: Any, candidates: list[Any]) -> float:
    if candidate.confidence < 0.995 or candidate.min_cost > 0.45:
        return 0.0
    margin = candidate.score - (candidates[1].score if len(candidates) > 1 else -99.0)
    if any(name.startswith("d4:") for name in candidate.programs) and (len(candidates) == 1 or margin >= 0.65):
        return 0.999
    return 0.0


def _record(grid: np.ndarray, source: str, priority: float, confidence: float = 0.0, programs: list[str] | None = None) -> dict[str, Any]:
    return {
        "beam_score": 99.0,
        "beam_nll": 9.9,
        "beam_len": int(grid.size + grid.shape[0]),
        "score_aug": [],
        "score_aug_norm": [],
        "answer_len": int(grid.size + grid.shape[0]),
        "solution": np.asarray(grid, dtype=np.int8),
        "source": source,
        "fallback_priority": float(priority),
        "symbolic_confidence": float(confidence),
        "structural_score": 0.0,
        "shape_score": 0.0,
        "programs": programs or [],
    }


def precompute_fallbacks(data: dict[str, Any], keys: list[str], output_dir: str, retrieval_index: Any | None = None) -> set[str]:
    """Write valid pass@2 records before GPU work so timeouts never leave blanks."""
    from arc_symbolic import (
        SymbolicSolver,
        as_grid,
        crop_nonbackground,
        grid_key,
        infer_output_shapes,
        mode_color,
        valid_grid,
    )

    solver = SymbolicSolver(max_programs=500, max_outputs=8)
    start = time.time()
    written = 0
    fully_retrieved: set[str] = set()
    for key in keys:
        task = data[key]
        key_retrieved = True
        for test_idx, test in enumerate(task["test"]):
            retrieved = retrieval_index.solve(task, test_idx) if retrieval_index is not None else []
            key_retrieved = key_retrieved and bool(retrieved)
            candidates = [] if retrieved else solver.synthesize(task, test_idx)
            test_grid = as_grid(test["input"])
            records: list[dict[str, Any]] = []
            seen: set[tuple[tuple[int, ...], ...]] = set()

            for retrieved_grid, source_key in retrieved:
                h = grid_key(retrieved_grid)
                if h not in seen:
                    seen.add(h)
                    records.append(
                        _record(
                            retrieved_grid,
                            source="retrieval",
                            priority=1000.0,
                            confidence=1.0,
                            programs=[f"canonical-retrieval:{source_key}"],
                        )
                    )

            for rank, candidate in enumerate(candidates[:4]):
                h = grid_key(candidate.grid)
                if h in seen:
                    continue
                seen.add(h)
                confidence = _strict_symbolic(candidate, candidates)
                source = "symbolic" if confidence >= 0.999 else "symbolic_low"
                records.append(
                    _record(
                        candidate.grid,
                        source=source,
                        priority=80.0 - rank + candidate.score,
                        confidence=confidence,
                        programs=candidate.programs,
                    )
                )

            generic: list[tuple[np.ndarray, float]] = [(test_grid.copy(), 20.0), (test_grid.T.copy(), 18.0)]
            for bg, priority in ((0, 16.0), (mode_color(test_grid), 15.0)):
                cropped = crop_nonbackground(test_grid, bg)
                if cropped is not None:
                    generic.append((cropped, priority))

            # A monochrome shape prior covers the rare all-background/all-color task.
            shapes = infer_output_shapes(task, test_grid, max_shapes=2)
            if shapes:
                h, w = int(shapes[0][0]), int(shapes[0][1])
                train_modes = [mode_color(as_grid(example["output"])) for example in task["train"]]
                fill = train_modes[0] if len(set(train_modes)) == 1 else 0
                generic.append((np.full((h, w), fill, dtype=np.int8), 5.0))

            for grid, priority in generic:
                if not valid_grid(grid):
                    continue
                h = grid_key(grid)
                if h in seen:
                    continue
                seen.add(h)
                records.append(_record(grid, source="fallback", priority=priority))
                if len(records) >= 6:
                    break

            # ARC requires two attempts; ensure at least two distinct valid grids.
            if len(records) < 2:
                for value in range(10):
                    candidate = np.full(test_grid.shape, value, dtype=np.int8)
                    h = grid_key(candidate)
                    if h not in seen:
                        seen.add(h)
                        records.append(_record(candidate, source="fallback", priority=-float(value)))
                    if len(records) >= 2:
                        break

            from arc_exec_rl.common import atomic_candidate_pickle
            atomic_candidate_pickle(os.path.join(output_dir, f"{key}_{test_idx}.fallback"), records)
            written += 1
        if key_retrieved:
            fully_retrieved.add(key)
    print(
        f"*** Precomputed fallbacks for {written} outputs in {time.time() - start:.1f}s; "
        f"fully retrieved puzzles={len(fully_retrieved)}"
    )
    return fully_retrieved


def local_worker(rank: int, queue: Any, end_time: float, nprocs: int, marker_dir: str) -> None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    torch.set_default_device("cpu")
    torch.set_num_threads(max(1, int(os.getenv("OMP_NUM_THREADS", "12"))))

    if rank > 0:
        ready = os.path.join(marker_dir, f"worker_{rank - 1}.ready")
        failed = os.path.join(marker_dir, f"worker_{rank - 1}.failed")
        deadline = time.time() + 900
        while not os.path.exists(ready):
            if os.path.exists(failed):
                raise RuntimeError(f"Previous worker failed during Unsloth import: {failed}")
            if time.time() > deadline:
                raise TimeoutError(f"Timed out waiting for {ready}")
            time.sleep(2)

    try:
        # Sequential import prevents concurrent global patching inside Unsloth.
        from arc_solver import worker
    except Exception:
        open(os.path.join(marker_dir, f"worker_{rank}.failed"), "w").close()
        raise

    open(os.path.join(marker_dir, f"worker_{rank}.ready"), "w").close()
    print(f"[Rank {rank}] start on visible GPU 0")
    worker(rank, queue, end_time, n_workers=nprocs)
    print(f"[Rank {rank}] done")


def _complexity(task: dict[str, Any]) -> int:
    total = 0
    for split in ("train", "test"):
        for example in task[split]:
            total += int(np.asarray(example["input"]).size)
            if "output" in example:
                total += int(np.asarray(example["output"]).size)
    return total * max(1, len(task["test"]))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--nprocs", type=int, default=4)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() not in ("", "0", "false", "no")
    filename = "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json"
    test_path = os.path.join(COMPETITION_DIR, filename)
    with open(test_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    requested_raw = os.getenv("ARC_DEV_KEYS", "").strip() or os.getenv("ARC_DEBUG_KEYS", "").strip()
    if requested_raw:
        requested = [item.strip() for item in requested_raw.split(",") if item.strip()]
        keys = [key for key in requested if key in data]
    else:
        # A serious interactive run evaluates all 120 public tasks; a competition
        # rerun evaluates every hidden task.  No silent four-task debug subset.
        keys = sorted(data, key=lambda key: (_complexity(data[key]), key), reverse=True)
    if not keys:
        raise RuntimeError("No ARC tasks selected; inspect ARC_DEV_KEYS/competition input")

    end_time = args.end_time if args.end_time > time.time() else time.time() + 11.5 * 3600

    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    from arc_retrieval import RetrievalIndex

    retrieval_index = RetrievalIndex.from_files(
        [
            (
                os.path.join(COMPETITION_DIR, "arc-agi_training_challenges.json"),
                os.path.join(COMPETITION_DIR, "arc-agi_training_solutions.json"),
            )
        ]
    )
    print(f"*** Retrieval index contains {len(retrieval_index)} reference puzzles")
    fully_retrieved = precompute_fallbacks(data, keys, OUTPUT_DIR, retrieval_index=retrieval_index)
    gpu_keys = [key for key in keys if key not in fully_retrieved]
    if not gpu_keys:
        print("*** Every requested puzzle was solved by exact canonical retrieval; GPU stage skipped")
        return

    nprocs = max(1, min(int(args.nprocs), 4, len(gpu_keys)))
    manager = mp.Manager()
    queue = manager.Queue()
    for key in gpu_keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    marker_dir = tempfile.mkdtemp(prefix="arc_unsloth_", dir="/kaggle")
    print(f"*** Solving {len(gpu_keys)} unresolved puzzles with {nprocs} workers; deadline={end_time:.0f}")
    mp.spawn(local_worker, args=(queue, end_time, nprocs, marker_dir), nprocs=nprocs, join=True)


if __name__ == "__main__":
    main()


## 1. Offline preflight and immediate complete fallback

In [ ]:
import subprocess
from arc_exec_rl.common import atomic_json, initial_submission, validate_submission, public_task, json_hash, truthy
from arc_exec_rl.hybrid import gate_valid
from arc_exec_rl.infer import DEFAULT_INFER
from arc_dataset_guard import find_competition_dir, resolve_primary_model, build_safe_routes
subprocess.run([sys.executable, "-m", "compileall", "-q", "arc_exec_rl"], check=True)
subprocess.run([sys.executable, "-m", "py_compile", "arc_loader.py", "arc_solver.py", "arc_decoder.py", "starter.py"], check=True)
from arc_exec_rl.kaggle_inputs import resolve_data,model_contract,scan,install_offline,dependency_report
if INSTALL_OFFLINE_DEPS: install_offline(WHEELHOUSE)
print('Dependencies:',json.dumps(dependency_report(),indent=2))
resolved_data,data_audit=resolve_data(INPUT_ROOT,SAFE_DIR/'normalized_data',COMPETITION_DIR_OVERRIDE)
competition_dir=Path(resolved_data)
atomic_json(SAFE_DIR/'mounted_dataset_audit.json',data_audit)
os.environ["ARC_COMPETITION_DIR"]=str(competition_dir)
rerun=truthy(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
challenge_path=competition_dir/("arc-agi_test_challenges.json" if rerun else "arc-agi_evaluation_challenges.json")
tasks={k:public_task(v) for k,v in json.loads(challenge_path.read_text()).items()}
if ARC_DEV_KEYS:
    if rerun: raise ValueError("DEV_KEYS must be empty in competition rerun")
    keys=set(ARC_DEV_KEYS.split(","));tasks={k:v for k,v in tasks.items() if k in keys}
    if not tasks: raise ValueError("Empty debug task set")
submission_path=Path("/kaggle/working/submission.json")
atomic_json(submission_path, initial_submission(tasks))
print("FALLBACK BANKED:",len(tasks),"tasks",sum(len(t["test"]) for t in tasks.values()),"outputs")
if (competition_dir/"arc-agi_training_challenges.json").exists():
    training=json.loads((competition_dir/"arc-agi_training_challenges.json").read_text())
    same=sum(k in training and public_task(training[k])==v for k,v in tasks.items())
    print("Exact overlap with public training:",same,"/",len(tasks),"— overlapping placeholders are not hidden-score evidence")
route_report=build_safe_routes(SAFE_DIR,RUN_PROFILE)
model_path,model_audit=resolve_primary_model(SAFE_DIR, override=PRIMARY_MODEL_OVERRIDE,
    allow_custom=ALLOW_CUSTOM_PRIMARY,require_exact_ids=REQUIRE_EXACT_16_TOKEN_IDS)
os.environ["ARC_MODEL_PATH"]=model_path
print("G0 CHECKPOINT:", model_path)
atomic_json(SAFE_DIR/'g0_header_contract.json',model_contract(model_path,'grid'))
if not PROGRAM_MODEL_DIR and PROGRAM_MODE!='off':
    manifests=scan(INPUT_ROOT,{'model_manifest.json'})
    eligible=[]
    for candidate in manifests:
        try:
            m=json.loads(candidate.read_text())
            if m.get('architecture')=='qwen3-program-dsl' and m.get('reload_verified'):
                eligible.append(str(candidate.parent))
        except (ValueError,OSError): pass
    if len(eligible)==1: PROGRAM_MODEL_DIR=eligible[0]
    elif len(eligible)>1: print('P1 AUTO-DISABLED: multiple exports; set PROGRAM_MODEL_DIR:',eligible)
if PROGRAM_MODE not in ("off","shadow","promoted"): raise ValueError("Invalid PROGRAM_MODE")
if rerun and PROGRAM_MODE=="shadow": raise ValueError("Shadow experiments are not allowed in competition reruns")
if PROGRAM_MODEL_DIR and PROGRAM_MODE!="off":
    from arc_exec_rl.modeling import audit_program_model
    try:
        audit_program_model(PROGRAM_MODEL_DIR)
        manifest=json.loads((Path(PROGRAM_MODEL_DIR)/"model_manifest.json").read_text())
        gate_path=Path(PROGRAM_MODEL_DIR)/"hybrid_promotion.json"
        gate=json.loads(gate_path.read_text()) if gate_path.exists() else None
        approved=gate_valid(gate,manifest["weights_id"],json_hash({**DEFAULT_INFER,**PROGRAM_INFER}))
        PROGRAM_READY=bool(manifest.get("reload_verified") and (PROGRAM_MODE=="shadow" or approved))
        print("P1 reload verified:",manifest.get("reload_verified"),"hybrid promotion:",approved)
    except (OSError,ValueError,KeyError) as exc:
        print("P1 DISABLED:",type(exc).__name__,str(exc)); PROGRAM_READY=False
FINAL_MODEL_DEADLINE=NOTEBOOK_START_TIME+TOTAL_NOTEBOOK_HOURS*3600-FINAL_RESERVE_MINUTES*60
GLOBAL_END_TIME=FINAL_MODEL_DEADLINE-(PROGRAM_RESERVE_SECONDS if PROGRAM_READY else 0)
print("P1 ready:",PROGRAM_READY,"G0 remaining seconds:",round(GLOBAL_END_TIME-time.time()))
atomic_json(SAFE_DIR/"execution_plan.json",{"program_ready":PROGRAM_READY,"program_mode":PROGRAM_MODE,
             "program_model":PROGRAM_MODEL_DIR,"program_infer":PROGRAM_INFER,"anchor_deadline":GLOBAL_END_TIME,
             "final_deadline":FINAL_MODEL_DEADLINE,"target_accuracy":TARGET_ACCURACY})


## 2. G0 broad pass — four independent L4 workers

In [ ]:
import signal,subprocess
cmd=[sys.executable,"-u","starter.py","--end-time",str(GLOBAL_END_TIME),"--nprocs",str(NPROCS)]
env=dict(os.environ,UNSLOTH_DISABLE_STATISTICS="1",TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas",OMP_NUM_THREADS="2")
if time.time()>=GLOBAL_END_TIME: raise TimeoutError("No primary phase time remains")
log_path=SAFE_DIR/"anchor_stdout.log"
with log_path.open("w") as log:
    proc=subprocess.Popen(cmd,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    while proc.poll() is None and time.time()<GLOBAL_END_TIME:
        time.sleep(10)
    if proc.poll() is None:
        os.killpg(proc.pid,signal.SIGTERM)
        try:proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid,signal.SIGKILL);proc.wait()
print("G0 exit:",proc.returncode,"log:",log_path)
print(log_path.read_text()[-10000:])


## 3. Bank the exact baseline and all retained candidates

In [ ]:
import numpy as np
from arc_decoder import ArcDecoder
from arc_loader import ArcDataset
from arc_exec_rl.common import validate_grid
data=ArcDataset.from_file(str(challenge_path))
decoder=ArcDecoder(data.split_multi_replies(),n_guesses=2)
decoder.load_decoded_results(str(OUTPUT_DIR))
selection=decoder.run_selection_algo()
anchor=json.loads(submission_path.read_text())
anchor_bank={}; primary_ready=0
for key,task in tasks.items():
    for i in range(len(task["test"])):
        bk=f"{key}_{i}"; chosen=selection.get(bk,[])
        if chosen:
            a=np.asarray(chosen[0]).tolist();validate_grid(a);anchor[key][i]["attempt_1"]=a
            b=np.asarray(chosen[1]).tolist() if len(chosen)>1 else a
            validate_grid(b);anchor[key][i]["attempt_2"]=b
        values=decoder.decoded_results.get(bk,{})
        primary_ready+=int(any(v.get("source")=="llm" for v in values.values()))
        grids=[]
        for v in values.values():
            try:
                g=np.asarray(v["solution"]).tolist();validate_grid(g)
                if g not in grids: grids.append(g)
            except (KeyError,ValueError,TypeError):pass
        anchor_bank[f"{key}:{i}"]=grids
validate_submission(tasks,anchor)
atomic_json(submission_path,anchor)
anchor_path=SAFE_DIR/"anchor_submission.json";atomic_json(anchor_path,anchor)
atomic_json(SAFE_DIR/"anchor_candidate_bank.json",anchor_bank)
coverage=primary_ready/max(1,sum(len(t["test"]) for t in tasks.values()))
print("ANCHOR SUBMISSION VALID; neural coverage:",coverage)
# A deficient primary pass is an error signal, not a reason to claim solver success.


## 4. P1 gated program phase — bounded generation and demonstration-only repair

In [ ]:
from arc_exec_rl.hybrid import run_stage
stage_report={"status":"NOT_RUN_NO_ELIGIBLE_MODEL"}
if PROGRAM_READY and coverage>=MIN_PRIMARY_COVERAGE and time.time()+300<FINAL_MODEL_DEADLINE:
    clean_challenges=SAFE_DIR/"challenge_inputs_only.json";atomic_json(clean_challenges,tasks)
    try:
        stage_report=run_stage(str(clean_challenges),str(anchor_path),PROGRAM_MODEL_DIR,str(PROGRAM_OUTPUT),
                         PROGRAM_INFER,FINAL_MODEL_DEADLINE,NPROCS,apply_if_promoted=PROGRAM_MODE=="promoted")
        if stage_report.get("applied"):
            proposed=json.loads((PROGRAM_OUTPUT/"final_submission.json").read_text())
            validate_submission(tasks,proposed);atomic_json(submission_path,proposed)
    except Exception as exc:
        stage_report={"status":"P1_FAILED_ANCHOR_PRESERVED","error_type":type(exc).__name__,"error":str(exc)}
elif PROGRAM_READY and coverage<MIN_PRIMARY_COVERAGE:
    stage_report={"status":"NOT_RUN_PRIMARY_COVERAGE_GATE_FAILED","coverage":coverage}
atomic_json(SAFE_DIR/"program_stage_summary.json",{k:v for k,v in stage_report.items() if k!="candidate_bank"})
print(json.dumps({k:v for k,v in stage_report.items() if k!="candidate_bank"},indent=2)[-8000:])


## 5. Write and score — no training or selection after opening public solutions

In [ ]:
from arc_exec_rl.common import score_submission
final=json.loads(submission_path.read_text());validate_submission(tasks,final)
report={"submission_valid":True,"tasks":len(tasks),"program_stage":stage_report.get("status"),
        "program_applied":stage_report.get("applied",False),"target_accuracy":TARGET_ACCURACY,
        "actual_public_result":None,"hidden_result":None,"seconds":time.time()-NOTEBOOK_START_TIME}
if not rerun:
    sp=competition_dir/"arc-agi_evaluation_solutions.json"
    if sp.exists():
        all_solutions=json.loads(sp.read_text());solutions={k:all_solutions[k] for k in tasks}
        union=dict(anchor_bank)
        for k,v in stage_report.get("candidate_bank",{}).items():union[k]=union.get(k,[])+v
        actual=score_submission(tasks,solutions,final,union)
        report["actual_public_result"]=actual
        print(f"PAIR MICRO PASS@2: {actual['pass2_exact']}/{actual['pairs']} = {actual['pair_micro_pass2']:.2%}")
        print(f"TASK MACRO PASS@2: {actual['task_macro_pass2']:.2%}")
        print("FULLY SOLVED TASKS:",actual["fully_solved_tasks"],"/",actual["tasks"])
        print("CANDIDATE ORACLE:",actual["candidate_oracle_exact"],"/",actual["pairs"])
        required37=math.ceil(actual['pairs']*TARGET_MILESTONE)
        report['milestone_37']={'required_pairs':required37,'met':actual['pass2_exact']>=required37}
        print('37% PAIR TARGET:','MEASURED PASS' if actual['pass2_exact']>=required37 else 'NOT MET','required:',required37)
        print("60% PAIR TARGET:","MEASURED PASS" if actual["target_60_pair_micro_met"] else "NOT MET",
              "required:",actual["target_60_required_pairs"])
        if (PROGRAM_OUTPUT/"proposed_submission.json").exists():
            shadow=json.loads((PROGRAM_OUTPUT/"proposed_submission.json").read_text())
            report["shadow_proposal_diagnostic"]=score_submission(tasks,solutions,shadow,union)
atomic_json(SAFE_DIR/"final_report.json",report)
print("FINAL SUBMISSION VALID:",submission_path)
print("A valid file is not evidence of 37% or 60% accuracy. The actual score above is authoritative for this run only.")
